# Bias-Aware Political Content Detection on X: Reproducible Experiments and Model Benchmarks

## JupyterLab environment and project layout

**Runtime:** CPython 3.12.14, kernel `Python (tweetsCYG)`. The notebook runs inside the existing shared JupyterLab service. Use the private virtual environment under `~/data/cayala/envs/tweetsCYG` and do not modify the server-wide Python installation.

**Data:** `~/data/cayala/datasets/` holds the original event corpus, trend archive, and annotations. Original annotations are never overwritten.

**Results:** `~/data/cayala/results/` holds all tables, plots, predictions, protocol audits, and benchmark reports. Model checkpoints and final fine-tuned weights are stored in `~/data/cayala/checkpoints/` and `~/data/cayala/models/`. Hugging Face and PyTorch caches reside in `~/data/cayala/cache/`.

**Dependencies:** the environment specification is `~/data/cayala/envs/requirements.txt`. PyTorch with CUDA is installed separately before these requirements. The notebook writes an observed version snapshot to `~/data/cayala/envs/requirements_runtime.txt` and a complete `pip freeze` snapshot to `~/data/cayala/envs/requirements_lock_tweetsCYG.txt`. These snapshots do not overwrite the installation requirements.

**Compute policy:** use only GPUs available to your account and avoid devices currently occupied by other users. The notebook selects available devices at kernel start and supports optional multi-GPU training. The original repeated cross-validation and human/silver evaluation protocols remain intact. New model comparisons use the same training rows and weak-supervision target, with human consensus labels reserved exclusively for final evaluation.

Run the notebook in order using **Kernel → Restart Kernel and Run All Cells**. A fresh kernel is important for the GPU device visibility settings.

# 1. Libraries, reproducibility and hardware

In [302]:
from __future__ import annotations

import os as _os
import subprocess as _subprocess
from pathlib import Path as _Path
_HOME = _Path.home()
_ROOT = _HOME / "data" / "cayala"
for _subdir in ("cache/huggingface", "cache/torch", "cache/uv", "cache/pip", "tmp"):
    (_ROOT / _subdir).mkdir(parents=True, exist_ok=True)
_os.environ["HF_HOME"] = str(_ROOT / "cache" / "huggingface")
_os.environ["HF_HUB_CACHE"] = str(_ROOT / "cache" / "huggingface" / "hub")
_os.environ["TORCH_HOME"] = str(_ROOT / "cache" / "torch")
_os.environ["XDG_CACHE_HOME"] = str(_ROOT / "cache")
_os.environ["TMPDIR"] = str(_ROOT / "tmp")
_os.environ["TMP"] = str(_ROOT / "tmp")
_os.environ["TEMP"] = str(_ROOT / "tmp")
_os.environ["TOKENIZERS_PARALLELISM"] = "false"
_os.environ["PIP_NO_CACHE_DIR"] = "1"

import gc
import hashlib
import importlib.metadata as importlib_metadata
import json
import math
import os
import platform
import random
import re
import sys
import time
import warnings
from collections import Counter
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence

import numpy as np
import pandas as pd
import psutil
import spacy
from gensim.models import FastText, Word2Vec
from IPython.display import display
from scipy.stats import binomtest, t as student_t, wilcoxon
from spacy.language import Language
from spacy.tokens import Doc
from sklearn.cluster import HDBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    calinski_harabasz_score,
    cohen_kappa_score,
    confusion_matrix,
    davies_bouldin_score,
    f1_score,
    fbeta_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    silhouette_score,
)
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import normalize
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

from bokeh.io import output_notebook, show
from bokeh.layouts import column, row
from bokeh.models import (
    ColumnDataSource,
    DataTable,
    Div,
    HoverTool,
    PanTool,
    WheelZoomTool,
    BoxZoomTool,
    ResetTool,
    SaveTool,
    HelpTool,
    TabPanel,
    TableColumn,
    Tabs,
    LinearColorMapper,
    ColorBar,
    BasicTicker,
    LabelSet,
)
from bokeh.palettes import Category10, Category20, Turbo256, Blues9
from bokeh.plotting import figure, show
from bokeh.transform import dodge

import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer

output_notebook(hide_banner=True)

In [303]:
CPU_WORKERS = max(1, min(8, int(_os.environ.get("CAYALA_CPU_WORKERS", "8")), _os.cpu_count() or 1))
for _var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    _os.environ[_var] = str(CPU_WORKERS)

In [304]:
def _choose_visible_cuda_devices():
    """Select idle GPUs without taking ownership of busy shared-server devices."""
    if "CUDA_VISIBLE_DEVICES" in _os.environ:
        return _os.environ["CUDA_VISIBLE_DEVICES"]
    explicit = _os.environ.get("CAYALA_GPU_IDS", "").strip()
    if explicit:
        _os.environ["CUDA_VISIBLE_DEVICES"] = explicit
        return explicit
    try:
        result = _subprocess.run([
            "nvidia-smi", "--query-gpu=index,memory.used,utilization.gpu",
            "--format=csv,noheader,nounits"
        ], check=True, capture_output=True, text=True, timeout=10)
        candidates = []
        for line in result.stdout.strip().splitlines():
            fields=[x.strip() for x in line.split(",")]
            if len(fields) >= 3:
                index, memory, utilization = map(int, fields[:3])
                if memory <= 1200 and utilization <= 15:
                    candidates.append((memory, utilization, index))
        count = max(1, min(8, int(_os.environ.get("CAYALA_MAX_GPUS", "2"))))
        selected = [str(row[2]) for row in sorted(candidates)[:count]]
        if selected:
            _os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(selected)
    except (FileNotFoundError, ValueError, _subprocess.SubprocessError):
        pass
    return _os.environ.get("CUDA_VISIBLE_DEVICES", "not restricted")

In [305]:
VISIBLE_CUDA_DEVICES = _choose_visible_cuda_devices()
print("Visible GPU selection (physical indices):", VISIBLE_CUDA_DEVICES)
print("CPU workers allowed:", CPU_WORKERS)

Visible GPU selection (physical indices): 0,1
CPU workers allowed: 8


In [306]:
warnings.filterwarnings("ignore")
random.seed(5)
np.random.seed(5)
torch.manual_seed(5)

In [307]:
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(5)
    torch.set_float32_matmul_precision("high")
if torch.cuda.is_available():
    print("CUDA devices visible:", torch.cuda.device_count())
    print("GPU names:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


CUDA devices visible: 2
GPU names: ['NVIDIA A16', 'NVIDIA A16']


In [7]:
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

In [8]:
DISPLAY_COLUMN_TRANSLATIONS = {
    "evento":"Event", "texto_limpio":"Clean text", "texto_original":"Original text",
    "id_registro":"Record ID", "etiqueta_consenso":"Consensus label",
    "etiqueta_silver_hashtag":"Silver label", "etiqueta_humana":"Human label",
    "tokens_sin_stopwords":"Filtered tokens", "trend_limpio":"Clean trend",
    "Porcentaje_tweets":"Tweet percentage", 
    "Escenario":"Scenario", "Escenario 1":"Scenario 1", "Escenario 2":"Scenario 2",
    "Algoritmo":"Algorithm", "Metrica":"Metric", "Exactitud":"Accuracy",
    "Precisión":"Precision", "Especificidad":"Specificity", "Sensibilidad":"Sensitivity",
    "Dimensión":"Dimension", "Representacion":"Representation", "Referencia":"Reference",
}

def display_bokeh_table(df: pd.DataFrame, title: str, width: int = 950, height: int = 280,
                        max_rows: int = 25, **legacy_kwargs) -> None:
    """Display an English-labeled, scrollable Bokeh table without changing its source DataFrame."""
    title = str(title)
    width = legacy_kwargs.pop("ancho", width)
    height = legacy_kwargs.pop("alto", height)
    max_rows = legacy_kwargs.pop("max_filas", max_rows)
    if legacy_kwargs:
        raise TypeError(f"Unexpected table options: {sorted(legacy_kwargs)}")
    view = df.head(max_rows).copy()
    view.columns = [DISPLAY_COLUMN_TRANSLATIONS.get(str(col),str(col)) for col in view.columns]
    for col in view.columns:
        if pd.api.types.is_object_dtype(view[col]) or pd.api.types.is_string_dtype(view[col]):
            view[col] = view[col].astype(str).replace({"Escenario 1":"Scenario 1","Escenario 2":"Scenario 2"})
    source = ColumnDataSource(view)
    columns = [TableColumn(field=str(col), title=str(col)) for col in view.columns]
    table = DataTable(source=source, columns=columns, width=width, height=height,index_position=None)
    show(column(Div(text=f"<h3>{title}</h3>"), table))

In [9]:
def audit_library_versions() -> pd.DataFrame:
    """Report the actually installed versions used in this execution."""
    packages = {
        "Python": None,
        "NumPy": "numpy", "pandas": "pandas", "SciPy": "scipy",
        "scikit-learn": "scikit-learn", "Gensim": "gensim", "Bokeh": "bokeh",
        "openpyxl": "openpyxl", "PyTorch": "torch",
        "Transformers": "transformers", "Tokenizers": "tokenizers",
        "Accelerate": "accelerate", "Hugging Face Hub": "huggingface-hub",
        "Datasets": "datasets", "SafeTensors": "safetensors",
        "SentencePiece": "sentencepiece", "pysentimiento": "pysentimiento",
        "XGBoost": "xgboost", "psutil": "psutil", "tqdm": "tqdm",
        "ipykernel": "ipykernel", "spaCy": "spacy",
        "Spanish spaCy model": "es_core_news_sm",
        "Matplotlib": "matplotlib", "PyYAML": "PyYAML",
    }
    rows = []
    for label, distribution in packages.items():
        try:
            observed = platform.python_version() if distribution is None else importlib_metadata.version(distribution)
        except importlib_metadata.PackageNotFoundError:
            observed = "NOT INSTALLED"
        rows.append({"Component":label, "Distribution":distribution or "python", "Installed":observed,
                     "Status":"OK" if observed != "NOT INSTALLED" else "MISSING"})
    return pd.DataFrame(rows)

In [10]:
def audit_hardware() -> pd.DataFrame:
    """Audit hardware. Retains the original experimental logic and data-partition rules."""
    if torch.cuda.is_available():
        dispositivo = "cuda"
        gpu = torch.cuda.get_device_name(0)
        cuda = str(torch.version.cuda)
        memoria_gpu = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        memoria_gpu_txt = f"{memoria_gpu:.2f} GB"
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        dispositivo = "mps"
        gpu = "Apple Metal Performance Shaders"
        cuda = "Not applicable"
        memoria_gpu_txt = "Unified memory"
    else:
        dispositivo = "cpu"
        gpu = "Unavailable"
        cuda = "Not applicable"
        memoria_gpu_txt = "Not applicable"

    filas = [
        {"Indicator": "Operating system", "Valor": platform.platform()},
        {"Indicator": "Python", "Valor": platform.python_version()},
        {"Indicator": "Python executable", "Valor": sys.executable},
        {"Indicator": "Logical CPUs", "Valor": str(os.cpu_count() or 1)},
        {"Indicator": "Total RAM", "Valor": f"{psutil.virtual_memory().total / (1024 ** 3):.2f} GB"},
        {"Indicator": "PyTorch device", "Valor": dispositivo},
        {"Indicator": "GPU", "Valor": gpu},
        {"Indicator": "GPU memory", "Valor": memoria_gpu_txt},
        {"Indicator": "PyTorch CUDA", "Valor": cuda},
        {"Indicator": "PyTorch", "Valor": torch.__version__},
    ]
    return pd.DataFrame(filas)

In [11]:
def select_torch_device() -> torch.device:
    """Select torch device. Retains the original experimental logic and data-partition rules."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

In [12]:
def save_runtime_requirements(library_audit: pd.DataFrame, target_path: Path) -> Path:
    """Write the observed package versions next to the source requirements file."""
    target_path.parent.mkdir(parents=True, exist_ok=True)
    lines=[]
    for _, row in library_audit.iterrows():
        if row["Component"] == "Python":
            lines.append(f"# Python=={row['Installed']}")
        elif row["Installed"] != "NOT INSTALLED":
            lines.append(f"{row['Distribution']}=={row['Installed']}")
    target_path.write_text("\n".join(lines)+"\n", encoding="utf-8")
    return target_path

In [13]:
AUDITORIA_LIBRERIAS = audit_library_versions()
AUDITORIA_HARDWARE = audit_hardware()
DISPOSITIVO_TORCH = select_torch_device()

In [308]:
display_bokeh_table(
    AUDITORIA_LIBRERIAS,
    "Library-version reproducibility audit",
    ancho=1000,
    alto=460,
    max_filas=30,
)

In [309]:
display_bokeh_table(
    AUDITORIA_HARDWARE,
    "Hardware and compute-device audit",
    ancho=1050,
    alto=300,
    max_filas=20,
)

In [16]:
print(f"Selected Transformer device: {DISPOSITIVO_TORCH}")
if platform.python_version() != "3.12.14":
    raise RuntimeError(f"Python 3.12.14 is required. Active interpreter: {platform.python_version()}")
print("Active Python executable:",sys.executable)

Selected Transformer device: cuda
Active Python executable: /home/lrecalde/data/cayala/envs/tweetsCYG/bin/python


# 2. Experimental configuration

In [17]:
PROJECT_ROOT = Path.home() / "notebooks" / "cayala"
CAYALA_ROOT = Path.home() / "data" / "cayala"
DATA_DIR = CAYALA_ROOT / "datasets"
RESULTS_DIR = CAYALA_ROOT / "results"
MODELS_DIR = CAYALA_ROOT / "models"
CHECKPOINTS_DIR = CAYALA_ROOT / "checkpoints"
ENVS_DIR = CAYALA_ROOT / "envs"
CACHE_DIR = CAYALA_ROOT / "cache"
TMP_DIR = CAYALA_ROOT / "tmp"
ANNOTATIONS_WORK_DIR = DATA_DIR / "anotaciones_trabajo"
REQUIREMENTS_SOURCE = ENVS_DIR / "requirements.txt"
RUNTIME_REQUIREMENTS = ENVS_DIR / "requirements_runtime.txt"
FULL_REQUIREMENTS_LOCK = ENVS_DIR / "requirements_lock_tweetsCYG.txt"
OUTPUT_DIR = RESULTS_DIR
REVISION_OUTPUT_DIR = RESULTS_DIR

In [18]:
for directory in (DATA_DIR, RESULTS_DIR, MODELS_DIR, CHECKPOINTS_DIR, ENVS_DIR, CACHE_DIR, TMP_DIR, ANNOTATIONS_WORK_DIR):
    directory.mkdir(parents=True, exist_ok=True)

In [19]:
if not REQUIREMENTS_SOURCE.is_file():
    raise FileNotFoundError(f"The environment requirements are missing: {REQUIREMENTS_SOURCE}")

In [20]:
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)
print("Environment requirements:", REQUIREMENTS_SOURCE)

Data directory: /home/lrecalde/data/cayala/datasets
Results directory: /home/lrecalde/data/cayala/results
Environment requirements: /home/lrecalde/data/cayala/envs/requirements.txt


In [21]:
ARCHIVOS = {
    "tweets": "tweets_consolidado_limpio_final.xlsx",
    "trends": "trends_limpio_final.xlsx",
    "resumen": "resumen_preprocesamiento.xlsx",
    "paro": "paro_nacional_limpio_final.xlsx",
    "muerte": "muerte_cruzada_limpio_final.xlsx",
}

In [22]:
RANDOM_STATE = 5
DIMENSIONES = (50, 100, 200)
TEST_SIZE = 0.20

In [23]:
W2V_WINDOW = 5
W2V_MIN_COUNT = 1
W2V_EPOCHS = 15
W2V_WORKERS = max(1, min(CPU_WORKERS, int(os.environ.get("CAYALA_W2V_WORKERS", "1"))))

In [24]:
K_CANDIDATOS = tuple(range(2, 11))
KNN_VECINOS = 5

In [25]:
EXPANSION_TOP_N_FINAL = 100
EXPANSION_TOP_N_CANDIDATOS = (25, 50, 100, 200)

In [26]:
HDBSCAN_MIN_CLUSTER_SIZE = 20
HDBSCAN_MIN_SAMPLES = 5

In [27]:
CV_FOLDS = 5
CV_REPEATS = 2
CV_EPOCHS = 10
METRICAS_ESTADISTICAS = (
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "F2 score",
    "F0.5 score",
)

In [28]:
N_ANOTACION_MANUAL = 600
SOURCE_HUMAN_ANNOTATION = DATA_DIR / "anotacion_manual_politica.xlsx"
RUTA_ANOTACION_MANUAL = ANNOTATIONS_WORK_DIR / SOURCE_HUMAN_ANNOTATION.name
if SOURCE_HUMAN_ANNOTATION.is_file() and not RUTA_ANOTACION_MANUAL.is_file():
    import shutil
    shutil.copy2(SOURCE_HUMAN_ANNOTATION, RUTA_ANOTACION_MANUAL)
print("Human annotation working copy:", RUTA_ANOTACION_MANUAL)

Human annotation working copy: /home/lrecalde/data/cayala/datasets/anotaciones_trabajo/anotacion_manual_politica.xlsx


In [29]:
N_MANUAL_DESACUERDOS_HASHTAG = 250
SOURCE_HASHTAG_AUDIT = DATA_DIR / "anotacion_desacuerdos_hashtag.xlsx"
RUTA_ANOTACION_DESACUERDOS_HASHTAG = ANNOTATIONS_WORK_DIR / SOURCE_HASHTAG_AUDIT.name
if SOURCE_HASHTAG_AUDIT.is_file() and not RUTA_ANOTACION_DESACUERDOS_HASHTAG.is_file():
    import shutil
    shutil.copy2(SOURCE_HASHTAG_AUDIT, RUTA_ANOTACION_DESACUERDOS_HASHTAG)
print("Human annotation reference:", RUTA_ANOTACION_MANUAL)
print("Hashtag audit working copy:", RUTA_ANOTACION_DESACUERDOS_HASHTAG)

Human annotation reference: /home/lrecalde/data/cayala/datasets/anotaciones_trabajo/anotacion_manual_politica.xlsx
Hashtag audit working copy: /home/lrecalde/data/cayala/datasets/anotaciones_trabajo/anotacion_desacuerdos_hashtag.xlsx


In [30]:
DIMENSION_BENCHMARK = 100
FASTTEXT_EPOCHS = 15
TRANSFORMER_MAX_LENGTH = 128
TRANSFORMER_BATCH_SIZE = 16 if DISPOSITIVO_TORCH.type == "cuda" else 8
TRANSFORMER_FINE_TUNE_EPOCHS = 2
TRANSFORMER_FINE_TUNE_LR = 2e-5
EJECUTAR_FINE_TUNING_TRANSFORMER = False  # Replaced by the controlled five-model fine-tuning benchmark below.

In [31]:
MODELOS_TRANSFORMER = {
    "BETO": "dccuchile/bert-base-spanish-wwm-uncased",
    "XLM-RoBERTa": "xlm-roberta-base",
}

In [32]:
MODELO_SPACY_ES = "es_core_news_sm"
SPACY_BATCH_SIZE = 512

LONGITUD_MINIMA_TOKEN = 3
PALABRAS_INCLUIDAS = {"ec"}
PALABRAS_EXCLUIDAS = {"jajaja"}
POS_EXCLUIDOS = {"VERB", "AUX", "DET"}

CONSERVAR_HASHTAGS_MODELO = True
CONSERVAR_MENCIONES_MODELO = True

VERSION_PROTOCOLO_ANOTACION = "2026-09-limpieza-anotaciones-v1"

In [33]:
MAX_NGRAM_TREND = 4

In [34]:
SEMILLAS_POLITICAS_EXTRA = [
    "politica", "politico", "presidente", "gobierno", "asamblea",
    "elecciones", "paro", "nacional", "muerte", "cruzada", "lasso",
    "guillermo", "correa", "iza", "protesta", "@asambleaecuador",
    "#paronacional", "ecuador",
]

In [35]:
HASHTAGS_POLITICOS_ESTRICTOS = {
    "#paronacionalec", "#paronacional", "#paronacionalecuador", "#paroecuador",
    "#muertecruzada", "#muertecruzadaya", "#muertecruzadaec",
    "#juiciopolitico", "#juiciopoliticoalasso", "#asambleanacional",
    "#lassoseva", "#guillermolasso", "#fueralassofuera", "#lassodestruyoecuador",
    "#juiciosinpruebas", "#conaie", "#izaterrorista", "#eleccionesec", "#cne",
    "#lassorenuncia", "#lassoasesino", "#ecuadorquierepaz", "#leonidasiza",
}

In [36]:
HASHTAGS_NO_POLITICOS_ESTRICTOS = {
    "#championsleague", "#emelec", "#ligapro", "#ligaprobet", "#copaecuadorecuabet",
    "#premierleague", "#realmadrid", "#laacademia", "#domingodeacademia",
    "#diadelpadre", "#felizdiadelpadre", "#tiktok", "#seguros", "#vida",
    "#portuseguridad", "#diadelamujermaritima", "#webinar", "#ecdfmundialista",
}

In [37]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REVISION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [38]:
RUTA_REQUIREMENTS_RUNTIME = save_runtime_requirements(AUDITORIA_LIBRERIAS, RUNTIME_REQUIREMENTS)
import subprocess as _subprocess
with FULL_REQUIREMENTS_LOCK.open("w", encoding="utf-8") as _requirements_lock:
    _subprocess.run([sys.executable,"-m","pip","freeze"],stdout=_requirements_lock,check=True)
print("Runtime versions:", RUTA_REQUIREMENTS_RUNTIME)
print("Full environment lock:", FULL_REQUIREMENTS_LOCK)
import importlib.util as _pkg_util
_REQUIRED_MODULES = ('transformers','accelerate','datasets','sentencepiece','xgboost','pysentimiento','safetensors')
_MISSING_MODULES = [m for m in _REQUIRED_MODULES if _pkg_util.find_spec(m) is None]
if _MISSING_MODULES:
    raise RuntimeError('Install all packages from ~/data/cayala/envs/requirements.txt in the tweetsCYG virtual environment before running the notebook: '+', '.join(_MISSING_MODULES))


Runtime versions: /home/lrecalde/data/cayala/envs/requirements_runtime.txt
Full environment lock: /home/lrecalde/data/cayala/envs/requirements_lock_tweetsCYG.txt


In [39]:
print(f"Dataset root: {DATA_DIR}")
print(f"Results root: {RESULTS_DIR}")
print(f"Runtime requirements: {RUTA_REQUIREMENTS_RUNTIME}")
print(f"Transformers device: {DISPOSITIVO_TORCH}")
print(f"Legacy silver fine-tuning enabled: {EJECUTAR_FINE_TUNING_TRANSFORMER}")

Dataset root: /home/lrecalde/data/cayala/datasets
Results root: /home/lrecalde/data/cayala/results
Runtime requirements: /home/lrecalde/data/cayala/envs/requirements_runtime.txt
Transformers device: cuda
Legacy silver fine-tuning enabled: False


# 3. Corpus loading, cleaning and dataset auditing

In [40]:
def resolve_data_file(filename: str, data_dir: Path) -> Path:
    """Resolve a required input file exclusively inside the datasets directory."""
    path = (data_dir / filename).resolve()
    if not path.is_file() or not path.is_relative_to(data_dir.resolve()):
        raise FileNotFoundError(f"Required dataset file not found: {path}")
    return path

In [41]:
def verify_input_files(archivos: Mapping[str, str], data_dir: Path) -> dict[str, Path]:
    """Verify input files. Retains the original experimental logic and data-partition rules."""
    rutas: dict[str, Path] = {}
    for clave, nombre in archivos.items():
        rutas[clave] = resolve_data_file(nombre, data_dir)
    return rutas

In [42]:
def load_input_datasets(rutas: Mapping[str, Path]) -> dict[str, Any]:
    """Load input datasets. Retains the original experimental logic and data-partition rules."""
    resumen = pd.read_excel(rutas["resumen"], sheet_name=None)
    return {
        "tweets": pd.read_excel(rutas["tweets"]),
        "trends": pd.read_excel(rutas["trends"]),
        "paro": pd.read_excel(rutas["paro"]),
        "muerte": pd.read_excel(rutas["muerte"]),
        "resumen": resumen,
    }

In [43]:
def parse_tokens(valor: Any) -> list[str]:
    """Parse tokens. Retains the original experimental logic and data-partition rules."""
    if isinstance(valor, list):
        return [str(x).strip() for x in valor if str(x).strip()]
    if pd.isna(valor):
        return []
    return [token for token in str(valor).strip().split() if token]

In [44]:
def extract_dataframe_tokens(df: pd.DataFrame, columna_tokens: str = "tokens_sin_stopwords") -> list[list[str]]:
    """Extract dataframe tokens. Retains the original experimental logic and data-partition rules."""
    if columna_tokens not in df.columns:
        raise KeyError(f"La columna '{columna_tokens}' no existe. Columnas: {list(df.columns)}")
    return [parse_tokens(valor) for valor in df[columna_tokens].tolist()]

In [45]:
def validate_event_consistency(tweets: pd.DataFrame, paro: pd.DataFrame, muerte: pd.DataFrame) -> pd.DataFrame:
    """Execute validate event consistency. Retains the original experimental logic and data-partition rules."""
    conteos = tweets["evento"].astype(str).str.strip().value_counts()
    filas = []
    for evento, separado in [("Paro Nacional", paro), ("Muerte Cruzada", muerte)]:
        consolidado = int(conteos.get(evento, 0))
        separado_n = int(len(separado))
        filas.append({
            "Evento": evento,
            "Consolidado": consolidado,
            "Archivo_separado": separado_n,
            "Diferencia": consolidado - separado_n,
        })
    return pd.DataFrame(filas)

In [46]:
def create_record_id(df: pd.DataFrame) -> pd.Series:
    """Create record id. Retains the original experimental logic and data-partition rules."""
    if "evento" not in df.columns:
        raise KeyError("The `evento` column is required to construct stable identifiers.")

    candidatas = [
        "texto_id_fuente",
        "texto_limpio_fuente",
        "texto_limpio",
        "texto_original",
        "texto",
    ]
    columna = next((c for c in candidatas if c in df.columns), None)
    if columna is None:
        raise KeyError("No usable text column is available for constructing `id_registro`.")

    claves = (
        df["evento"].fillna("").astype(str).str.strip()
        + "||"
        + df[columna].fillna("").astype(str).str.strip()
    )
    return claves.map(
        lambda valor: hashlib.sha256(valor.encode("utf-8")).hexdigest()[:20]
    )

In [47]:
def assign_stable_ids_before_cleaning(df: pd.DataFrame) -> pd.DataFrame:
    """Assign stable ids before cleaning. Retains the original experimental logic and data-partition rules."""
    trabajo = df.copy()
    if "texto_limpio" not in trabajo.columns:
        raise KeyError("The `texto_limpio` field is required to stabilize IDs before cleaning.")
    trabajo["texto_id_fuente"] = (
        trabajo["texto_limpio"].fillna("").astype(str).str.strip()
    )
    trabajo["id_registro"] = create_record_id(trabajo)
    return trabajo

In [48]:
def resolve_source_text_column(
    df: pd.DataFrame,
    candidatas: Sequence[str],
) -> str:
    """Resolve source text column. Retains the original experimental logic and data-partition rules."""
    for columna in candidatas:
        if columna in df.columns:
            return str(columna)
    raise KeyError(
        "No usable text column was found. "
        f"Candidatas evaluadas: {list(candidatas)}"
    )

In [49]:
def load_spanish_spacy_model(nombre_modelo: str) -> Language:
    """Load spanish spacy model. Retains the original experimental logic and data-partition rules."""
    try:
        return spacy.load(nombre_modelo, disable=["parser", "ner"])
    except OSError as exc:
        raise RuntimeError(
            f"No está instalado `{nombre_modelo}`. Active tweetsCYG y ejecute: "
            f"`python -m spacy download {nombre_modelo}`; luego reinicie el kernel."
        ) from exc

In [50]:
def protect_social_media_entities(texto: str) -> str:
    """Protect social media entities. Retains the original experimental logic and data-partition rules."""
    valor = str(texto)
    valor = re.sub(
        r"(?<!\w)#([\wáéíóúüñÁÉÍÓÚÜÑ]+)",
        r" hashtagzz\1 ",
        valor,
    )
    valor = re.sub(
        r"(?<!\w)@([\wáéíóúüñÁÉÍÓÚÜÑ]+)",
        r" mentionzz\1 ",
        valor,
    )
    return valor

In [51]:
def filter_spacy_document(
    doc: Doc,
    inclusiones: set[str],
    exclusiones: set[str],
    pos_excluidos: set[str],
    longitud_minima: int,
    conservar_hashtags: bool,
    conservar_menciones: bool,
) -> tuple[list[str], dict[str, int]]:
    """Filter spacy document. Retains the original experimental logic and data-partition rules."""
    incl = {str(x).strip().lower() for x in inclusiones}
    excl = {str(x).strip().lower() for x in exclusiones}
    pos_bloqueados = {str(x).strip().upper() for x in pos_excluidos}

    conteos: Counter[str] = Counter()
    salida: list[str] = []

    for token in doc:
        bruto = token.text.strip().lower()
        if not bruto or token.is_space:
            continue

        es_hashtag = bruto.startswith("hashtagzz")
        es_mencion = bruto.startswith("mentionzz")

        if es_hashtag:
            nucleo = bruto[len("hashtagzz"):].strip("_")
            normal = f"#{nucleo}" if nucleo else ""
            clave_regla = nucleo
        elif es_mencion:
            nucleo = bruto[len("mentionzz"):].strip("_")
            normal = f"@{nucleo}" if nucleo else ""
            clave_regla = nucleo
        else:
            normal = bruto
            clave_regla = normal

        conteos["tokens_evaluados"] += 1

        if not normal:
            conteos["eliminados_ruido"] += 1
            continue

        if clave_regla in excl or normal in excl:
            conteos["eliminados_lista_negra"] += 1
            continue

        if clave_regla in incl or normal in incl:
            salida.append(normal)
            conteos["conservados_lista_blanca"] += 1
            conteos["tokens_conservados"] += 1
            continue

        if es_hashtag:
            if not conservar_hashtags:
                conteos["eliminados_hashtag"] += 1
                continue
            if len(clave_regla) < int(longitud_minima):
                conteos["eliminados_longitud"] += 1
                continue
            salida.append(normal)
            conteos["tokens_conservados"] += 1
            continue

        if es_mencion:
            if not conservar_menciones:
                conteos["eliminados_mencion"] += 1
                continue
            if len(clave_regla) < int(longitud_minima):
                conteos["eliminados_longitud"] += 1
                continue
            salida.append(normal)
            conteos["tokens_conservados"] += 1
            continue

        if token.like_url or token.like_email:
            conteos["eliminados_url_email"] += 1
            continue

        if not token.is_alpha:
            conteos["eliminados_ruido"] += 1
            continue

        if token.is_stop:
            conteos["eliminados_stopword"] += 1
            continue

        if token.pos_.upper() in pos_bloqueados:
            conteos["eliminados_pos"] += 1
            continue

        if len(normal) < int(longitud_minima):
            conteos["eliminados_longitud"] += 1
            continue

        salida.append(normal)
        conteos["tokens_conservados"] += 1

    return salida, dict(conteos)

In [52]:
def clean_dataset_with_spacy(
    df: pd.DataFrame,
    nlp: Language,
    candidatas_texto: Sequence[str],
    inclusiones: set[str],
    exclusiones: set[str],
    pos_excluidos: set[str],
    longitud_minima: int,
    batch_size: int,
    tipo: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Clean dataset with spacy. Retains the original experimental logic and data-partition rules."""
    trabajo = df.copy()
    columna_fuente = resolve_source_text_column(trabajo, candidatas_texto)

    fuente = trabajo[columna_fuente].fillna("").astype(str)
    trabajo["texto_para_anotacion"] = fuente
    trabajo["texto_para_hashtags"] = fuente

    if "texto_limpio" in trabajo.columns:
        trabajo["texto_limpio_fuente"] = (
            trabajo["texto_limpio"].fillna("").astype(str)
        )
    elif tipo == "trends" and "trend_limpio" in trabajo.columns:
        trabajo["texto_limpio_fuente"] = (
            trabajo["trend_limpio"].fillna("").astype(str)
        )
    else:
        trabajo["texto_limpio_fuente"] = fuente

    textos_protegidos = [
        protect_social_media_entities(x)
        for x in fuente.tolist()
    ]

    tokens_finales: list[list[str]] = []
    acumulado: Counter[str] = Counter()
    documentos_vacios = 0

    for doc in nlp.pipe(
        textos_protegidos,
        batch_size=int(batch_size),
        n_process=1,
    ):
        tokens, conteos = filter_spacy_document(
            doc=doc,
            inclusiones=inclusiones,
            exclusiones=exclusiones,
            pos_excluidos=pos_excluidos,
            longitud_minima=longitud_minima,
            conservar_hashtags=CONSERVAR_HASHTAGS_MODELO,
            conservar_menciones=CONSERVAR_MENCIONES_MODELO,
        )
        tokens_finales.append(tokens)
        acumulado.update(conteos)
        if not tokens:
            documentos_vacios += 1

    textos_finales = [" ".join(tokens) for tokens in tokens_finales]
    tokens_serializados = [" ".join(tokens) for tokens in tokens_finales]

    trabajo["texto_limpio"] = textos_finales
    trabajo["tokens_sin_stopwords"] = tokens_serializados
    trabajo["cantidad_tokens_sin_stopwords"] = [
        len(tokens) for tokens in tokens_finales
    ]

    if tipo == "trends":
        trabajo["trend_limpio"] = textos_finales

    auditoria = pd.DataFrame([
        {"Indicator": "Dataset", "Valor": str(tipo)},
        {"Indicator": "Columna fuente", "Valor": str(columna_fuente)},
        {"Indicator": "Documentos", "Valor": int(len(trabajo))},
        {"Indicator": "Empty documents after cleaning", "Valor": int(documentos_vacios)},
        {"Indicator": "Tokens evaluados", "Valor": int(acumulado.get("tokens_evaluados", 0))},
        {"Indicator": "Tokens conservados", "Valor": int(acumulado.get("tokens_conservados", 0))},
        {"Indicator": "Retained by whitelist", "Valor": int(acumulado.get("conservados_lista_blanca", 0))},
        {"Indicator": "Eliminados stopword", "Valor": int(acumulado.get("eliminados_stopword", 0))},
        {"Indicator": "Removed by POS filtering", "Valor": int(acumulado.get("eliminados_pos", 0))},
        {"Indicator": "Removed by token length", "Valor": int(acumulado.get("eliminados_longitud", 0))},
        {"Indicator": "Eliminados lista negra", "Valor": int(acumulado.get("eliminados_lista_negra", 0))},
        {"Indicator": "Eliminados URL/email", "Valor": int(acumulado.get("eliminados_url_email", 0))},
        {"Indicator": "Eliminados ruido", "Valor": int(acumulado.get("eliminados_ruido", 0))},
        {"Indicator": "Eliminados hashtag", "Valor": int(acumulado.get("eliminados_hashtag", 0))},
        {"Indicator": "Removed mentions", "Valor": int(acumulado.get("eliminados_mencion", 0))},
    ])
    return trabajo.reset_index(drop=True), auditoria

In [53]:
def audit_cleaned_dataset(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Audit cleaned dataset. Retains the original experimental logic and data-partition rules."""
    if "texto_limpio" not in df.columns:
        raise KeyError("The dataset must contain the `texto_limpio` column.")

    trabajo = df.copy()
    texto = trabajo["texto_limpio"].fillna("").astype(str).str.strip()
    evento = trabajo["evento"].fillna("Sin evento").astype(str).str.strip() if "evento" in trabajo.columns else pd.Series(["Sin evento"] * len(trabajo))
    location = trabajo["location"] if "location" in trabajo.columns else pd.Series([np.nan] * len(trabajo))
    tokens = trabajo["tokens_sin_stopwords"] if "tokens_sin_stopwords" in trabajo.columns else texto

    resumen = pd.DataFrame([
        {"Indicator": "Total records", "Valor": int(len(trabajo))},
        {"Indicator": "Textos únicos globales", "Valor": int(texto.nunique(dropna=False))},
        {"Indicator": "Global text duplicates", "Valor": int(texto.duplicated(keep="first").sum())},
        {"Indicator": "Unique event-and-text records", "Valor": int(trabajo.assign(_texto=texto, _evento=evento).drop_duplicates(["_evento", "_texto"]).shape[0])},
        {"Indicator": "Event-and-text duplicates", "Valor": int(trabajo.assign(_texto=texto, _evento=evento).duplicated(["_evento", "_texto"], keep="first").sum())},
        {"Indicator": "Textos vacíos", "Valor": int((texto == "").sum())},
        {"Indicator": "Tokens vacíos", "Valor": int(tokens.fillna("").astype(str).str.strip().eq("").sum())},
        {"Indicator": "Missing or empty location", "Valor": int(location.fillna("").astype(str).str.strip().eq("").sum())},
    ])

    eventos = (
        evento.value_counts(dropna=False)
        .rename_axis("Evento")
        .reset_index(name="Registros")
    )
    eventos["Porcentaje"] = eventos["Registros"] / max(len(trabajo), 1) * 100.0

    faltantes = pd.DataFrame({
        "Columna": trabajo.columns,
        "Faltantes": [int(trabajo[c].isna().sum()) for c in trabajo.columns],
        "Porcentaje_faltante": [float(trabajo[c].isna().mean() * 100.0) for c in trabajo.columns],
    }).sort_values("Porcentaje_faltante", ascending=False).reset_index(drop=True)

    longitudes = tokens.fillna("").astype(str).map(lambda x: len(parse_tokens(x)))
    longitud_tokens = pd.DataFrame([{
        "Mean": float(longitudes.mean()),
        "Mediana": float(longitudes.median()),
        "Std_dev": float(longitudes.std(ddof=1)) if len(longitudes) > 1 else 0.0,
        "Min": int(longitudes.min()) if len(longitudes) else 0,
        "P25": float(longitudes.quantile(0.25)) if len(longitudes) else 0.0,
        "P75": float(longitudes.quantile(0.75)) if len(longitudes) else 0.0,
        "Max": int(longitudes.max()) if len(longitudes) else 0,
    }])

    return {
        "resumen": resumen,
        "eventos": eventos,
        "faltantes": faltantes,
        "longitud_tokens": longitud_tokens,
    }

In [54]:
def plot_event_balance(eventos: pd.DataFrame) -> None:
    """Plot event balance. Retains the original experimental logic and data-partition rules."""
    vista = eventos.copy()
    vista["Evento"] = vista["Evento"].astype(str)
    source = ColumnDataSource(vista)
    p = figure(
        x_range=vista["Evento"].tolist(),
        height=360,
        width=850,
        title="Tweets by political event",
        toolbar_location="above",
    )
    p.vbar(x="Evento", top="Registros", width=0.65, source=source)
    p.add_tools(HoverTool(tooltips=[("Evento", "@Evento"), ("Registros", "@Registros{0,0}"), ("Porcentaje", "@Porcentaje{0.00}%")]))
    p.xgrid.grid_line_color = None
    p.xaxis.major_label_orientation = 0.5
    show(p)

In [55]:
def prepare_cleaned_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare cleaned dataset. Retains the original experimental logic and data-partition rules."""
    trabajo = df.copy()
    if "texto_limpio" not in trabajo.columns:
        raise KeyError("The `texto_limpio` field is missing after cleaning.")

    trabajo["texto_limpio"] = (
        trabajo["texto_limpio"].fillna("").astype(str).str.strip()
    )
    trabajo = trabajo[trabajo["texto_limpio"] != ""].copy()

    subconjunto = ["texto_limpio"]
    if "evento" in trabajo.columns:
        subconjunto = ["evento", "texto_limpio"]

    trabajo = (
        trabajo
        .drop_duplicates(subset=subconjunto, keep="first")
        .reset_index(drop=True)
    )

    if "id_registro" not in trabajo.columns:
        trabajo["id_registro"] = create_record_id(trabajo)

    return trabajo

In [56]:
def plot_preprocessing_trace(trazabilidad: pd.DataFrame, titulo: str) -> None:
    """Plot preprocessing trace. Retains the original experimental logic and data-partition rules."""
    df = trazabilidad.copy()
    etapa_col = df.columns[0]
    etapas = df[etapa_col].astype(str).tolist()
    p = figure(
        x_range=etapas,
        width=1000,
        height=420,
        title=titulo,
        toolbar_location="above",
    )
    num_cols = [c for c in df.columns[1:] if pd.api.types.is_numeric_dtype(df[c])]
    colores = Category10[10]
    for i, col_name in enumerate(num_cols):
        p.line(etapas, df[col_name].astype(float), line_width=2, legend_label=str(col_name), color=colores[i % len(colores)])
        p.scatter(etapas, df[col_name].astype(float), size=7, color=colores[i % len(colores)])
    p.xaxis.major_label_orientation = math.pi / 3
    p.xaxis.axis_label = "Etapa"
    p.yaxis.axis_label = "Cantidad"
    p.legend.click_policy = "hide"
    show(p)

In [57]:
def plot_top_tokens(top_tokens: pd.DataFrame, top_n: int = 50) -> None:
    """Plot top tokens. Retains the original experimental logic and data-partition rules."""
    df = top_tokens.iloc[:, :2].copy()
    df.columns = ["Token", "Frecuencia"]
    df = df.sort_values("Frecuencia", ascending=False).head(top_n)
    factores = df["Token"].astype(str).tolist()[::-1]
    valores = df["Frecuencia"].astype(float).tolist()[::-1]
    source = ColumnDataSource({"token": factores, "frecuencia": valores})
    p = figure(y_range=factores, width=1000, height=max(450, 15 * len(factores)), title=f"Top {len(factores)} tokens por frecuencia")
    p.hbar(y="token", right="frecuencia", height=0.75, source=source)
    p.xaxis.axis_label = "Frecuencia"
    p.yaxis.axis_label = "Token"
    p.add_tools(HoverTool(tooltips=[("Token", "@token"), ("Frecuencia", "@frecuencia{0,0}")]))
    show(p)

In [58]:
def plot_location_availability(tweets: pd.DataFrame, columna: str = "location") -> pd.DataFrame:
    """Plot location availability. Retains the original experimental logic and data-partition rules."""
    ubicacion = tweets[columna].fillna("").astype(str).str.strip()
    con = int((ubicacion != "").sum())
    sin = int((ubicacion == "").sum())
    total = max(con + sin, 1)
    df = pd.DataFrame({
        "Categoria": ["Location available", "No location"],
        "Cantidad": [con, sin],
        "Porcentaje": [100 * con / total, 100 * sin / total],
    })
    df["angulo"] = df["Cantidad"] / total * 2 * math.pi
    df["inicio"] = np.cumsum([0.0] + df["angulo"].tolist()[:-1])
    df["fin"] = np.cumsum(df["angulo"].tolist())
    df["color"] = [Category10[10][0], Category10[10][1]]
    source = ColumnDataSource(df)
    p = figure(width=650, height=420, title="Tweets with and without location metadata", toolbar_location=None, tools="hover", tooltips="@Categoria: @Porcentaje{0.00}%")
    p.wedge(x=0, y=1, radius=0.35, start_angle="inicio", end_angle="fin", color="color", legend_field="Categoria", source=source)
    p.axis.visible = False
    p.grid.grid_line_color = None
    show(p)
    return df.drop(columns=["angulo", "inicio", "fin", "color"])

In [59]:
def calculate_trend_frequency(tweets: pd.DataFrame, trends: pd.DataFrame, max_ngram: int = 4) -> pd.DataFrame:
    """Calculate trend frequency. Retains the original experimental logic and data-partition rules."""
    if "texto_limpio" not in tweets.columns or "trend_limpio" not in trends.columns:
        raise KeyError("The `texto_limpio` and `trend_limpio` columns are required.")

    candidatos = []
    for trend in trends["trend_limpio"].dropna().astype(str).str.strip().str.lower():
        n = len(trend.split())
        if trend and 1 <= n <= max_ngram:
            candidatos.append(trend)
    candidatos = list(dict.fromkeys(candidatos))
    vocabulario = {term: i for i, term in enumerate(candidatos)}

    vectorizador = CountVectorizer(
        vocabulary=vocabulario,
        ngram_range=(1, max_ngram),
        binary=True,
        lowercase=True,
        token_pattern=r"(?u)[#@]?\b\w+\b",
    )
    matriz = vectorizador.transform(tweets["texto_limpio"].fillna("").astype(str))
    frecuencias = np.asarray(matriz.sum(axis=0)).ravel()
    total = max(len(tweets), 1)
    salida = pd.DataFrame({
        "Trend": candidatos,
        "Frecuencia": frecuencias.astype(int),
        "Tweet_percentage": 100 * frecuencias / total,
    })
    return salida.sort_values(["Frecuencia", "Trend"], ascending=[False, True]).reset_index(drop=True)

In [60]:
def plot_top_trends(frecuencias: pd.DataFrame, top_n: int = 30) -> None:
    """Plot top trends. Retains the original experimental logic and data-partition rules."""
    df = frecuencias.head(top_n).iloc[::-1].copy()
    source = ColumnDataSource(df)
    p = figure(y_range=df["Trend"].astype(str).tolist(), width=1000, height=max(450, 18 * len(df)), title=f"Top {len(df)} trends por presencia en tweets")
    p.hbar(y="Trend", right="Frecuencia", height=0.7, source=source)
    p.xaxis.axis_label = "Tweets containing the trend"
    p.add_tools(HoverTool(tooltips=[("Trend", "@Trend"), ("Frecuencia", "@Frecuencia{0,0}"), ("Porcentaje", "@Porcentaje_tweets{0.000}%")]))
    show(p)

In [61]:
rutas = verify_input_files(ARCHIVOS, DATA_DIR)
datos = load_input_datasets(rutas)

In [62]:
tweets = datos["tweets"].reset_index(drop=True)
trends = datos["trends"].reset_index(drop=True)
paro = datos["paro"].reset_index(drop=True)
muerte = datos["muerte"].reset_index(drop=True)
resumen_preprocesamiento = datos["resumen"]

In [63]:
tweets = assign_stable_ids_before_cleaning(tweets)

In [64]:
nlp_es = load_spanish_spacy_model(MODELO_SPACY_ES)
print(f"Spanish-language preprocessing model loaded: {MODELO_SPACY_ES}")

Spanish-language preprocessing model loaded: es_core_news_sm


In [65]:
tweets, AUDITORIA_LIMPIEZA_TWEETS = clean_dataset_with_spacy(
    tweets,
    nlp_es,
    ["texto_original", "texto", "full_text", "text", "texto_limpio"],
    PALABRAS_INCLUIDAS,
    PALABRAS_EXCLUIDAS,
    POS_EXCLUIDOS,
    LONGITUD_MINIMA_TOKEN,
    SPACY_BATCH_SIZE,
    "tweets",
)

In [66]:
paro, AUDITORIA_LIMPIEZA_PARO = clean_dataset_with_spacy(
    paro,
    nlp_es,
    ["texto_original", "texto", "full_text", "text", "texto_limpio"],
    PALABRAS_INCLUIDAS,
    PALABRAS_EXCLUIDAS,
    POS_EXCLUIDOS,
    LONGITUD_MINIMA_TOKEN,
    SPACY_BATCH_SIZE,
    "evento_paro",
)

In [67]:
muerte, AUDITORIA_LIMPIEZA_MUERTE = clean_dataset_with_spacy(
    muerte,
    nlp_es,
    ["texto_original", "texto", "full_text", "text", "texto_limpio"],
    PALABRAS_INCLUIDAS,
    PALABRAS_EXCLUIDAS,
    POS_EXCLUIDOS,
    LONGITUD_MINIMA_TOKEN,
    SPACY_BATCH_SIZE,
    "evento_muerte_cruzada",
)

In [68]:
trends, AUDITORIA_LIMPIEZA_TRENDS = clean_dataset_with_spacy(
    trends,
    nlp_es,
    ["trend", "trend_original", "trend_limpio"],
    PALABRAS_INCLUIDAS,
    PALABRAS_EXCLUIDAS,
    POS_EXCLUIDOS,
    LONGITUD_MINIMA_TOKEN,
    SPACY_BATCH_SIZE,
    "trends",
)

In [69]:
AUDITORIA_LIMPIEZA = pd.concat(
    [
        AUDITORIA_LIMPIEZA_TWEETS.assign(Origen="tweets"),
        AUDITORIA_LIMPIEZA_PARO.assign(Origen="paro"),
        AUDITORIA_LIMPIEZA_MUERTE.assign(Origen="muerte"),
        AUDITORIA_LIMPIEZA_TRENDS.assign(Origen="trends"),
    ],
    ignore_index=True,
)

tweets_revision = prepare_cleaned_dataset(tweets)

In [70]:
AUDITORIA_DATASET = audit_cleaned_dataset(tweets_revision)

print(f"Consolidated tweets loaded and cleaned: {len(tweets):,}")
print(f"Final corpus after cleaning and deduplication: {len(tweets_revision):,}")
print(f"Trends after cleaning: {len(trends):,}")
print(f"National Strike after cleaning: {len(paro):,}")
print(f"Muerte Cruzada after cleaning: {len(muerte):,}")
print(f"Whitelist: {sorted(PALABRAS_INCLUIDAS)}")
print(f"Blacklist: {sorted(PALABRAS_EXCLUIDAS)}")
print(f"Excluded POS tags: {sorted(POS_EXCLUIDOS)}")
print(f"Minimum token length: {LONGITUD_MINIMA_TOKEN}")

Consolidated tweets loaded and cleaned: 144,704
Final corpus after cleaning and deduplication: 138,400
Trends after cleaning: 1,161
National Strike after cleaning: 29,642
Muerte Cruzada after cleaning: 115,062
Whitelist: ['ec']
Blacklist: ['jajaja']
Excluded POS tags: ['AUX', 'DET', 'VERB']
Minimum token length: 3


In [71]:
display_bokeh_table(
    AUDITORIA_LIMPIEZA,
    "Language preprocessing audit",
    ancho=1300,
    alto=520,
    max_filas=80,
)

In [72]:
display_bokeh_table(
    validate_event_consistency(tweets, paro, muerte),
    "Consistency of consolidated corpus and event files after cleaning",
    alto=180,
)

In [73]:
display_bokeh_table(
    AUDITORIA_DATASET["resumen"],
    "Cleaned and deduplicated corpus audit",
    alto=300,
)

In [74]:
display_bokeh_table(
    AUDITORIA_DATASET["eventos"],
    "Event distribution after cleaning and deduplication",
    alto=200,
)

In [75]:
display_bokeh_table(
    AUDITORIA_DATASET["faltantes"],
    "Missing values after cleaning and deduplication",
    alto=380,
)

In [76]:
display_bokeh_table(
    AUDITORIA_DATASET["longitud_tokens"],
    "Tweet length after cleaning and deduplication",
    alto=170,
)

In [77]:
plot_event_balance(AUDITORIA_DATASET["eventos"])

In [78]:
resumen_ubicacion = plot_location_availability(tweets_revision)

In [79]:
display_bokeh_table(
    resumen_ubicacion,
    "Location metadata availability after cleaning",
    alto=170,
)

In [80]:
frecuencias_trends = calculate_trend_frequency(
    tweets_revision,
    trends,
    MAX_NGRAM_TREND,
)

In [81]:
plot_top_trends(frecuencias_trends, 30)

In [82]:
display_bokeh_table(
    frecuencias_trends,
    "Trend frequency after cleaning",
    alto=340,
    max_filas=30,
)

# 4. External references and human annotation preparation

In [83]:
def normalize_hashtag(hashtag: str) -> str:
    """Normalize hashtag. Retains the original experimental logic and data-partition rules."""
    valor = str(hashtag).strip().lower()
    if not valor:
        return ""
    return valor if valor.startswith("#") else f"#{valor}"

In [84]:
def extract_text_hashtags(texto: str) -> list[str]:
    """Extract text hashtags. Retains the original experimental logic and data-partition rules."""
    encontrados = re.findall(r"(?<!\w)#[\wáéíóúüñÁÉÍÓÚÜÑ]+", str(texto).lower())
    return list(dict.fromkeys(normalize_hashtag(x) for x in encontrados if x))

In [85]:
def build_hashtag_inventory(df: pd.DataFrame, hashtags_politicos: set[str], hashtags_no_politicos: set[str]) -> pd.DataFrame:
    """Build hashtag inventory. Retains the original experimental logic and data-partition rules."""
    politicos = {normalize_hashtag(x) for x in hashtags_politicos}
    no_politicos = {normalize_hashtag(x) for x in hashtags_no_politicos}
    contador_total: Counter[str] = Counter()
    contador_evento: dict[str, Counter[str]] = {}

    for _, fila in df.iterrows():
        evento = str(fila.get("evento", "Sin evento"))
        columna_hash = "texto_para_hashtags" if "texto_para_hashtags" in df.columns else "texto_limpio"
        hashtags = extract_text_hashtags(fila.get(columna_hash, ""))
        contador_total.update(hashtags)
        contador_evento.setdefault(evento, Counter()).update(hashtags)

    filas: list[dict[str, Any]] = []
    eventos = sorted(contador_evento)
    for hashtag, frecuencia in contador_total.most_common():
        if hashtag in politicos:
            categoria = "Político estricto"
        elif hashtag in no_politicos:
            categoria = "Strict non-political"
        else:
            categoria = "Unclassified"
        registro: dict[str, Any] = {
            "Hashtag": hashtag,
            "Frecuencia": int(frecuencia),
            "Categoria_silver": categoria,
        }
        for evento in eventos:
            registro[f"Frecuencia_{evento}"] = int(contador_evento[evento].get(hashtag, 0))
        filas.append(registro)

    return pd.DataFrame(filas)

In [86]:
def assign_hashtag_silver_label(texto: str, hashtags_politicos: set[str], hashtags_no_politicos: set[str]) -> tuple[int | None, str, list[str], list[str], list[str]]:
    """Assign hashtag silver label. Retains the original experimental logic and data-partition rules."""
    politicos = {normalize_hashtag(x) for x in hashtags_politicos}
    no_politicos = {normalize_hashtag(x) for x in hashtags_no_politicos}
    hashtags = extract_text_hashtags(texto)
    encontrados_politicos = sorted(set(hashtags).intersection(politicos))
    encontrados_no_politicos = sorted(set(hashtags).intersection(no_politicos))

    if encontrados_politicos and not encontrados_no_politicos:
        return 1, "Strict political hashtag", hashtags, encontrados_politicos, encontrados_no_politicos
    if encontrados_no_politicos and not encontrados_politicos:
        return 0, "Strict non-political hashtag", hashtags, encontrados_politicos, encontrados_no_politicos
    if encontrados_politicos and encontrados_no_politicos:
        return None, "Ambiguous: contains signals from both classes", hashtags, encontrados_politicos, encontrados_no_politicos
    return None, "No classifiable strict hashtag", hashtags, encontrados_politicos, encontrados_no_politicos

In [87]:
def build_hashtag_silver_standard(df: pd.DataFrame, hashtags_politicos: set[str], hashtags_no_politicos: set[str]) -> pd.DataFrame:
    """Build hashtag silver standard. Retains the original experimental logic and data-partition rules."""
    if "id_registro" not in df.columns:
        raise KeyError("The dataset needs `id_registro`. Run prepare_cleaned_dataset first.")

    filas: list[dict[str, Any]] = []
    for _, fila in df.iterrows():
        etiqueta, motivo, hashtags, politicos, no_politicos = assign_hashtag_silver_label(
            fila.get("texto_para_hashtags", fila.get("texto_limpio", "")),
            hashtags_politicos,
            hashtags_no_politicos,
        )
        if etiqueta is None:
            continue
        registro = fila.to_dict()
        registro.update({
            "etiqueta_silver_hashtag": int(etiqueta),
            "motivo_silver": motivo,
            "hashtags_detectados": " ".join(hashtags),
            "hashtags_politicos": " ".join(politicos),
            "hashtags_no_politicos": " ".join(no_politicos),
        })
        filas.append(registro)

    resultado = pd.DataFrame(filas)
    if resultado.empty:
        raise ValueError("No silver labels were generated from the configured hashtags.")
    return resultado.reset_index(drop=True)

In [88]:
def remove_text_hashtags(texto: str) -> str:
    """Remove text hashtags. Retains the original experimental logic and data-partition rules."""
    limpio = re.sub(r"(?<!\w)#[\wáéíóúüñÁÉÍÓÚÜÑ]+", " ", str(texto))
    limpio = limpio.replace("#", " ")
    return re.sub(r"\s+", " ", limpio).strip()

In [89]:
def prepare_hashtag_blind_corpus(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare hashtag blind corpus. Retains the original experimental logic and data-partition rules."""
    trabajo = df.copy()
    for columna in ["texto_limpio", "texto", "texto_original", "tokens", "tokens_sin_stopwords"]:
        if columna in trabajo.columns:
            trabajo[columna] = trabajo[columna].fillna("").astype(str).map(remove_text_hashtags)
    if "tokens_sin_stopwords" in trabajo.columns:
        trabajo["cantidad_tokens_sin_stopwords"] = trabajo["tokens_sin_stopwords"].map(lambda x: len(parse_tokens(x)))
    return trabajo

In [90]:
def separate_human_annotation_holdout(df: pd.DataFrame, n_muestra: int, seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Separate human annotation holdout. Retains the original experimental logic and data-partition rules."""
    if n_muestra <= 0 or n_muestra >= len(df):
        raise ValueError("The audit sample size must be positive and smaller than the number of records.")

    estrato = df["evento"].fillna("Sin evento").astype(str)
    desarrollo, holdout = train_test_split(
        df,
        test_size=int(n_muestra),
        random_state=seed,
        stratify=estrato,
    )
    return desarrollo.reset_index(drop=True), holdout.reset_index(drop=True)

In [91]:
def save_human_annotation_template(
    holdout: pd.DataFrame,
    ruta: Path,
    version_protocolo: str,
) -> Path:
    """Save human annotation template. Retains the original experimental logic and data-partition rules."""
    ruta.parent.mkdir(parents=True, exist_ok=True)

    if "id_registro" not in holdout.columns:
        raise KeyError(
            "The holdout is missing the mandatory `id_registro` column."
        )

    ids_esperados = set(
        holdout["id_registro"].astype(str).str.strip()
    )

    if ruta.exists():
        try:
            with pd.ExcelFile(ruta, engine="openpyxl") as xls:
                if "Anotacion" not in xls.sheet_names:
                    existente = pd.DataFrame()
                else:
                    existente = pd.read_excel(
                        xls,
                        sheet_name="Anotacion",
                    )

                if "id_registro" in existente.columns:
                    ids_existentes = set(
                        existente["id_registro"]
                        .astype(str)
                        .str.strip()
                    )
                else:
                    ids_existentes = set()

                columnas_etiquetas = [
                    columna
                    for columna in [
                        "etiqueta_anotador_1",
                        "etiqueta_anotador_2",
                        "etiqueta_consenso",
                    ]
                    if columna in existente.columns
                ]

                tiene_anotaciones = bool(
                    columnas_etiquetas
                    and existente[columnas_etiquetas]
                    .notna()
                    .any()
                    .any()
                )

                version_ok = False
                if "Metadatos" in xls.sheet_names:
                    metadatos_existentes = pd.read_excel(
                        xls,
                        sheet_name="Metadatos",
                    )
                    if {
                        "Clave",
                        "Valor",
                    }.issubset(metadatos_existentes.columns):
                        mapa_metadatos = dict(
                            zip(
                                metadatos_existentes["Clave"].astype(str),
                                metadatos_existentes["Valor"].astype(str),
                            )
                        )
                        version_ok = (
                            mapa_metadatos.get("version_protocolo")
                            == str(version_protocolo)
                        )

        except PermissionError as exc:
            raise PermissionError(
                "\nCould not read the annotation workbook:\n"
                f"{ruta}\n\n"
                "Close Microsoft Excel, LibreOffice and the file preview panel "
                "in the file browser, then rerun the cell."
            ) from exc

        if ids_existentes == ids_esperados and version_ok:
            print(
                "Existing annotation workbook matches the current protocol. "
                "Existing annotations will be preserved."
            )
            return ruta

        if tiene_anotaciones:
            raise RuntimeError(
                "\nAn existing human annotation workbook was found:\n"
                f"{ruta}\n\n"
                "It will not be overwritten automatically. Back it up manually "
                "before generating an incompatible workbook."
            )

        respaldo = ruta.with_name(
            f"{ruta.stem}_backup_"
            f"{time.strftime('%Y%m%d_%H%M%S')}"
            f"{ruta.suffix}"
        )

        try:
            ruta.replace(respaldo)
        except PermissionError as exc:
            raise PermissionError(
                "\nCould not back up the previous annotation workbook:\n"
                f"{ruta}\n\n"
                "The file is locked. Close the application, "
                "disable the preview panel and ensure "
                "that no application holds the workbook open."
            ) from exc

        print(f"Unlabeled previous annotation workbook backed up to: {respaldo}")

    columnas = [
        columna
        for columna in [
            "id_registro",
            "evento",
            "texto_para_anotacion",
            "texto_limpio",
            "location",
        ]
        if columna in holdout.columns
    ]

    plantilla = holdout[columnas].copy()
    plantilla = plantilla.rename(
        columns={
            "texto_para_anotacion":
            "texto_original_para_anotar"
        }
    )

    plantilla["etiqueta_anotador_1"] = pd.NA
    plantilla["etiqueta_anotador_2"] = pd.NA
    plantilla["etiqueta_consenso"] = pd.NA
    plantilla["observacion"] = ""

    instrucciones = pd.DataFrame(
        {
            "Instrucción": [
                "Read `texto_original_para_anotar` and judge the tweet without relying on the event label.",
                "Annotator 1 fills only `etiqueta_anotador_1` with 1=political or 0=non-political.",
                "Annotator 2 fills only `etiqueta_anotador_2` with 1=political or 0=non-political.",
                "Both annotators must work independently.",
                "Do not fill initially `etiqueta_consenso`.",
                "After both primary annotations, run the notebook's consensus stage.",
                "When both annotators agree, the notebook automatically fills `etiqueta_consenso`.",
                "When the annotators disagree, the case appears in the `Desacuerdos_Adjudicar` sheet.",
                "A third adjudicator, makes the final 0/1 decision for disagreements.",
                "Do not modify `id_registro`.",
            ]
        }
    )

    metadatos = pd.DataFrame(
        {
            "Clave": [
                "version_protocolo",
                "tipo_muestra",
                "n_registros",
                "semilla",
                "politico",
                "no_politico",
            ],
            "Valor": [
                str(version_protocolo),
                "holdout humano independiente previo al modelado",
                str(len(plantilla)),
                str(RANDOM_STATE),
                "1",
                "0",
            ],
        }
    )

    try:
        with pd.ExcelWriter(
            ruta,
            engine="openpyxl",
            mode="w",
        ) as writer:
            plantilla.to_excel(
                writer,
                sheet_name="Anotacion",
                index=False,
            )
            instrucciones.to_excel(
                writer,
                sheet_name="Instrucciones",
                index=False,
            )
            metadatos.to_excel(
                writer,
                sheet_name="Metadatos",
                index=False,
            )
    except PermissionError as exc:
        raise PermissionError(
            "\nCould not create the annotation template:\n"
            f"{ruta}\n\n"
            "Close Excel and any other application using the file."
        ) from exc

    print(f"Human annotation template created: {ruta}")
    return ruta

In [92]:
def load_human_annotations(ruta: Path) -> pd.DataFrame:
    """Load human annotations. Retains the original experimental logic and data-partition rules."""
    if not ruta.exists():
        raise FileNotFoundError(f"No existe el archivo de anotación: {ruta}")

    anotaciones = pd.read_excel(ruta, sheet_name="Anotacion")
    requeridas = {"id_registro", "etiqueta_anotador_1", "etiqueta_anotador_2", "etiqueta_consenso"}
    faltantes = requeridas.difference(anotaciones.columns)
    if faltantes:
        raise KeyError(f"Faltan columnas en la plantilla: {sorted(faltantes)}")

    for columna in ["etiqueta_anotador_1", "etiqueta_anotador_2", "etiqueta_consenso"]:
        valores = pd.to_numeric(anotaciones[columna], errors="coerce")
        invalidos = valores.dropna()[~valores.dropna().isin([0, 1])]
        if not invalidos.empty:
            raise ValueError(f"La columna {columna} contiene etiquetas distintas de 0/1.")
        anotaciones[columna] = valores.astype("Int64")

    return anotaciones

In [93]:
def calculate_annotator_agreement(anotaciones: pd.DataFrame) -> pd.DataFrame:
    """Calculate annotator agreement. Retains the original experimental logic and data-partition rules."""
    pares = anotaciones.dropna(subset=["etiqueta_anotador_1", "etiqueta_anotador_2"]).copy()
    if pares.empty:
        return pd.DataFrame([{"N_pares": 0, "Acuerdo_porcentual": np.nan, "Cohen_kappa": np.nan}])

    y1 = pares["etiqueta_anotador_1"].astype(int).to_numpy()
    y2 = pares["etiqueta_anotador_2"].astype(int).to_numpy()
    acuerdo = float(np.mean(y1 == y2) * 100.0)
    kappa = float(cohen_kappa_score(y1, y2)) if len(np.unique(np.concatenate([y1, y2]))) > 1 else np.nan
    return pd.DataFrame([{"N_pares": int(len(pares)), "Acuerdo_porcentual": acuerdo, "Cohen_kappa": kappa}])

In [94]:
def update_consensus_worksheet(
    ruta: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Update consensus worksheet. Retains the original experimental logic and data-partition rules."""
    if not ruta.exists():
        raise FileNotFoundError(
            f"No existe el archivo de anotaciones: {ruta}"
        )

    try:
        with pd.ExcelFile(ruta, engine="openpyxl") as xls:
            hojas = {
                nombre: pd.read_excel(
                    xls,
                    sheet_name=nombre,
                )
                for nombre in xls.sheet_names
            }
    except PermissionError as exc:
        raise PermissionError(
            "\nThe workbook is open or locked and cannot be read:\n"
            f"{ruta}\n\n"
            "Close Excel and rerun this cell."
        ) from exc

    if "Anotacion" not in hojas:
        raise KeyError(
            "The workbook is missing the mandatory `Anotacion` sheet."
        )

    anotaciones = hojas["Anotacion"].copy()

    requeridas = {
        "id_registro",
        "etiqueta_anotador_1",
        "etiqueta_anotador_2",
        "etiqueta_consenso",
    }
    faltantes = requeridas.difference(anotaciones.columns)
    if faltantes:
        raise KeyError(
            f"Faltan columnas requeridas: {sorted(faltantes)}"
        )

    for columna in [
        "etiqueta_anotador_1",
        "etiqueta_anotador_2",
        "etiqueta_consenso",
    ]:
        valores = pd.to_numeric(
            anotaciones[columna],
            errors="coerce",
        )
        invalidos = valores.dropna()[
            ~valores.dropna().isin([0, 1])
        ]
        if not invalidos.empty:
            raise ValueError(
                f"La columna `{columna}` contiene valores distintos de 0/1."
            )
        anotaciones[columna] = valores.astype("Int64")

    # Restore adjudications previously saved on the dedicated worksheet.
    adjudicacion_previa = hojas.get(
        "Desacuerdos_Adjudicar",
        pd.DataFrame(),
    ).copy()

    if (
        not adjudicacion_previa.empty
        and {
            "id_registro",
            "etiqueta_consenso",
        }.issubset(adjudicacion_previa.columns)
    ):
        adjudicacion_previa["etiqueta_consenso"] = pd.to_numeric(
            adjudicacion_previa["etiqueta_consenso"],
            errors="coerce",
        ).astype("Int64")

        mapa_adjudicado = (
            adjudicacion_previa
            .dropna(subset=["etiqueta_consenso"])
            .set_index("id_registro")["etiqueta_consenso"]
            .to_dict()
        )

        for indice, identificador in anotaciones["id_registro"].items():
            if identificador in mapa_adjudicado:
                anotaciones.at[
                    indice,
                    "etiqueta_consenso",
                ] = int(mapa_adjudicado[identificador])

    ambos_completos = (
        anotaciones["etiqueta_anotador_1"].notna()
        & anotaciones["etiqueta_anotador_2"].notna()
    )

    coinciden = (
        ambos_completos
        & (
            anotaciones["etiqueta_anotador_1"]
            == anotaciones["etiqueta_anotador_2"]
        )
    )

    desacuerdan = (
        ambos_completos
        & (
            anotaciones["etiqueta_anotador_1"]
            != anotaciones["etiqueta_anotador_2"]
        )
    )

    # Agreements produce direct, reproducible consensus labels.
    anotaciones.loc[
        coinciden,
        "etiqueta_consenso",
    ] = anotaciones.loc[
        coinciden,
        "etiqueta_anotador_1",
    ]

    consenso_valido = anotaciones["etiqueta_consenso"].isin([0, 1])

    anotaciones["requiere_adjudicacion"] = desacuerdan
    anotaciones["estado_consenso"] = "ANOTACION_INCOMPLETA"

    anotaciones.loc[
        coinciden,
        "estado_consenso",
    ] = "ACUERDO_AUTOMATICO"

    anotaciones.loc[
        desacuerdan & consenso_valido,
        "estado_consenso",
    ] = "ADJUDICADO"

    anotaciones.loc[
        desacuerdan & ~consenso_valido,
        "estado_consenso",
    ] = "PENDIENTE_ADJUDICACION"

    pendientes = anotaciones.loc[
        desacuerdan & ~consenso_valido
    ].copy()

    columnas_adjudicacion = [
        columna
        for columna in [
            "id_registro",
            "evento",
            "texto_original_para_anotar",
            "texto_limpio",
            "location",
            "etiqueta_anotador_1",
            "etiqueta_anotador_2",
            "etiqueta_consenso",
        ]
        if columna in anotaciones.columns
    ]

    desacuerdos_todos = anotaciones.loc[
        desacuerdan,
        columnas_adjudicacion,
    ].copy()

    # Preserve the adjudicator and existing notes.
    for columna in [
        "adjudicado_por",
        "observacion_adjudicacion",
    ]:
        if columna not in desacuerdos_todos.columns:
            desacuerdos_todos[columna] = ""

    if (
        not adjudicacion_previa.empty
        and "id_registro" in adjudicacion_previa.columns
    ):
        extras = [
            columna
            for columna in [
                "id_registro",
                "adjudicado_por",
                "observacion_adjudicacion",
            ]
            if columna in adjudicacion_previa.columns
        ]
        if len(extras) > 1:
            prev = adjudicacion_previa[extras].copy()
            desacuerdos_todos = (
                desacuerdos_todos
                .drop(
                    columns=[
                        c for c in [
                            "adjudicado_por",
                            "observacion_adjudicacion",
                        ]
                        if c in desacuerdos_todos.columns
                    ],
                    errors="ignore",
                )
                .merge(
                    prev,
                    on="id_registro",
                    how="left",
                )
            )
            for columna in [
                "adjudicado_por",
                "observacion_adjudicacion",
            ]:
                if columna not in desacuerdos_todos.columns:
                    desacuerdos_todos[columna] = ""
                desacuerdos_todos[columna] = (
                    desacuerdos_todos[columna]
                    .fillna("")
                )

    desacuerdos_todos["instruccion"] = (
        "Fill `etiqueta_consenso` with 1=political or 0=non-political. "
        "The adjudicator and adjudication notes may also be recorded."
    )

    hojas["Anotacion"] = anotaciones
    hojas["Desacuerdos_Adjudicar"] = desacuerdos_todos

    instrucciones_actualizadas = pd.DataFrame(
        {
            "Instrucción": [
                "First stage: each annotator fills only their own 0/1 column.",
                "Do not manually duplicate agreement labels in `etiqueta_consenso`.",
                "Run the notebook's automatic consensus stage.",
                "Agreed labels are copied automatically into `etiqueta_consenso`.",
                "Disagreements appear on the `Desacuerdos_Adjudicar` worksheet.",
                "A third adjudicator is preferred for independent adjudication of disagreements.",
                "On `Desacuerdos_Adjudicar`, fill `etiqueta_consenso` only for unresolved cases.",
                "Save and close the workbook, then rerun the notebook cell.",
                "The pipeline will not proceed while unresolved disagreements remain.",
                "Do not modify id_registro.",
            ]
        }
    )
    hojas["Instrucciones"] = instrucciones_actualizadas

    try:
        with pd.ExcelWriter(
            ruta,
            engine="openpyxl",
            mode="w",
        ) as writer:
            for nombre_hoja, dataframe in hojas.items():
                dataframe.to_excel(
                    writer,
                    sheet_name=nombre_hoja,
                    index=False,
                )
    except PermissionError as exc:
        raise PermissionError(
            "\nCould not update the annotation workbook:\n"
            f"{ruta}\n\n"
            "Close Excel and rerun this cell."
        ) from exc

    print(
        f"Automatic consensus agreements: {int(coinciden.sum()):,}"
    )
    print(
        f"Total annotator disagreements: {int(desacuerdan.sum()):,}"
    )
    print(
        f"Unresolved adjudication disagreements: {len(pendientes):,}"
    )

    return anotaciones, pendientes

In [95]:
def calculate_silver_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """Calculate silver metrics. Retains the original experimental logic and data-partition rules."""
    resultado = calculate_binary_metrics(np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int))
    resultado.update({
        "Balanced_Accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "Cohen_Kappa": float(cohen_kappa_score(y_true, y_pred)),
        "MCC": float(matthews_corrcoef(y_true, y_pred)),
    })
    return resultado

In [96]:
def exact_mcnemar_test(y_true: np.ndarray, pred_a: np.ndarray, pred_b: np.ndarray) -> pd.DataFrame:
    """Execute exact mcnemar test. Retains the original experimental logic and data-partition rules."""
    verdadero = np.asarray(y_true).astype(int)
    a = np.asarray(pred_a).astype(int)
    b = np.asarray(pred_b).astype(int)
    acierto_a = a == verdadero
    acierto_b = b == verdadero
    a_si_b_no = int(np.sum(acierto_a & ~acierto_b))
    a_no_b_si = int(np.sum(~acierto_a & acierto_b))
    discordantes = a_si_b_no + a_no_b_si
    p_valor = 1.0 if discordantes == 0 else float(binomtest(min(a_si_b_no, a_no_b_si), n=discordantes, p=0.5, alternative="two-sided").pvalue)
    return pd.DataFrame([{
        "A_acierta_B_falla": a_si_b_no,
        "A_falla_B_acierta": a_no_b_si,
        "Discordantes": discordantes,
        "Diferencia": int(a_no_b_si - a_si_b_no),
        "p_valor_exacto": p_valor,
    }])

In [97]:
def plot_hashtag_confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, titulo: str) -> None:
    """Plot hashtag confusion matrix. Retains the original experimental logic and data-partition rules."""
    matriz = confusion_matrix(y_true, y_pred, labels=[0, 1])
    xs = ["Pred 0", "Pred 1", "Pred 0", "Pred 1"]
    ys = ["Real 0", "Real 0", "Real 1", "Real 1"]
    valores = [int(matriz[0, 0]), int(matriz[0, 1]), int(matriz[1, 0]), int(matriz[1, 1])]
    source = ColumnDataSource({"x": xs, "y": ys, "valor": valores, "texto": [str(v) for v in valores]})
    p = figure(x_range=["Pred 0", "Pred 1"], y_range=["Real 1", "Real 0"], width=520, height=360, title=titulo, toolbar_location=None)
    p.rect(x="x", y="y", width=1, height=1, source=source, fill_alpha=0.55, line_color="white")
    p.text(x="x", y="y", text="texto", source=source, text_align="center", text_baseline="middle", text_font_size="16px")
    p.xaxis.axis_label = "Predicción"
    p.yaxis.axis_label = "Hashtag-based silver label"
    show(p)

In [98]:
def sample_hashtag_audit(
    silver: pd.DataFrame,
    n_muestra: int,
    seed: int,
) -> pd.DataFrame:
    """Sample hashtag audit. Retains the original experimental logic and data-partition rules."""
    if "etiqueta_silver_hashtag" not in silver.columns:
        raise KeyError("The silver reference is missing `etiqueta_silver_hashtag`.")

    n = min(int(n_muestra), len(silver))
    if n <= 0:
        raise ValueError("No records are available for the hashtag audit.")

    if n == len(silver):
        return silver.copy().reset_index(drop=True)

    etiqueta = silver["etiqueta_silver_hashtag"].astype(str)
    if "evento" in silver.columns:
        estrato_compuesto = (
            silver["evento"].fillna("Sin evento").astype(str)
            + "||"
            + etiqueta
        )
        conteos = estrato_compuesto.value_counts()
        estrato = estrato_compuesto if int(conteos.min()) >= 2 else etiqueta
    else:
        estrato = etiqueta

    _, muestra = train_test_split(
        silver,
        test_size=n,
        random_state=int(seed),
        stratify=estrato,
    )
    return muestra.reset_index(drop=True)

In [99]:
def save_hashtag_audit_template(
    muestra: pd.DataFrame,
    ruta: Path,
    version_protocolo: str,
) -> Path:
    """Save hashtag audit template. Retains the original experimental logic and data-partition rules."""
    ruta.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if "id_registro" not in muestra.columns:
        raise KeyError(
            "The sample is missing the column "
            "obligatoria `id_registro`."
        )

    ids_esperados = set(
        muestra["id_registro"]
        .astype(str)
        .str.strip()
    )

    if ruta.exists():
        try:
            with pd.ExcelFile(
                ruta,
                engine="openpyxl",
            ) as xls:

                if "Anotacion" not in xls.sheet_names:
                    existente = pd.DataFrame()
                else:
                    existente = pd.read_excel(
                        xls,
                        sheet_name="Anotacion",
                    )

                if "id_registro" in existente.columns:
                    ids_existentes = set(
                        existente["id_registro"]
                        .astype(str)
                        .str.strip()
                    )
                else:
                    ids_existentes = set()

                tiene_anotaciones = bool(
                    "etiqueta_humana"
                    in existente.columns
                    and existente[
                        "etiqueta_humana"
                    ]
                    .notna()
                    .any()
                )

                version_ok = False

                if "Metadatos" in xls.sheet_names:

                    metadatos_existentes = pd.read_excel(
                        xls,
                        sheet_name="Metadatos",
                    )

                    if {
                        "Clave",
                        "Valor",
                    }.issubset(
                        metadatos_existentes.columns
                    ):

                        mapa_metadatos = dict(
                            zip(
                                metadatos_existentes[
                                    "Clave"
                                ].astype(str),
                                metadatos_existentes[
                                    "Valor"
                                ].astype(str),
                            )
                        )

                        version_ok = (
                            mapa_metadatos.get(
                                "version_protocolo"
                            )
                            == str(version_protocolo)
                        )

        except PermissionError as exc:
            raise PermissionError(
                "\nCould not read the annotation workbook:\n"
                f"{ruta}\n\n"
                "The workbook may be open in Microsoft Excel, "
                "LibreOffice or a file preview panel. "
                "Another process may hold the file.\n\n"
                "Close the file completely and "
                "ejecutar esta celda."
            ) from exc

        if (
            ids_existentes == ids_esperados
            and version_ok
        ):
            print(
                "Existing hashtag audit workbook is compatible "
                "with the current protocol."
            )
            return ruta

        if tiene_anotaciones:
            raise RuntimeError(
                "\nAn existing hashtag audit template was found that "
                "contains existing human annotations:\n"
                f"{ruta}\n\n"
                "The existing workbook will not be overwritten automatically. "
                "Back it up before regenerating "
                "the current protocol sample."
            )

        respaldo = ruta.with_name(
            f"{ruta.stem}_backup_"
            f"{time.strftime('%Y%m%d_%H%M%S')}"
            f"{ruta.suffix}"
        )

        try:
            ruta.replace(respaldo)

        except PermissionError as exc:
            raise PermissionError(
                "\nCould not back up the hashtag audit template "
                "anterior.\n\n"
                f"Archivo origen:\n{ruta}\n\n"
                f"Archivo de respaldo:\n{respaldo}\n\n"
                "The operating system reports that this file is "
                "in use by another process.\n\n"
                "Cierre Excel, cierre cualquier vista previa "
                "and confirm that Excel is not "
                "still running."
            ) from exc

        print(
            "Previous unlabeled hashtag audit workbook "
            "backed up to:"
        )
        print(respaldo)

    columnas = [
        columna
        for columna in [
            "id_registro",
            "evento",
            "texto_para_anotacion",
            "texto_limpio",
            "location",
        ]
        if columna in muestra.columns
    ]

    anotacion = muestra[columnas].copy()

    anotacion = anotacion.rename(
        columns={
            "texto_para_anotacion":
            "texto_original_para_anotar"
        }
    )

    anotacion["etiqueta_humana"] = pd.NA
    anotacion["observacion"] = ""

    instrucciones = pd.DataFrame(
        {
            "Instrucción": [
                (
                    "Read `texto_original_para_anotar` and label "
                    "without seeing automatic labels."
                ),
                (
                    "Label EVERY row: "
                    "1 = political, 0 = non-political."
                ),
                "Do not modify id_registro.",
                (
                    "The filename is retained for "
                    "compatibility; the sample was selected "
                    "before model development."
                ),
                (
                    "After modeling, a new worksheet will be added "
                    "`Auditoria_Modelo` with silver labels and model predictions."
                ),
            ]
        }
    )

    metadatos = pd.DataFrame(
        {
            "Clave": [
                "version_protocolo",
                "tipo_muestra",
                "n_registros",
                "semilla",
            ],
            "Valor": [
                str(version_protocolo),
                (
                    "stratified human audit of the "
                    "silver standard previa al modelado"
                ),
                str(len(anotacion)),
                str(RANDOM_STATE),
            ],
        }
    )

    try:
        with pd.ExcelWriter(
            ruta,
            engine="openpyxl",
            mode="w",
        ) as writer:

            anotacion.to_excel(
                writer,
                sheet_name="Anotacion",
                index=False,
            )

            instrucciones.to_excel(
                writer,
                sheet_name="Instrucciones",
                index=False,
            )

            metadatos.to_excel(
                writer,
                sheet_name="Metadatos",
                index=False,
            )

    except PermissionError as exc:
        raise PermissionError(
            "\nCould not create the new hashtag audit template:\n"
            f"{ruta}\n\n"
            "The file is still in use by another process. "
            "Close Excel and any other application "
            "holding the file before rerunning "
            "this function."
        ) from exc

    print(
        "Hashtag audit workbook created "
        "successfully:"
    )
    print(ruta)

    return ruta

In [100]:
def load_human_hashtag_audit(ruta: Path) -> pd.DataFrame:
    """Load human hashtag audit. Retains the original experimental logic and data-partition rules."""
    if not ruta.exists():
        raise FileNotFoundError(f"No existe el archivo de anotación hashtag: {ruta}")

    tabla = pd.read_excel(ruta, sheet_name="Anotacion")
    requeridas = {"id_registro", "etiqueta_humana"}
    faltantes = requeridas.difference(tabla.columns)
    if faltantes:
        raise KeyError(f"Faltan columnas requeridas: {sorted(faltantes)}")

    valores = pd.to_numeric(tabla["etiqueta_humana"], errors="coerce")
    invalidos = valores.dropna()[~valores.dropna().isin([0, 1])]
    if not invalidos.empty:
        raise ValueError("`etiqueta_humana` contains values other than 0 or 1.")

    tabla["etiqueta_humana"] = valores.astype("Int64")
    return tabla

In [101]:
def audit_annotation_completeness(
    ruta_manual: Path,
    ruta_hashtag: Path,
) -> pd.DataFrame:
    """Audit annotation completeness. Retains the original experimental logic and data-partition rules."""
    filas: list[dict[str, Any]] = []

    if ruta_manual.exists():
        try:
            with pd.ExcelFile(
                ruta_manual,
                engine="openpyxl",
            ) as xls:
                manual = pd.read_excel(
                    xls,
                    sheet_name="Anotacion",
                )
        except PermissionError as exc:
            raise PermissionError(
                f"Cierre el archivo antes de auditarlo: {ruta_manual}"
            ) from exc

        total = len(manual)

        for columna in [
            "etiqueta_anotador_1",
            "etiqueta_anotador_2",
        ]:
            completas = (
                int(manual[columna].notna().sum())
                if columna in manual.columns
                else 0
            )
            filas.append({
                "Archivo": ruta_manual.name,
                "Etapa": columna,
                "Total": total,
                "Completas": completas,
                "Pendientes": int(total - completas),
            })

        if {
            "etiqueta_anotador_1",
            "etiqueta_anotador_2",
        }.issubset(manual.columns):
            a1 = pd.to_numeric(
                manual["etiqueta_anotador_1"],
                errors="coerce",
            )
            a2 = pd.to_numeric(
                manual["etiqueta_anotador_2"],
                errors="coerce",
            )
            ambos = a1.notna() & a2.notna()
            acuerdos = ambos & (a1 == a2)
            desacuerdos = ambos & (a1 != a2)
        else:
            acuerdos = pd.Series(False, index=manual.index)
            desacuerdos = pd.Series(False, index=manual.index)

        consenso = (
            pd.to_numeric(
                manual["etiqueta_consenso"],
                errors="coerce",
            )
            if "etiqueta_consenso" in manual.columns
            else pd.Series(np.nan, index=manual.index)
        )

        consensos_validos = consenso.isin([0, 1])
        pendientes_adjudicacion = desacuerdos & ~consensos_validos

        filas.extend([
            {
                "Archivo": ruta_manual.name,
                "Etapa": "acuerdos_automaticos",
                "Total": int(acuerdos.sum()),
                "Completas": int(
                    (acuerdos & consensos_validos).sum()
                ),
                "Pendientes": int(
                    (acuerdos & ~consensos_validos).sum()
                ),
            },
            {
                "Archivo": ruta_manual.name,
                "Etapa": "desacuerdos_entre_anotadores",
                "Total": int(desacuerdos.sum()),
                "Completas": int(
                    (desacuerdos & consensos_validos).sum()
                ),
                "Pendientes": int(
                    pendientes_adjudicacion.sum()
                ),
            },
            {
                "Archivo": ruta_manual.name,
                "Etapa": "etiqueta_consenso_final",
                "Total": total,
                "Completas": int(consensos_validos.sum()),
                "Pendientes": int(
                    total - consensos_validos.sum()
                ),
            },
        ])
    else:
        filas.append({
            "Archivo": ruta_manual.name,
            "Etapa": "ARCHIVO",
            "Total": 0,
            "Completas": 0,
            "Pendientes": 1,
        })

    if ruta_hashtag.exists():
        try:
            with pd.ExcelFile(
                ruta_hashtag,
                engine="openpyxl",
            ) as xls:
                hash_df = pd.read_excel(
                    xls,
                    sheet_name="Anotacion",
                )
        except PermissionError as exc:
            raise PermissionError(
                f"Cierre el archivo antes de auditarlo: {ruta_hashtag}"
            ) from exc

        total_hash = len(hash_df)
        completas_hash = (
            int(hash_df["etiqueta_humana"].notna().sum())
            if "etiqueta_humana" in hash_df.columns
            else 0
        )
        filas.append({
            "Archivo": ruta_hashtag.name,
            "Etapa": "etiqueta_humana",
            "Total": total_hash,
            "Completas": completas_hash,
            "Pendientes": int(
                total_hash - completas_hash
            ),
        })
    else:
        filas.append({
            "Archivo": ruta_hashtag.name,
            "Etapa": "ARCHIVO",
            "Total": 0,
            "Completas": 0,
            "Pendientes": 1,
        })

    return pd.DataFrame(filas)

In [102]:
def require_complete_annotations(
    ruta_manual: Path,
    ruta_hashtag: Path,
    holdout_manual: pd.DataFrame,
    muestra_hashtag: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Require complete annotations. Retains the original experimental logic and data-partition rules."""
    manual = load_human_annotations(ruta_manual)
    hashtag = load_human_hashtag_audit(ruta_hashtag)

    ids_manual_esperados = set(
        holdout_manual["id_registro"].astype(str)
    )
    ids_manual = set(
        manual["id_registro"].astype(str)
    )
    if ids_manual != ids_manual_esperados:
        raise RuntimeError(
            "The IDs in `anotacion_manual_politica.xlsx` do not match "
            "the reserved holdout for this protocol."
        )

    ids_hash_esperados = set(
        muestra_hashtag["id_registro"].astype(str)
    )
    ids_hash = set(
        hashtag["id_registro"].astype(str)
    )
    if ids_hash != ids_hash_esperados:
        raise RuntimeError(
            "The IDs in `anotacion_desacuerdos_hashtag.xlsx` do not match "
            "to the sample reserved before model development."
        )

    pendientes_a1 = int(
        manual["etiqueta_anotador_1"].isna().sum()
    )
    pendientes_a2 = int(
        manual["etiqueta_anotador_2"].isna().sum()
    )
    pendientes_hash = int(
        hashtag["etiqueta_humana"].isna().sum()
    )

    if pendientes_a1 > 0 or pendientes_a2 > 0:
        raise RuntimeError(
            "INDEPENDENT HUMAN ANNOTATIONS ARE INCOMPLETE. "
            f"Remaining labels for annotator 1={pendientes_a1}; "
            f"annotator 2={pendientes_a2}. "
            "Both annotators must finish before consensus can be calculated."
        )

    if pendientes_hash > 0:
        raise RuntimeError(
            "HASHTAG AUDIT ANNOTATIONS ARE INCOMPLETE. "
            f"Missing {pendientes_hash} human labels in "
            "anotacion_desacuerdos_hashtag.xlsx."
        )

    a1 = manual["etiqueta_anotador_1"].astype(int)
    a2 = manual["etiqueta_anotador_2"].astype(int)
    consenso = manual["etiqueta_consenso"]

    coinciden = a1 == a2
    desacuerdan = a1 != a2

    incoherentes = manual.loc[
        coinciden
        & (
            consenso.isna()
            | (
                consenso.astype("Int64")
                != a1.astype("Int64")
            )
        )
    ]
    if not incoherentes.empty:
        raise RuntimeError(
            "Some directly agreed labels have incomplete or inconsistent automatic consensus. "
            "Rerun the consensus stage. "
            "`update_consensus_worksheet()`."
        )

    pendientes_adjudicacion = manual.loc[
        desacuerdan
        & ~consenso.isin([0, 1])
    ]

    if not pendientes_adjudicacion.empty:
        raise RuntimeError(
            "ADJUDICATION IS INCOMPLETE. There are "
            f"{len(pendientes_adjudicacion)} disagreements without final decisions. "
            "Open `anotacion_manual_politica.xlsx`, worksheet "
            "`Desacuerdos_Adjudicar`, and fill `etiqueta_consenso` "
            "with 1=political or 0=non-political. The decision may be made by "
            "an independent adjudicator. Please preserve the original annotator labels "
            "after an explicit review. Save and close Excel, then "
            "rerun the consensus stage."
        )

    if manual["etiqueta_consenso"].isna().any():
        raise RuntimeError(
            "Some consensus labels are missing. Section 5 cannot start until all consensus labels are complete."
        )

    if manual["etiqueta_consenso"].astype(int).nunique() < 2:
        raise RuntimeError(
            "The human reference must contain both classes 0 and 1."
        )

    if hashtag["etiqueta_humana"].astype(int).nunique() < 2:
        raise RuntimeError(
            "The human hashtag audit must contain both classes 0 and 1."
        )

    return manual, hashtag

In [103]:
inventario_hashtags = build_hashtag_inventory(
    tweets_revision,
    HASHTAGS_POLITICOS_ESTRICTOS,
    HASHTAGS_NO_POLITICOS_ESTRICTOS,
)

In [104]:
silver_hashtags = build_hashtag_silver_standard(
    tweets_revision,
    HASHTAGS_POLITICOS_ESTRICTOS,
    HASHTAGS_NO_POLITICOS_ESTRICTOS,
)

In [105]:
ids_silver = set(silver_hashtags["id_registro"].astype(str))
restantes_sin_silver = tweets_revision[
    ~tweets_revision["id_registro"].astype(str).isin(ids_silver)
].reset_index(drop=True)

In [106]:
n_manual = min(
    N_ANOTACION_MANUAL,
    max(20, len(restantes_sin_silver) // 10),
)
tweets_desarrollo, holdout_manual = separate_human_annotation_holdout(
    restantes_sin_silver,
    n_manual,
    RANDOM_STATE,
)

In [107]:
muestra_auditoria_hashtag = sample_hashtag_audit(
    silver_hashtags,
    N_MANUAL_DESACUERDOS_HASHTAG,
    RANDOM_STATE,
)

In [108]:
ruta_plantilla_manual = save_human_annotation_template(
    holdout_manual,
    RUTA_ANOTACION_MANUAL,
    VERSION_PROTOCOLO_ANOTACION,
)

Existing annotation workbook matches the current protocol. Existing annotations will be preserved.


In [109]:
ruta_plantilla_hashtag = save_hashtag_audit_template(
    muestra_auditoria_hashtag,
    RUTA_ANOTACION_DESACUERDOS_HASHTAG,
    VERSION_PROTOCOLO_ANOTACION,
)

Existing hashtag audit workbook is compatible with the current protocol.


In [110]:
tweets_desarrollo_ciego_hashtags = prepare_hashtag_blind_corpus(
    tweets_desarrollo
)
silver_hashtags_ciego = prepare_hashtag_blind_corpus(
    silver_hashtags
)
trends_ciegos_hashtags = prepare_hashtag_blind_corpus(
    trends
)

In [111]:
resumen_silver = (
    silver_hashtags["etiqueta_silver_hashtag"]
    .value_counts()
    .sort_index()
    .rename_axis("Etiqueta_silver")
    .reset_index(name="N")
)
resumen_silver["Clase"] = resumen_silver["Etiqueta_silver"].map(
    {0: "No político", 1: "Político"}
)
resumen_silver["Porcentaje"] = (
    resumen_silver["N"] / resumen_silver["N"].sum() * 100.0
)

In [112]:
print(f"Final deduplicated corpus: {len(tweets_revision):,}")
print(f"Complete silver standard: {len(silver_hashtags):,}")
print(f"Independent human holdout: {len(holdout_manual):,}")
print(f"Human hashtag audit sample: {len(muestra_auditoria_hashtag):,}")
print(f"Development corpus: {len(tweets_desarrollo):,}")
print(f"File 1: {ruta_plantilla_manual}")
print(f"File 2: {ruta_plantilla_hashtag}")

Final deduplicated corpus: 138,400
Complete silver standard: 4,240
Independent human holdout: 600
Human hashtag audit sample: 250
Development corpus: 133,560
File 1: /home/lrecalde/data/cayala/datasets/anotaciones_trabajo/anotacion_manual_politica.xlsx
File 2: /home/lrecalde/data/cayala/datasets/anotaciones_trabajo/anotacion_desacuerdos_hashtag.xlsx


In [113]:
display_bokeh_table(
    resumen_silver,
    "Hashtag-based silver-standard distribution",
    alto=180,
)

In [114]:
display_bokeh_table(
    inventario_hashtags.head(80),
    "Hashtag inventory: 80 most frequent",
    ancho=1250,
    alto=420,
    max_filas=80,
)

## Human annotation and adjudicated consensus verification

In [115]:
ESTADO_ANTES_CONSENSO = audit_annotation_completeness(
    RUTA_ANOTACION_MANUAL,
    RUTA_ANOTACION_DESACUERDOS_HASHTAG,
)

In [116]:
display_bokeh_table(
    ESTADO_ANTES_CONSENSO,
    "Annotation status before consensus and adjudication",
    ancho=1250,
    alto=330,
    max_filas=20,
)

In [117]:
pendientes_iniciales = ESTADO_ANTES_CONSENSO.loc[
    ESTADO_ANTES_CONSENSO["Etapa"].isin(
        [
            "etiqueta_anotador_1",
            "etiqueta_anotador_2",
            "etiqueta_humana",
        ]
    ),
    "Pendientes",
].sum()

In [118]:
if int(pendientes_iniciales) > 0:
    raise RuntimeError(
        "The two primary annotators have not completed their labels "
        "or the hashtag audit labels are incomplete. Complete both annotation workbooks, save and close them, "
        "then rerun this cell."
    )

In [119]:
ANOTACIONES_CON_CONSENSO, DESACUERDOS_PENDIENTES = (
    update_consensus_worksheet(
        RUTA_ANOTACION_MANUAL
    )
)

Automatic consensus agreements: 513
Total annotator disagreements: 87
Unresolved adjudication disagreements: 0


In [120]:
ACUERDO_PREVIO_ANOTADORES = calculate_annotator_agreement(
    ANOTACIONES_CON_CONSENSO
)

In [121]:
display_bokeh_table(
    ACUERDO_PREVIO_ANOTADORES,
    "Agreement between primary annotators before adjudication",
    alto=180,
)

In [122]:
ESTADO_DESPUES_CONSENSO = audit_annotation_completeness(
    RUTA_ANOTACION_MANUAL,
    RUTA_ANOTACION_DESACUERDOS_HASHTAG,
)

In [123]:
display_bokeh_table(
    ESTADO_DESPUES_CONSENSO,
    "Annotation status after automatic consensus",
    ancho=1250,
    alto=330,
    max_filas=20,
)

In [124]:
if not DESACUERDOS_PENDIENTES.empty:
    columnas_mostrar = [
        columna
        for columna in [
            "id_registro",
            "texto_original_para_anotar",
            "etiqueta_anotador_1",
            "etiqueta_anotador_2",
            "etiqueta_consenso",
        ]
        if columna in DESACUERDOS_PENDIENTES.columns
    ]

    display_bokeh_table(
        DESACUERDOS_PENDIENTES[columnas_mostrar],
        "Disagreements awaiting human adjudication",
        ancho=1500,
        alto=480,
        max_filas=100,
    )

    raise RuntimeError(
        "ADJUDICATION INCOMPLETE. Automatic consensus finished, but the following disagreements remain "
        f"{len(DESACUERDOS_PENDIENTES)} desacuerdos. Abra "
        "`anotacion_manual_politica.xlsx`, hoja `Desacuerdos_Adjudicar`, "
        "fill `etiqueta_consenso` with 0 or 1, save the workbook, and "
        "rerun this cell. The notebook cannot continue until adjudication is complete."
    )

In [125]:
ANOTACIONES_MANUALES_PREVALIDADAS, ANOTACIONES_HASHTAG_PREVALIDADAS = (
    require_complete_annotations(
        RUTA_ANOTACION_MANUAL,
        RUTA_ANOTACION_DESACUERDOS_HASHTAG,
        holdout_manual,
        muestra_auditoria_hashtag,
    )
)

In [126]:
ESTADO_FINAL_ANOTACIONES = audit_annotation_completeness(
    RUTA_ANOTACION_MANUAL,
    RUTA_ANOTACION_DESACUERDOS_HASHTAG,
)

In [127]:
display_bokeh_table(
    ESTADO_FINAL_ANOTACIONES,
    "Final annotation verification before model training",
    ancho=1250,
    alto=330,
    max_filas=20,
)

# 5. Semantic representations, clustering and lexicon expansion

In [128]:
def train_word2vec(corpus: Sequence[Sequence[str]], dimension: int, window: int, min_count: int, epochs: int, seed: int) -> Word2Vec:
    """Train word2vec. Retains the original experimental logic and data-partition rules."""
    corpus_util = [list(doc) for doc in corpus if len(doc) > 0]
    if not corpus_util:
        raise ValueError("The corpus is empty, so Word2Vec cannot be trained.")
    return Word2Vec(
        sentences=corpus_util,
        vector_size=dimension,
        window=window,
        min_count=min_count,
        sg=1,
        workers=W2V_WORKERS,
        seed=seed,
        epochs=epochs,
    )

In [129]:
def get_unique_trend_terms(
    tokens_trends: Sequence[Sequence[str]],
    modelo: Word2Vec,
) -> list[str]:
    """Get unique trend terms. Retains the original experimental logic and data-partition rules."""
    palabras = sorted(
        {
            token
            for documento in tokens_trends
            for token in documento
            if token in modelo.wv
        }
    )

    if not K_CANDIDATOS:
        raise ValueError(
            "K_CANDIDATOS is empty. It must contain at least "
            "one k candidate to evaluate clustering."
        )

    k_maximo = max(K_CANDIDATOS)

    minimo_requerido = k_maximo + 1

    if len(palabras) < minimo_requerido:
        raise ValueError(
            "Too few trend tokens are present "
            "in the Word2Vec vocabulary to evaluate all "
            "candidate k values.\n"
            f"Palabras disponibles: {len(palabras)}\n"
            f"Mayor k candidato: {k_maximo}\n"
            f"Mínimo requerido: {minimo_requerido}"
        )

    return palabras

In [130]:
def get_word_matrix(palabras: Sequence[str], modelo: Word2Vec) -> np.ndarray:
    """Get word matrix. Retains the original experimental logic and data-partition rules."""
    return np.vstack([modelo.wv[p] for p in palabras]).astype(np.float32)

In [131]:
def fit_word_kmeans(vectores: np.ndarray, k: int, seed: int) -> tuple[KMeans, np.ndarray]:
    """Fit word kmeans. Retains the original experimental logic and data-partition rules."""
    modelo = KMeans(n_clusters=k, init="k-means++", n_init=20, max_iter=300, random_state=seed)
    etiquetas = modelo.fit_predict(vectores)
    return modelo, etiquetas.astype(int)

In [132]:
def summarize_clusters(palabras: Sequence[str], etiquetas: np.ndarray) -> pd.DataFrame:
    """Summarize clusters. Retains the original experimental logic and data-partition rules."""
    filas = []
    for cluster_id in sorted(np.unique(etiquetas)):
        terminos = sorted([p for p, c in zip(palabras, etiquetas) if int(c) == int(cluster_id)])
        filas.append({
            "Cluster": int(cluster_id),
            "Numero_palabras": len(terminos),
            "Muestra": ", ".join(terminos[:30]),
        })
    return pd.DataFrame(filas)

In [133]:
def select_political_cluster(palabras: Sequence[str], etiquetas: np.ndarray, vectores: np.ndarray, modelo: Word2Vec, semillas: Sequence[str]) -> tuple[int, pd.DataFrame]:
    """Select political cluster. Retains the original experimental logic and data-partition rules."""
    semillas_limpias = list(dict.fromkeys(str(s).strip().lower() for s in semillas if str(s).strip()))
    mapa_cluster = {p: int(c) for p, c in zip(palabras, etiquetas)}
    semillas_vector = [s for s in semillas_limpias if s in modelo.wv]
    filas = []

    for cluster_id in sorted(np.unique(etiquetas)):
        mask = etiquetas == cluster_id
        centroide = vectores[mask].mean(axis=0)
        norm_c = np.linalg.norm(centroide) or 1.0
        similitudes = []
        for semilla in semillas_vector:
            v = modelo.wv[semilla]
            denom = norm_c * (np.linalg.norm(v) or 1.0)
            similitudes.append(float(np.dot(centroide, v) / denom))
        coincidencias = sum(1 for s in semillas_limpias if mapa_cluster.get(s) == int(cluster_id))
        sim_media = float(np.mean(similitudes)) if similitudes else 0.0
        score = coincidencias * 2.0 + sim_media
        filas.append({
            "Cluster": int(cluster_id),
            "Semillas_en_cluster": int(coincidencias),
            "Similitud_media_semillas": sim_media,
            "Score_politico": score,
        })

    scores = pd.DataFrame(filas).sort_values(["Score_politico", "Semillas_en_cluster"], ascending=False).reset_index(drop=True)
    return int(scores.iloc[0]["Cluster"]), scores

In [134]:
def calculate_centroids(vectores: np.ndarray, etiquetas: np.ndarray, k: int) -> np.ndarray:
    """Calculate centroids. Retains the original experimental logic and data-partition rules."""
    centroides = []
    global_mean = vectores.mean(axis=0)
    for cluster_id in range(k):
        mask = etiquetas == cluster_id
        centroides.append(vectores[mask].mean(axis=0) if np.any(mask) else global_mean)
    return np.vstack(centroides).astype(np.float32)

In [135]:
def vectorize_documents(tokens_documentos: Sequence[Sequence[str]], modelo: Word2Vec, agregacion: str = "suma") -> np.ndarray:
    """Vectorize documents. Retains the original experimental logic and data-partition rules."""
    dimension = modelo.vector_size
    salida = np.zeros((len(tokens_documentos), dimension), dtype=np.float32)
    for i, doc in enumerate(tokens_documentos):
        validos = [modelo.wv[t] for t in doc if t in modelo.wv]
        if not validos:
            continue
        matriz = np.vstack(validos)
        salida[i] = matriz.mean(axis=0) if agregacion == "media" else matriz.sum(axis=0)
    return salida

In [136]:
def label_with_centroids(vectores_tweets: np.ndarray, centroides: np.ndarray, cluster_politico: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Label with centroids. Retains the original experimental logic and data-partition rules."""
    eps = 1e-12
    norma_x = np.linalg.norm(vectores_tweets, axis=1, keepdims=True)
    norma_c = np.linalg.norm(centroides, axis=1, keepdims=True).T
    similitudes = (vectores_tweets @ centroides.T) / np.maximum(norma_x * norma_c, eps)
    distancias = 1.0 - similitudes

    vacios = norma_x.ravel() <= eps
    cluster_cercano = np.argmin(distancias, axis=1).astype(int)
    cluster_cercano[vacios] = -1
    etiquetas = (cluster_cercano == int(cluster_politico)).astype(int)
    distancia_politica = distancias[:, int(cluster_politico)].astype(np.float32)
    distancia_politica[vacios] = np.nan
    return etiquetas, cluster_cercano, distancia_politica

In [137]:
def evaluate_kmeans_objectively(vectores: np.ndarray, valores_k: Sequence[int], seed: int) -> pd.DataFrame:
    """Evaluate kmeans objectively. Retains the original experimental logic and data-partition rules."""
    filas: list[dict[str, float | int]] = []
    for k in valores_k:
        if k < 2 or k >= len(vectores):
            continue
        inicio = time.perf_counter()
        modelo = KMeans(
            n_clusters=int(k),
            init="k-means++",
            n_init=20,
            max_iter=300,
            random_state=seed,
        )
        labels = modelo.fit_predict(vectores)
        tiempo = time.perf_counter() - inicio
        filas.append({
            "k": int(k),
            "Inercia": float(modelo.inertia_),
            "Silhouette": float(silhouette_score(vectores, labels)),
            "Davies_Bouldin": float(davies_bouldin_score(vectores, labels)),
            "Calinski_Harabasz": float(calinski_harabasz_score(vectores, labels)),
            "Tiempo_s": float(tiempo),
        })
    return pd.DataFrame(filas)

In [138]:
def select_optimal_k(tabla_k: pd.DataFrame) -> int:
    """Select optimal k. Retains the original experimental logic and data-partition rules."""
    if tabla_k.empty:
        raise ValueError("The k-selection validation table is empty.")
    ordenada = tabla_k.sort_values(["Silhouette", "Davies_Bouldin"], ascending=[False, True])
    return int(ordenada.iloc[0]["k"])

In [139]:
def plot_k_validation(tabla_k: pd.DataFrame, dimension: int) -> None:
    """Plot k validation. Retains the original experimental logic and data-partition rules."""

    columnas_requeridas = {
        "k",
        "Silhouette",
        "Davies_Bouldin",
        "Calinski_Harabasz",
        "Inercia",
    }

    columnas_faltantes = columnas_requeridas.difference(tabla_k.columns)

    if columnas_faltantes:
        raise ValueError(
            "Missing required columns in the k-selection table: "
            f"{sorted(columnas_faltantes)}. "
            f"Columnas disponibles: {tabla_k.columns.tolist()}"
        )

    datos = tabla_k.sort_values("k").copy()

    source = ColumnDataSource(datos)

    # ---------------------------------------------------------
    # 1. Silhouette
    # ---------------------------------------------------------
    p1 = figure(
        height=340,
        width=780,
        title=f"Silhouette por k — {dimension}D",
        x_axis_label="k",
        y_axis_label="Silhouette",
    )

    p1.line(
        "k",
        "Silhouette",
        source=source,
        line_width=2,
    )

    p1.scatter(
        "k",
        "Silhouette",
        source=source,
        size=8,
    )

    p1.add_tools(
        HoverTool(
            tooltips=[
                ("k", "@k"),
                ("Silhouette", "@Silhouette{0.0000}"),
            ]
        )
    )

    # ---------------------------------------------------------
    # 2. Davies-Bouldin
    # ---------------------------------------------------------
    p2 = figure(
        height=340,
        width=780,
        title=f"Davies-Bouldin por k — {dimension}D",
        x_axis_label="k",
        y_axis_label="Davies-Bouldin",
    )

    p2.line(
        "k",
        "Davies_Bouldin",
        source=source,
        line_width=2,
    )

    p2.scatter(
        "k",
        "Davies_Bouldin",
        source=source,
        size=8,
    )

    p2.add_tools(
        HoverTool(
            tooltips=[
                ("k", "@k"),
                ("Davies-Bouldin", "@Davies_Bouldin{0.0000}"),
            ]
        )
    )

    # ---------------------------------------------------------
    # 3. Calinski-Harabasz
    # ---------------------------------------------------------
    p3 = figure(
        height=340,
        width=780,
        title=f"Calinski-Harabasz por k — {dimension}D",
        x_axis_label="k",
        y_axis_label="Calinski-Harabasz",
    )

    p3.line(
        "k",
        "Calinski_Harabasz",
        source=source,
        line_width=2,
    )

    p3.scatter(
        "k",
        "Calinski_Harabasz",
        source=source,
        size=8,
    )

    p3.add_tools(
        HoverTool(
            tooltips=[
                ("k", "@k"),
                ("Calinski-Harabasz", "@Calinski_Harabasz{0.00}"),
            ]
        )
    )

    # ---------------------------------------------------------
    # 4. Inertia
    # ---------------------------------------------------------
    p4 = figure(
        height=340,
        width=780,
        title=f"Inercia por k — {dimension}D",
        x_axis_label="k",
        y_axis_label="Inercia",
    )

    p4.line(
        "k",
        "Inercia",
        source=source,
        line_width=2,
    )

    p4.scatter(
        "k",
        "Inercia",
        source=source,
        size=8,
    )

    p4.add_tools(
        HoverTool(
            tooltips=[
                ("k", "@k"),
                ("Inercia", "@Inercia{0.00}"),
            ]
        )
    )

    # ---------------------------------------------------------
    # Tab panels
    # ---------------------------------------------------------
    tabs = Tabs(
        tabs=[
            TabPanel(
                child=p1,
                title="Silhouette",
            ),
            TabPanel(
                child=p2,
                title="Davies-Bouldin",
            ),
            TabPanel(
                child=p3,
                title="Calinski-Harabasz",
            ),
            TabPanel(
                child=p4,
                title="Inercia",
            ),
        ]
    )

    show(tabs)

In [140]:
def evaluate_hdbscan(vectores: np.ndarray, min_cluster_size: int, min_samples: int) -> tuple[np.ndarray, pd.DataFrame]:
    """Evaluate hdbscan. Retains the original experimental logic and data-partition rules."""
    if HDBSCAN is None:
        return np.full(len(vectores), -1, dtype=int), pd.DataFrame([{
            "Algorithm": "HDBSCAN",
            "Clusters": 0,
            "Ruido_pct": 100.0,
            "Silhouette": np.nan,
            "Davies_Bouldin": np.nan,
            "Calinski_Harabasz": np.nan,
            "Tiempo_s": np.nan,
            "Status": "HDBSCAN is unavailable in this scikit-learn version",
        }])

    inicio = time.perf_counter()
    modelo = HDBSCAN(
        min_cluster_size=int(min_cluster_size),
        min_samples=int(min_samples),
        metric="euclidean",
        cluster_selection_method="eom",
    )
    labels = modelo.fit_predict(vectores).astype(int)
    tiempo = time.perf_counter() - inicio

    mask = labels != -1
    labels_validos = labels[mask]
    vectores_validos = vectores[mask]
    clusters = sorted(set(labels_validos.tolist()))
    ruido_pct = float(np.mean(labels == -1) * 100.0)

    sil = db = ch = np.nan
    if len(clusters) >= 2 and len(vectores_validos) > len(clusters):
        sil = float(silhouette_score(vectores_validos, labels_validos))
        db = float(davies_bouldin_score(vectores_validos, labels_validos))
        ch = float(calinski_harabasz_score(vectores_validos, labels_validos))

    tabla = pd.DataFrame([{
        "Algorithm": "HDBSCAN",
        "Clusters": int(len(clusters)),
        "Ruido_pct": ruido_pct,
        "Silhouette": sil,
        "Davies_Bouldin": db,
        "Calinski_Harabasz": ch,
        "Tiempo_s": float(tiempo),
        "Status": "OK",
    }])
    return labels, tabla

In [141]:
def plot_tsne(palabras: Sequence[str], vectores: np.ndarray, etiquetas: np.ndarray, titulo: str, seed: int, max_puntos: int = 1000) -> None:
    """Plot tsne. Retains the original experimental logic and data-partition rules."""
    n = min(len(vectores), int(max_puntos))
    if n < 5:
        print("Not enough points for t-SNE.")
        return

    rng = np.random.default_rng(seed)
    idx = np.arange(len(vectores))
    if len(idx) > n:
        idx = rng.choice(idx, size=n, replace=False)

    x = vectores[idx]
    labels = np.asarray(etiquetas)[idx]
    words = np.asarray(list(palabras), dtype=object)[idx]
    perplexity = max(5.0, min(30.0, (n - 1) / 3.0))

    coords = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        random_state=seed,
    ).fit_transform(x)

    vista = pd.DataFrame({
        "x": coords[:, 0],
        "y": coords[:, 1],
        "Palabra": words.astype(str),
        "Cluster": labels.astype(str),
    })
    source = ColumnDataSource(vista)
    p = figure(height=560, width=900, title=titulo, tools="pan,wheel_zoom,box_zoom,reset,save")
    p.scatter("x", "y", source=source, size=7, alpha=0.65, legend_field="Cluster")
    p.add_tools(HoverTool(tooltips=[("Palabra", "@Palabra"), ("Cluster", "@Cluster")]))
    p.legend.location = "top_left"
    show(p)

In [142]:
def expand_political_cluster_topn(palabras: Sequence[str], vectores: np.ndarray, etiquetas: np.ndarray, cluster_politico: int, top_n: int) -> tuple[np.ndarray, pd.DataFrame, float]:
    """Execute expand political cluster topn. Retains the original experimental logic and data-partition rules."""
    if top_n <= 0:
        raise ValueError("top_n must be positive.")

    etiquetas = np.asarray(etiquetas).astype(int)
    mask_pol = etiquetas == int(cluster_politico)
    if not np.any(mask_pol):
        raise ValueError("The political cluster contains no vocabulary terms.")

    centroide = vectores[mask_pol].mean(axis=0)
    norma_c = max(float(np.linalg.norm(centroide)), 1e-12)
    indices_externos = np.where(~mask_pol)[0]
    if len(indices_externos) == 0:
        return etiquetas.copy(), pd.DataFrame(), np.nan

    ext = vectores[indices_externos]
    denom = np.maximum(np.linalg.norm(ext, axis=1) * norma_c, 1e-12)
    similitudes = (ext @ centroide) / denom
    orden = np.argsort(-similitudes)
    n_real = min(int(top_n), len(orden))
    seleccion_local = orden[:n_real]
    seleccion_global = indices_externos[seleccion_local]

    nuevas = etiquetas.copy()
    nuevas[seleccion_global] = int(cluster_politico)
    detalle = pd.DataFrame({
        "Rango": np.arange(1, n_real + 1, dtype=int),
        "Palabra": [palabras[i] for i in seleccion_global],
        "Cluster_original": etiquetas[seleccion_global],
        "Cluster_nuevo": int(cluster_politico),
        "Similitud_coseno_politica": similitudes[seleccion_local],
        "Distancia_coseno_politica": 1.0 - similitudes[seleccion_local],
    }).sort_values("Rango").reset_index(drop=True)

    umbral = float(detalle["Similitud_coseno_politica"].iloc[-1]) if not detalle.empty else np.nan
    return nuevas, detalle, umbral

In [143]:
def get_similar_words(modelo: Word2Vec, palabra: str, top_n: int = 15) -> pd.DataFrame:
    """Get similar words. Retains the original experimental logic and data-partition rules."""
    palabra = palabra.strip().lower()
    if palabra not in modelo.wv:
        return pd.DataFrame(columns=["Palabra", "Similitud"])
    similares = modelo.wv.most_similar(palabra, topn=top_n)
    return pd.DataFrame(similares, columns=["Palabra", "Similitud"])

In [144]:
def calculate_set_similarity(modelo: Word2Vec, palabra: str, conjunto: Sequence[str]) -> pd.DataFrame:
    """Calculate set similarity. Retains the original experimental logic and data-partition rules."""
    palabra = palabra.strip().lower()
    if palabra not in modelo.wv:
        return pd.DataFrame(columns=["Termino", "Similitud"])
    filas = []
    for termino in conjunto:
        termino = str(termino).strip().lower()
        if termino in modelo.wv:
            filas.append({"Termino": termino, "Similitud": float(modelo.wv.similarity(palabra, termino))})
    return pd.DataFrame(filas).sort_values("Similitud", ascending=False).reset_index(drop=True) if filas else pd.DataFrame(columns=["Termino", "Similitud"])

In [145]:
def solve_embedding_analogy(modelo: Word2Vec, positivo: Sequence[str], negativo: Sequence[str], top_n: int = 10) -> pd.DataFrame:
    """Solve embedding analogy. Retains the original experimental logic and data-partition rules."""
    pos = [str(x).lower().strip() for x in positivo if str(x).lower().strip() in modelo.wv]
    neg = [str(x).lower().strip() for x in negativo if str(x).lower().strip() in modelo.wv]
    if not pos:
        return pd.DataFrame(columns=["Palabra", "Similitud"])
    try:
        resultado = modelo.wv.most_similar(positive=pos, negative=neg, topn=top_n)
    except Exception:
        return pd.DataFrame(columns=["Palabra", "Similitud"])
    return pd.DataFrame(resultado, columns=["Palabra", "Similitud"])

In [146]:
def plot_word_similarities(similares: pd.DataFrame, palabra: str, dimension: int) -> None:
    """Plot word similarities. Retains the original experimental logic and data-partition rules."""
    if similares.empty:
        show(Div(text=f"<b>No hubo resultados de similitud para '{palabra}'.</b>"))
        return
    df = similares.iloc[::-1].copy()
    source = ColumnDataSource(df)
    p = figure(y_range=df["Palabra"].astype(str).tolist(), width=850, height=max(350, 25 * len(df)), title=f"Palabras similares a '{palabra}' — {dimension}D")
    p.hbar(y="Palabra", right="Similitud", height=0.7, source=source)
    p.xaxis.axis_label = "Cosine similarity"
    p.add_tools(HoverTool(tooltips=[("Palabra", "@Palabra"), ("Similitud", "@Similitud{0.0000}")]))
    show(p)

In [147]:
def plot_pca_clusters(palabras: Sequence[str], vectores: np.ndarray, etiquetas: np.ndarray, cluster_politico: int, dimension: int, max_puntos: int = 900) -> None:
    """Plot pca clusters. Retains the original experimental logic and data-partition rules."""
    n = len(palabras)
    rng = np.random.default_rng(RANDOM_STATE)
    idx = np.arange(n)
    if n > max_puntos:
        idx = np.sort(rng.choice(idx, size=max_puntos, replace=False))

    coords = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(vectores[idx])
    labels_sub = etiquetas[idx]
    palabras_sub = [palabras[i] for i in idx]
    colores_base = Category10[10]
    colores = [colores_base[int(c) % len(colores_base)] for c in labels_sub]
    source = ColumnDataSource({
        "x": coords[:, 0],
        "y": coords[:, 1],
        "palabra": palabras_sub,
        "cluster": labels_sub.astype(str),
        "tipo": ["Político" if int(c) == cluster_politico else "Otro" for c in labels_sub],
        "color": colores,
    })
    p = figure(width=950, height=550, title=f"PCA projection of semantic clusters — Word2Vec {dimension}D", tools="pan,wheel_zoom,box_zoom,reset,save")
    p.scatter("x", "y", source=source, size=7, alpha=0.7, color="color")
    p.add_tools(HoverTool(tooltips=[("Palabra", "@palabra"), ("Clúster", "@cluster"), ("Tipo", "@tipo")]))
    p.xaxis.axis_label = "PCA 1"
    p.yaxis.axis_label = "PCA 2"
    show(p)

# 6. Supervised classification and leakage-free holdout evaluation

In [148]:
def create_classifiers(knn_vecinos: int, seed: int) -> dict[str, Any]:
    """Create classifiers. Retains the original experimental logic and data-partition rules."""
    return {
        "Decision Tree": DecisionTreeClassifier(random_state=seed),
        "KNN": KNeighborsClassifier(n_neighbors=knn_vecinos, n_jobs=-1),
        "Naïve Bayes": GaussianNB(),
        "SVM": LinearSVC(random_state=seed, max_iter=5000, dual="auto"),
    }

In [149]:
def calculate_binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """Calculate binary metrics. Retains the original experimental logic and data-partition rules."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    especificidad = tn / (tn + fp) if (tn + fp) else 0.0
    fpr = fp / (tn + fp) if (tn + fp) else 0.0
    return {
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Precision": float(precision),
        "Recall": float(recall),
        "F1-score": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "Sensitivity": float(recall),
        "Specificity": float(especificidad),
        "Positive precision": float(precision),
        "False-positive rate": float(fpr),
        "F2 score": float(fbeta_score(y_true, y_pred, beta=2.0, pos_label=1, zero_division=0)),
        "F0.5 score": float(fbeta_score(y_true, y_pred, beta=0.5, pos_label=1, zero_division=0)),
        "TP": float(tp),
        "TN": float(tn),
        "FP": float(fp),
        "FN": float(fn),
    }

In [150]:
def train_and_evaluate_classifiers(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray, y_test: np.ndarray, dimension: int, escenario: str, seed: int, knn_vecinos: int) -> pd.DataFrame:
    """Train and evaluate classifiers. Retains the original experimental logic and data-partition rules."""
    clases, conteos = np.unique(y_train, return_counts=True)
    if len(clases) < 2 or np.min(conteos) < 2:
        raise ValueError(f"El entrenamiento requiere dos clases. Distribución: {dict(zip(clases, conteos))}")

    filas: list[dict[str, Any]] = []
    for nombre, modelo in create_classifiers(knn_vecinos, seed).items():
        inicio = time.perf_counter()
        modelo.fit(X_train, y_train)
        pred = modelo.predict(X_test)
        tiempo = time.perf_counter() - inicio
        fila: dict[str, Any] = {
            "Dimension": int(dimension),
            "Scenario": escenario,
            "Algorithm": nombre,
            "Tiempo_clasificador_s": float(tiempo),
        }
        fila.update(calculate_binary_metrics(y_test, pred))
        filas.append(fila)
    return pd.DataFrame(filas)

In [151]:
def build_train_semantic_pipeline(tokens_train: Sequence[Sequence[str]], tokens_trends: Sequence[Sequence[str]], dimension: int, semillas: Sequence[str], valores_k: Sequence[int], top_n: int, epochs: int, seed: int) -> dict[str, Any]:
    """Build train semantic pipeline. Retains the original experimental logic and data-partition rules."""
    inicio_w2v = time.perf_counter()
    modelo = train_word2vec(tokens_train, dimension, W2V_WINDOW, W2V_MIN_COUNT, epochs, seed)
    tiempo_w2v = time.perf_counter() - inicio_w2v

    palabras = get_unique_trend_terms(tokens_trends, modelo)
    if len(palabras) < 3:
        raise ValueError("Too few eligible trend words appear in the training vocabulary.")

    vectores_raw = get_word_matrix(palabras, modelo)
    vectores_cluster = normalize(vectores_raw, norm="l2")

    tabla_k = evaluate_kmeans_objectively(vectores_cluster, valores_k, seed)
    k_seleccionado = select_optimal_k(tabla_k)
    kmeans, labels_1 = fit_word_kmeans(vectores_cluster, k_seleccionado, seed)

    cluster_politico, scores = select_political_cluster(
        palabras,
        labels_1,
        vectores_raw,
        modelo,
        semillas,
    )
    centroides_1 = calculate_centroids(vectores_raw, labels_1, k_seleccionado)

    labels_2, palabras_anadidas, umbral_similitud = expand_political_cluster_topn(
        palabras,
        vectores_raw,
        labels_1,
        cluster_politico,
        top_n,
    )
    centroides_2 = calculate_centroids(vectores_raw, labels_2, k_seleccionado)

    return {
        "modelo_word2vec": modelo,
        "tiempo_word2vec_s": float(tiempo_w2v),
        "palabras": palabras,
        "vectores_palabras_raw": vectores_raw,
        "vectores_palabras_cluster": vectores_cluster,
        "tabla_k": tabla_k,
        "k_seleccionado": int(k_seleccionado),
        "kmeans": kmeans,
        "labels_palabras_1": labels_1,
        "labels_palabras_2": labels_2,
        "cluster_politico": int(cluster_politico),
        "scores_cluster": scores,
        "centroides_1": centroides_1,
        "centroides_2": centroides_2,
        "palabras_anadidas": palabras_anadidas,
        "umbral_similitud": float(umbral_similitud),
        "top_n_solicitado": int(top_n),
    }

In [152]:
def run_leakage_free_holdout(tweets_desarrollo: pd.DataFrame, tokens_trends: Sequence[Sequence[str]], dimension: int, semillas: Sequence[str], top_n: int, test_size: float, epochs: int, seed: int) -> dict[str, Any]:
    """Run leakage free holdout. Retains the original experimental logic and data-partition rules."""
    estrato = tweets_desarrollo["evento"].fillna("Sin evento").astype(str)
    train_df, test_df = train_test_split(
        tweets_desarrollo,
        test_size=test_size,
        random_state=seed,
        stratify=estrato,
    )
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    tokens_train = extract_dataframe_tokens(train_df, "tokens_sin_stopwords")
    tokens_test = extract_dataframe_tokens(test_df, "tokens_sin_stopwords")

    pipeline = build_train_semantic_pipeline(
        tokens_train,
        tokens_trends,
        dimension,
        semillas,
        K_CANDIDATOS,
        top_n,
        epochs,
        seed,
    )

    modelo = pipeline["modelo_word2vec"]
    X_train = vectorize_documents(tokens_train, modelo, agregacion="suma")
    X_test = vectorize_documents(tokens_test, modelo, agregacion="suma")

    y1_train, _, _ = label_with_centroids(X_train, pipeline["centroides_1"], pipeline["cluster_politico"])
    y1_test, _, _ = label_with_centroids(X_test, pipeline["centroides_1"], pipeline["cluster_politico"])
    y2_train, _, _ = label_with_centroids(X_train, pipeline["centroides_2"], pipeline["cluster_politico"])
    y2_test, _, _ = label_with_centroids(X_test, pipeline["centroides_2"], pipeline["cluster_politico"])

    met1 = train_and_evaluate_classifiers(X_train, y1_train, X_test, y1_test, dimension, "Scenario 1", seed, KNN_VECINOS)
    met2 = train_and_evaluate_classifiers(X_train, y2_train, X_test, y2_test, dimension, "Scenario 2", seed, KNN_VECINOS)
    metricas = pd.concat([met1, met2], ignore_index=True)

    balance = pd.DataFrame([
        {"Conjunto": "Train", "Scenario": "Scenario 1", "N": len(y1_train), "Politicos": int(y1_train.sum()), "No_politicos": int((y1_train == 0).sum()), "Politicos_pct": float(y1_train.mean() * 100)},
        {"Conjunto": "Test", "Scenario": "Scenario 1", "N": len(y1_test), "Politicos": int(y1_test.sum()), "No_politicos": int((y1_test == 0).sum()), "Politicos_pct": float(y1_test.mean() * 100)},
        {"Conjunto": "Train", "Scenario": "Scenario 2", "N": len(y2_train), "Politicos": int(y2_train.sum()), "No_politicos": int((y2_train == 0).sum()), "Politicos_pct": float(y2_train.mean() * 100)},
        {"Conjunto": "Test", "Scenario": "Scenario 2", "N": len(y2_test), "Politicos": int(y2_test.sum()), "No_politicos": int((y2_test == 0).sum()), "Politicos_pct": float(y2_test.mean() * 100)},
    ])

    return {
        "dimension": int(dimension),
        "train_df": train_df,
        "test_df": test_df,
        "tokens_train": tokens_train,
        "tokens_test": tokens_test,
        "pipeline": pipeline,
        "X_train": X_train,
        "X_test": X_test,
        "y1_train": y1_train,
        "y1_test": y1_test,
        "y2_train": y2_train,
        "y2_test": y2_test,
        "metricas": metricas,
        "balance": balance,
    }

In [153]:
def summarize_dimension_holdouts(resultados: Mapping[int, Mapping[str, Any]]) -> pd.DataFrame:
    """Summarize dimension holdouts. Retains the original experimental logic and data-partition rules."""
    filas: list[dict[str, Any]] = []
    for dimension, resultado in resultados.items():
        pipe = resultado["pipeline"]
        y1_test = np.asarray(resultado["y1_test"]).astype(int)
        y2_test = np.asarray(resultado["y2_test"]).astype(int)
        filas.append({
            "Dimension": int(dimension),
            "k_seleccionado": int(pipe["k_seleccionado"]),
            "Cluster_politico": int(pipe["cluster_politico"]),
            "Vocabulario_trends_en_train": int(len(pipe["palabras"])),
            "Top_N_solicitado": int(pipe.get("top_n_solicitado", len(pipe["palabras_anadidas"]))),
            "N_terminos_anadidos": int(len(pipe["palabras_anadidas"])),
            "Umbral_similitud": float(pipe["umbral_similitud"]),
            "Politicos_test_E1": int(y1_test.sum()),
            "Politicos_test_E1_pct": float(y1_test.mean() * 100.0),
            "Politicos_test_E2": int(y2_test.sum()),
            "Politicos_test_E2_pct": float(y2_test.mean() * 100.0),
            "Ganancia_politicos_test": int(y2_test.sum() - y1_test.sum()),
        })
    return pd.DataFrame(filas).sort_values("Dimension").reset_index(drop=True)

In [154]:
def compare_kmeans_hdbscan(resultado_holdout: Mapping[str, Any], min_cluster_size: int, min_samples: int) -> tuple[pd.DataFrame, np.ndarray]:
    """Compare kmeans hdbscan. Retains the original experimental logic and data-partition rules."""
    pipe = resultado_holdout["pipeline"]
    vectores = pipe["vectores_palabras_cluster"]
    labels_k = pipe["labels_palabras_1"]
    k = pipe["k_seleccionado"]
    fila_k = pd.DataFrame([{
        "Algorithm": f"K-Means (k={k})",
        "Clusters": int(k),
        "Ruido_pct": 0.0,
        "Silhouette": float(silhouette_score(vectores, labels_k)),
        "Davies_Bouldin": float(davies_bouldin_score(vectores, labels_k)),
        "Calinski_Harabasz": float(calinski_harabasz_score(vectores, labels_k)),
        "Tiempo_s": float(pipe["tabla_k"].loc[pipe["tabla_k"]["k"] == k, "Tiempo_s"].iloc[0]),
        "Status": "OK",
    }])
    labels_hdb, tabla_hdb = evaluate_hdbscan(vectores, min_cluster_size, min_samples)
    return pd.concat([fila_k, tabla_hdb], ignore_index=True), labels_hdb

In [155]:
def evaluate_topn_sensitivity(resultado_holdout: Mapping[str, Any], valores_top_n: Sequence[int], seed: int) -> pd.DataFrame:
    """Evaluate topn sensitivity. Retains the original experimental logic and data-partition rules."""
    pipe = resultado_holdout["pipeline"]
    palabras = pipe["palabras"]
    vectores = pipe["vectores_palabras_raw"]
    labels_1 = pipe["labels_palabras_1"]
    cluster_politico = pipe["cluster_politico"]
    X_train = resultado_holdout["X_train"]
    X_test = resultado_holdout["X_test"]
    dimension = resultado_holdout["dimension"]
    filas: list[pd.DataFrame] = []

    for top_n in valores_top_n:
        labels_2, detalle, umbral = expand_political_cluster_topn(
            palabras,
            vectores,
            labels_1,
            cluster_politico,
            int(top_n),
        )
        centroides_2 = calculate_centroids(vectores, labels_2, pipe["k_seleccionado"])
        y_train, _, _ = label_with_centroids(X_train, centroides_2, cluster_politico)
        y_test, _, _ = label_with_centroids(X_test, centroides_2, cluster_politico)
        tabla = train_and_evaluate_classifiers(
            X_train,
            y_train,
            X_test,
            y_test,
            dimension,
            f"Top-{top_n}",
            seed,
            KNN_VECINOS,
        )
        tabla["Top_N_solicitado"] = int(top_n)
        tabla["N_terminos_anadidos"] = int(len(detalle))
        tabla["Umbral_similitud"] = float(umbral)
        tabla["Politicos_test"] = int(y_test.sum())
        tabla["Politicos_test_pct"] = float(y_test.mean() * 100.0)
        filas.append(tabla)

    return pd.concat(filas, ignore_index=True) if filas else pd.DataFrame()

In [156]:
def plot_topn_sensitivity(
    sensibilidad: pd.DataFrame,
    metrica: str = "F1-score"
) -> None:
    """Plot topn sensitivity. Retains the original experimental logic and data-partition rules."""
    if sensibilidad.empty:
        print("No sensitivity results available to plot.")
        return

    if metrica not in sensibilidad.columns:
        raise ValueError(
            f"La métrica '{metrica}' no existe en el DataFrame. "
            f"Columnas disponibles: {sensibilidad.columns.tolist()}"
        )

    algoritmos = list(sensibilidad["Algorithm"].unique())

    if len(algoritmos) <= 10:
        paleta = Category10[10]
    else:
        paleta = Category20[20]

    colores_algoritmos = {
        algoritmo: paleta[i % len(paleta)]
        for i, algoritmo in enumerate(algoritmos)
    }

    p = figure(
        height=430,
        width=900,
        title=f"Sensitivity of {metrica} frente a Top-N",
        x_axis_label="Top-N",
        y_axis_label=metrica
    )

    for algoritmo in algoritmos:

        sub = (
            sensibilidad[
                sensibilidad["Algorithm"] == algoritmo
            ]
            .sort_values("Top_N_solicitado")
            .copy()
        )

        source = ColumnDataSource(sub)

        color = colores_algoritmos[algoritmo]

        p.line(
            "Top_N_solicitado",
            metrica,
            source=source,
            line_width=2.5,
            color=color,
            legend_label=str(algoritmo)
        )

        p.scatter(
            "Top_N_solicitado",
            metrica,
            source=source,
            size=8,
            color=color,
            legend_label=str(algoritmo)
        )

    p.add_tools(
        HoverTool(
            tooltips=[
                ("Algorithm", "@Algoritmo"),
                ("Top-N", "@Top_N_solicitado"),
                (metrica, f"@{{{metrica}}}{{0.0000}}"),
                ("Added terms", "@N_terminos_anadidos"),
                ("Umbral", "@Umbral_similitud{0.0000}")
            ]
        )
    )

    p.legend.location = "bottom_right"
    p.legend.click_policy = "hide"
    p.legend.title = "Clasificador"

    show(p)

In [157]:
def plot_topn_coverage_and_cutoff(
    sensibilidad: pd.DataFrame,
    dimension: int = 100
) -> None:
    """Plot topn coverage and cutoff. Retains the original experimental logic and data-partition rules."""

    if sensibilidad.empty:
        print("No sensitivity results available to plot.")
        return

    columnas_requeridas = {
        "Top_N_solicitado",
        "Politicos_test_pct",
        "Umbral_similitud",
    }

    faltantes = columnas_requeridas.difference(
        sensibilidad.columns
    )

    if faltantes:
        raise ValueError(
            f"Faltan columnas: {sorted(faltantes)}. "
            f"Columnas disponibles: "
            f"{sensibilidad.columns.tolist()}"
        )

    datos = (
        sensibilidad[
            [
                "Top_N_solicitado",
                "Politicos_test_pct",
                "Umbral_similitud",
            ]
        ]
        .drop_duplicates(subset="Top_N_solicitado")
        .sort_values("Top_N_solicitado")
        .copy()
    )

    source = ColumnDataSource(datos)

    p1 = figure(
        height=380,
        width=500,
        title=f"Political coverage vs Top-N — {dimension}D",
        x_axis_label="Top-N",
        y_axis_label="Political coverage (%)",
    )

    p1.line(
        "Top_N_solicitado",
        "Politicos_test_pct",
        source=source,
        line_width=2.5,
    )

    p1.scatter(
        "Top_N_solicitado",
        "Politicos_test_pct",
        source=source,
        size=9,
    )

    p1.add_tools(
        HoverTool(
            tooltips=[
                ("Top-N", "@Top_N_solicitado"),
                (
                    "Political coverage",
                    "@Politicos_test_pct{0.00}%"
                ),
            ]
        )
    )

    p2 = figure(
        height=380,
        width=500,
        title=f"Observed similarity cutoff vs Top-N — {dimension}D",
        x_axis_label="Top-N",
        y_axis_label="Cosine similarity cutoff",
    )

    p2.line(
        "Top_N_solicitado",
        "Umbral_similitud",
        source=source,
        line_width=2.5,
    )

    p2.scatter(
        "Top_N_solicitado",
        "Umbral_similitud",
        source=source,
        size=9,
    )

    p2.add_tools(
        HoverTool(
            tooltips=[
                ("Top-N", "@Top_N_solicitado"),
                (
                    "Similarity cutoff",
                    "@Umbral_similitud{0.0000}"
                ),
            ]
        )
    )

    show(row(p1, p2))

In [158]:
tokens_trends = extract_dataframe_tokens(trends, "tokens_sin_stopwords")

In [159]:
semillas_tesis: list[str] = []
if "Terminos_Tesis" in resumen_preprocesamiento:
    semillas_tesis = (
        resumen_preprocesamiento["Terminos_Tesis"]
        .iloc[:, 0]
        .dropna()
        .astype(str)
        .str.lower()
        .str.strip()
        .tolist()
    )

In [160]:
semillas_politicas = list(dict.fromkeys(semillas_tesis + SEMILLAS_POLITICAS_EXTRA))
semillas_politicas_sin_hashtags = [
    s for s in semillas_politicas
    if s and not str(s).strip().startswith("#")
]

In [161]:
print(f"Total political seed terms: {len(semillas_politicas)}")

Total political seed terms: 20


In [162]:
print(semillas_politicas)

['ecu', 'lasso', 'guillermo', '@asambleaecuador', 'guerra', 'protesta', 'politica', 'politico', 'presidente', 'gobierno', 'asamblea', 'elecciones', 'paro', 'nacional', 'muerte', 'cruzada', 'correa', 'iza', '#paronacional', 'ecuador']


In [163]:
resultados_holdout: dict[int, dict[str, Any]] = {}

In [164]:
for dimension in DIMENSIONES:
    print("\n" + "=" * 100)
    print(f"HOLDOUT SIN FUGA — DIMENSIÓN {dimension}")
    print("=" * 100)

    resultados_holdout[int(dimension)] = run_leakage_free_holdout(
        tweets_desarrollo=tweets_desarrollo,
        tokens_trends=tokens_trends,
        dimension=int(dimension),
        semillas=semillas_politicas,
        top_n=EXPANSION_TOP_N_FINAL,
        test_size=TEST_SIZE,
        epochs=W2V_EPOCHS,
        seed=RANDOM_STATE,
    )


HOLDOUT SIN FUGA — DIMENSIÓN 50

HOLDOUT SIN FUGA — DIMENSIÓN 100

HOLDOUT SIN FUGA — DIMENSIÓN 200


In [165]:
resumen_holdout = summarize_dimension_holdouts(resultados_holdout)

In [166]:
metricas_holdout = pd.concat(
    [resultado["metricas"] for resultado in resultados_holdout.values()],
    ignore_index=True,
)

In [167]:
display_bokeh_table(
    resumen_holdout,
    "Cluster selection, expansion and class balance by dimension",
    ancho=1350,
    alto=300,
)

In [168]:
display_bokeh_table(
    metricas_holdout,
    "Leakage-free holdout: consistency with pseudo-labels",
    ancho=1500,
    alto=480,
    max_filas=40,
)

In [169]:
tablas_top100_suplementarias = []

for dimension in sorted(resultados_holdout.keys()):
    resultado = resultados_holdout[dimension]
    detalle = (
        resultado["pipeline"]["palabras_anadidas"]
        .copy()
    )
    if detalle.empty:
        continue
    detalle.insert(
        0,
        "Dimension",
        int(dimension)
    )
    detalle = detalle[
        [
            "Dimension",
            "Rango",
            "Palabra",
            "Similitud_coseno_politica",
            "Distancia_coseno_politica",
        ]
    ].copy()
    tablas_top100_suplementarias.append(detalle)

In [170]:
tabla_top100_suplementaria = pd.concat(
    tablas_top100_suplementarias,
    ignore_index=True
)

In [171]:
tabla_top100_suplementaria = (
    tabla_top100_suplementaria
    .rename(
        columns={
            "Dimension": "Embedding_dimension",
            "Rango": "Ranking",
            "Palabra": "Term",
            "Similitud_coseno_politica": "Cosine_similarity",
            "Distancia_coseno_politica": "Cosine_distance",
        }
    )
    .sort_values(
        ["Embedding_dimension", "Ranking"]
    )
    .reset_index(drop=True)
)

In [172]:
verificacion_top100 = (
    tabla_top100_suplementaria
    .groupby("Embedding_dimension")
    .agg(
        N_terms=("Term", "size"),
        Minimum_similarity=("Cosine_similarity", "min"),
        Maximum_similarity=("Cosine_similarity", "max"),
    )
    .reset_index()
)

display(verificacion_top100)

,Embedding_dimension,N_terms,Minimum_similarity,Maximum_similarity
0,50,100,0.711283,0.901391
1,100,100,0.720841,0.904289
2,200,100,0.720263,0.909156


In [173]:
display_bokeh_table(
    tabla_top100_suplementaria,
    "Top-100 expanded terms by embedding dimension",
    ancho=1450,
    alto=500,
    max_filas=300,
)

In [174]:
resultado_principal = resultados_holdout[DIMENSION_BENCHMARK]

In [175]:
pipe_principal = resultado_principal["pipeline"]

In [176]:
display_bokeh_table(
    pipe_principal["tabla_k"],
    f"Objective cluster-number selection — {DIMENSION_BENCHMARK}D",
    ancho=1000,
    alto=330,
)

In [177]:
plot_k_validation(
    pipe_principal["tabla_k"],
    DIMENSION_BENCHMARK,
)

In [178]:
print(f"Selected k: {pipe_principal['k_seleccionado']}")
print(f"Political cluster: {pipe_principal['cluster_politico']}")
print(f"Requested Top-N: {pipe_principal['top_n_solicitado']}")
print(f"Added terms: {len(pipe_principal['palabras_anadidas']):,}")
print(f"Observed cosine cutoff: {pipe_principal['umbral_similitud']:.6f}")

Selected k: 2
Political cluster: 1
Requested Top-N: 100
Added terms: 100
Observed cosine cutoff: 0.720841


In [179]:
display_bokeh_table(
    summarize_clusters(
        pipe_principal["palabras"],
        pipe_principal["labels_palabras_1"],
    ),
    "Initial vocabulary clusters from trend terms",
    ancho=1300,
    alto=320,
)

In [180]:
display_bokeh_table(
    pipe_principal["palabras_anadidas"],
    f"Terms added to Scenario 2 — Top-{EXPANSION_TOP_N_FINAL}",
    ancho=1250,
    alto=420,
    max_filas=60,
)

In [181]:
comparacion_clustering, labels_hdbscan = compare_kmeans_hdbscan(
    resultado_principal,
    HDBSCAN_MIN_CLUSTER_SIZE,
    HDBSCAN_MIN_SAMPLES,
)

In [182]:
display_bokeh_table(
    comparacion_clustering,
    "K-means vs HDBSCAN on training embeddings",
    ancho=1150,
    alto=230,
)

In [183]:
plot_tsne(
    pipe_principal["palabras"],
    pipe_principal["vectores_palabras_cluster"],
    pipe_principal["labels_palabras_1"],
    f"t-SNE — K-Means — {DIMENSION_BENCHMARK}D",
    RANDOM_STATE,
)

In [184]:
if np.any(labels_hdbscan != -1):
    plot_tsne(
        pipe_principal["palabras"],
        pipe_principal["vectores_palabras_cluster"],
        labels_hdbscan,
        f"t-SNE — HDBSCAN — {DIMENSION_BENCHMARK}D",
        RANDOM_STATE,
    )

In [185]:
sensibilidad_top_n = evaluate_topn_sensitivity(
    resultado_principal,
    EXPANSION_TOP_N_CANDIDATOS,
    RANDOM_STATE,
)

In [186]:
display_bokeh_table(
    sensibilidad_top_n,
    "Top-N lexicon-expansion sensitivity analysis",
    ancho=1500,
    alto=440,
    max_filas=50,
)

In [187]:
plot_topn_coverage_and_cutoff(
    sensibilidad_top_n,
    dimension=DIMENSION_BENCHMARK
)

In [188]:
plot_topn_sensitivity(sensibilidad_top_n, "F1-score")

In [189]:
plot_topn_sensitivity(sensibilidad_top_n, "Recall")

# 7. Repeated cross-validation and statistical testing

In [190]:
def run_repeated_cross_validation(tweets_desarrollo: pd.DataFrame, tokens_trends: Sequence[Sequence[str]], dimensiones: Sequence[int], semillas: Sequence[str], top_n: int, n_splits: int, n_repeats: int, epochs: int, seed: int) -> pd.DataFrame:
    """Run repeated cross validation. Retains the original experimental logic and data-partition rules."""
    if n_splits < 2 or n_repeats < 1:
        raise ValueError("At least two folds and one repetition are required.")

    estrato = tweets_desarrollo["evento"].fillna("Sin evento").astype(str).to_numpy()
    splitter = RepeatedStratifiedKFold(
        n_splits=int(n_splits),
        n_repeats=int(n_repeats),
        random_state=seed,
    )
    filas: list[pd.DataFrame] = []

    for dimension in dimensiones:
        print(f"\nRepeated CV: dimension {dimension}")
        for contador, (idx_train, idx_test) in enumerate(splitter.split(np.zeros(len(tweets_desarrollo)), estrato), start=1):
            repeticion = (contador - 1) // n_splits + 1
            fold = (contador - 1) % n_splits + 1
            print(f"  Repeat {repeticion}/{n_repeats}, fold {fold}/{n_splits}")

            train_df = tweets_desarrollo.iloc[idx_train].reset_index(drop=True)
            test_df = tweets_desarrollo.iloc[idx_test].reset_index(drop=True)
            tokens_train = extract_dataframe_tokens(train_df, "tokens_sin_stopwords")
            tokens_test = extract_dataframe_tokens(test_df, "tokens_sin_stopwords")

            pipe = build_train_semantic_pipeline(
                tokens_train,
                tokens_trends,
                int(dimension),
                semillas,
                K_CANDIDATOS,
                int(top_n),
                int(epochs),
                seed + contador,
            )
            X_train = vectorize_documents(tokens_train, pipe["modelo_word2vec"], agregacion="suma")
            X_test = vectorize_documents(tokens_test, pipe["modelo_word2vec"], agregacion="suma")

            for escenario, centroides in [("Scenario 1", pipe["centroides_1"]), ("Scenario 2", pipe["centroides_2"])]:
                y_train, _, _ = label_with_centroids(X_train, centroides, pipe["cluster_politico"])
                y_test, _, _ = label_with_centroids(X_test, centroides, pipe["cluster_politico"])
                tabla = train_and_evaluate_classifiers(
                    X_train,
                    y_train,
                    X_test,
                    y_test,
                    int(dimension),
                    escenario,
                    seed + contador,
                    KNN_VECINOS,
                )
                tabla["Repeticion"] = int(repeticion)
                tabla["Fold"] = int(fold)
                tabla["k_seleccionado"] = int(pipe["k_seleccionado"])
                tabla["N_terminos_anadidos"] = int(len(pipe["palabras_anadidas"])) if escenario == "Scenario 2" else 0
                tabla["Umbral_similitud"] = float(pipe["umbral_similitud"]) if escenario == "Scenario 2" else np.nan
                tabla["Politicos_test_pct"] = float(y_test.mean() * 100.0)
                filas.append(tabla)

    return pd.concat(filas, ignore_index=True) if filas else pd.DataFrame()

In [191]:
def summarize_cv_confidence_intervals(cv_resultados: pd.DataFrame, metricas: Sequence[str]) -> pd.DataFrame:
    """Summarize cv confidence intervals. Retains the original experimental logic and data-partition rules."""
    filas: list[dict[str, Any]] = []
    grupos = cv_resultados.groupby(["Dimension", "Scenario", "Algorithm"], dropna=False)

    for claves, sub in grupos:
        dim, escenario, algoritmo = claves
        for metrica in metricas:
            valores = pd.to_numeric(sub[metrica], errors="coerce").dropna().to_numpy(dtype=float)
            n = len(valores)
            if n == 0:
                continue
            media = float(np.mean(valores))
            std = float(np.std(valores, ddof=1)) if n > 1 else 0.0
            if n > 1:
                margen = float(student_t.ppf(0.975, df=n - 1) * std / math.sqrt(n))
            else:
                margen = 0.0
            limite_inf = max(0.0, media - margen)
            limite_sup = min(1.0, media + margen)
            filas.append({
                "Dimension": int(dim),
                "Scenario": str(escenario),
                "Algorithm": str(algoritmo),
                "Metric": str(metrica),
                "N": int(n),
                "Mean": media,
                "Std_dev": std,
                "CI95_lower": limite_inf,
                "CI95_upper": limite_sup,
            })

    return pd.DataFrame(filas)

In [192]:
def holm_adjust_pvalues(p_values: Sequence[float]) -> np.ndarray:
    """Execute holm adjust pvalues. Retains the original experimental logic and data-partition rules."""
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    if m == 0:
        return np.array([], dtype=float)

    orden = np.argsort(p)
    ajustados_orden = np.empty(m, dtype=float)
    max_prev = 0.0
    for rango, idx in enumerate(orden):
        valor = min(1.0, (m - rango) * p[idx])
        max_prev = max(max_prev, valor)
        ajustados_orden[rango] = max_prev

    salida = np.empty(m, dtype=float)
    for rango, idx in enumerate(orden):
        salida[idx] = ajustados_orden[rango]
    return salida

In [193]:
def run_paired_wilcoxon_tests(cv_resultados: pd.DataFrame, metricas: Sequence[str]) -> pd.DataFrame:
    """Run paired wilcoxon tests. Retains the original experimental logic and data-partition rules."""
    filas: list[dict[str, Any]] = []
    for (dimension, algoritmo), sub in cv_resultados.groupby(["Dimension", "Algorithm"]):
        for metrica in metricas:
            piv = sub.pivot_table(
                index=["Repeticion", "Fold"],
                columns="Scenario",
                values=metrica,
                aggfunc="first",
            ).dropna()
            if "Scenario 1" not in piv.columns or "Scenario 2" not in piv.columns or len(piv) < 2:
                continue

            a = piv["Scenario 1"].to_numpy(dtype=float)
            b = piv["Scenario 2"].to_numpy(dtype=float)
            if np.allclose(a, b):
                estadistico, p_valor = 0.0, 1.0
            else:
                prueba = wilcoxon(b, a, alternative="two-sided", zero_method="wilcox")
                estadistico, p_valor = float(prueba.statistic), float(prueba.pvalue)

            filas.append({
                "Dimension": int(dimension),
                "Algorithm": str(algoritmo),
                "Metric": str(metrica),
                "N_pares": int(len(piv)),
                "Media_E1": float(np.mean(a)),
                "Media_E2": float(np.mean(b)),
                "Diferencia_media_E2_menos_E1": float(np.mean(b - a)),
                "Wilcoxon_W": estadistico,
                "p_valor": p_valor,
            })

    tabla = pd.DataFrame(filas)
    if not tabla.empty:
        tabla["p_ajustado_Holm"] = holm_adjust_pvalues(tabla["p_valor"].to_numpy())
        tabla["Significativo_0_05"] = tabla["p_ajustado_Holm"] < 0.05
    return tabla

In [194]:
def plot_cv_intervals(resumen_cv: pd.DataFrame, metrica: str = "F1-score") -> None:
    """Plot cv intervals. Retains the original experimental logic and data-partition rules."""
    datos = resumen_cv[resumen_cv["Metric"] == metrica].copy()
    if datos.empty:
        print(f"No data available for metric {metrica}.")
        return

    datos["Categoria"] = (
        datos["Dimension"].astype(str)
        + "D | "
        + datos["Algorithm"].astype(str)
        + " | "
        + datos["Scenario"].astype(str).str.replace("Escenario ", "E", regex=False)
    )
    datos = datos.sort_values(["Dimension", "Algorithm", "Scenario"]).reset_index(drop=True)
    source = ColumnDataSource(datos)
    p = figure(
        x_range=datos["Categoria"].tolist(),
        height=480,
        width=1100,
        title=f"{metrica}: mean and 95% CI: repeated cross-validation",
        y_axis_label=metrica,
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )
    p.scatter(x="Categoria", y="Mean", source=source, size=9)
    p.segment(x0="Categoria", y0="CI95_lower", x1="Categoria", y1="CI95_upper", source=source, line_width=2)
    p.xaxis.major_label_orientation = 1.1
    p.add_tools(HoverTool(tooltips=[
        ("Configuration", "@Categoria"),
        ("Mean", "@Media{0.0000}"),
        ("Std", "@Desv_std{0.0000}"),
        ("IC95", "[@IC95_inf{0.0000}, @IC95_sup{0.0000}]"),
        ("N", "@N"),
    ]))
    show(p)

In [195]:
cv_resultados = run_repeated_cross_validation(
    tweets_desarrollo=tweets_desarrollo,
    tokens_trends=tokens_trends,
    dimensiones=DIMENSIONES,
    semillas=semillas_politicas,
    top_n=EXPANSION_TOP_N_FINAL,
    n_splits=CV_FOLDS,
    n_repeats=CV_REPEATS,
    epochs=CV_EPOCHS,
    seed=RANDOM_STATE,
)

cv_resumen = summarize_cv_confidence_intervals(
    cv_resultados,
    list(METRICAS_ESTADISTICAS),
)

wilcoxon_escenarios = run_paired_wilcoxon_tests(
    cv_resultados,
    ["Precision", "Recall", "F1-score", "F2 score"],
)

display_bokeh_table(
    cv_resumen,
    "Repeated cross-validation: mean, standard deviation and 95% CI",
    ancho=1450,
    alto=500,
    max_filas=70,
)
display_bokeh_table(
    wilcoxon_escenarios,
    "Paired Wilcoxon, Scenario 2 vs Scenario 1, Holm-adjusted",
    ancho=1400,
    alto=430,
    max_filas=60,
)
plot_cv_intervals(cv_resumen, "F1-score")


Repeated CV: dimension 50
  Repeat 1/2, fold 1/5
  Repeat 1/2, fold 2/5
  Repeat 1/2, fold 3/5
  Repeat 1/2, fold 4/5
  Repeat 1/2, fold 5/5
  Repeat 2/2, fold 1/5
  Repeat 2/2, fold 2/5
  Repeat 2/2, fold 3/5
  Repeat 2/2, fold 4/5
  Repeat 2/2, fold 5/5

Repeated CV: dimension 100
  Repeat 1/2, fold 1/5
  Repeat 1/2, fold 2/5
  Repeat 1/2, fold 3/5
  Repeat 1/2, fold 4/5
  Repeat 1/2, fold 5/5
  Repeat 2/2, fold 1/5
  Repeat 2/2, fold 2/5
  Repeat 2/2, fold 3/5
  Repeat 2/2, fold 4/5
  Repeat 2/2, fold 5/5

Repeated CV: dimension 200
  Repeat 1/2, fold 1/5
  Repeat 1/2, fold 2/5
  Repeat 1/2, fold 3/5
  Repeat 1/2, fold 4/5
  Repeat 1/2, fold 5/5
  Repeat 2/2, fold 1/5
  Repeat 2/2, fold 2/5
  Repeat 2/2, fold 3/5
  Repeat 2/2, fold 4/5
  Repeat 2/2, fold 5/5


In [196]:
metricas_apendice_b = (
    wilcoxon_escenarios["Metric"]
    .drop_duplicates()
    .tolist()
)

In [197]:
print(
    "Metrics included in Appendix B:",
    metricas_apendice_b
)

Metrics included in Appendix B: ['Precision', 'Recall', 'F1-score', 'F2 score']


In [198]:
cv_apendice_b = cv_resumen[
    cv_resumen["Metric"].isin(
        metricas_apendice_b
    )
].copy()

In [199]:
cv_e1 = (
    cv_apendice_b[
        cv_apendice_b["Scenario"] == "Scenario 1"
    ][
        [
            "Dimension",
            "Algorithm",
            "Metric",
            "N",
            "Mean",
            "Std_dev",
            "CI95_lower",
            "CI95_upper",
        ]
    ]
    .copy()
    .rename(
        columns={
            "N": "N_E1",
            "Mean": "E1_Mean",
            "Std_dev": "E1_SD",
            "CI95_lower": "E1_CI95_Lower",
            "CI95_upper": "E1_CI95_Upper",
        }
    )
)

In [200]:
cv_e2 = (
    cv_apendice_b[
        cv_apendice_b["Scenario"] == "Scenario 2"
    ][
        [
            "Dimension",
            "Algorithm",
            "Metric",
            "N",
            "Mean",
            "Std_dev",
            "CI95_lower",
            "CI95_upper",
        ]
    ]
    .copy()
    .rename(
        columns={
            "N": "N_E2",
            "Mean": "E2_Mean",
            "Std_dev": "E2_SD",
            "CI95_lower": "E2_CI95_Lower",
            "CI95_upper": "E2_CI95_Upper",
        }
    )
)

In [201]:
tabla_appendix_b_cv = cv_e1.merge(
    cv_e2,
    on=[
        "Dimension",
        "Algorithm",
        "Metric",
    ],
    how="inner",
    validate="one_to_one",
)

In [202]:
wilcoxon_apendice_b = wilcoxon_escenarios[
    [
        "Dimension",
        "Algorithm",
        "Metric",
        "N_pares",
        "Media_E1",
        "Media_E2",
        "Diferencia_media_E2_menos_E1",
        "Wilcoxon_W",
        "p_valor",
        "p_ajustado_Holm",
        "Significativo_0_05",
    ]
].copy()

In [203]:
tabla_appendix_b_cv = tabla_appendix_b_cv.merge(
    wilcoxon_apendice_b,
    on=[
        "Dimension",
        "Algorithm",
        "Metric",
    ],
    how="inner",
    validate="one_to_one",
)

In [204]:
if not np.allclose(
    tabla_appendix_b_cv["E1_Mean"],
    tabla_appendix_b_cv["Media_E1"],
    equal_nan=True,
):
    raise ValueError(
        "Scenario 1 means from cv_resumen and "
        "wilcoxon_escenarios do not match."
    )

In [205]:
if not np.allclose(
    tabla_appendix_b_cv["E2_Mean"],
    tabla_appendix_b_cv["Media_E2"],
    equal_nan=True,
):
    raise ValueError(
        "Scenario 2 means from cv_resumen and "
        "wilcoxon_escenarios do not match."
    )

In [206]:
tabla_appendix_b_cv = (
    tabla_appendix_b_cv
    .drop(
        columns=[
            "Media_E1",
            "Media_E2",
        ]
    )
)

In [207]:
tabla_appendix_b_cv = (
    tabla_appendix_b_cv
    .rename(
        columns={
            "Dimension": "Embedding_dimension",
            "Algorithm": "Model",
            "Metric": "Metric",
            "N_pares": "N_pairs",
            "Diferencia_media_E2_menos_E1":
                "Mean_difference_E2_minus_E1",
            "p_valor": "p_value",
            "p_ajustado_Holm": "Holm_adjusted_p",
            "Significativo_0_05":
                "Significant_after_Holm",
        }
    )
)

In [208]:
mapa_modelos = {
    "Decision Tree": "Decision Tree",
    "KNN": "KNN",
    "Naïve Bayes": "Naive Bayes",
    "SVM": "SVM",
}

mapa_metricas = {
    "Precision": "Precision",
    "Recall": "Recall",
    "F1-score": "F1-score",
    "F2 score": "F2-score",
}

In [209]:
tabla_appendix_b_cv["Model"] = (
    tabla_appendix_b_cv["Model"]
    .replace(mapa_modelos)
)

tabla_appendix_b_cv["Metric"] = (
    tabla_appendix_b_cv["Metric"]
    .replace(mapa_metricas)
)

In [210]:
orden_modelos = {
    "Decision Tree": 0,
    "KNN": 1,
    "Naive Bayes": 2,
    "SVM": 3,
}

orden_metricas = {
    "Precision": 0,
    "Recall": 1,
    "F1-score": 2,
    "F2-score": 3,
}

In [211]:
tabla_appendix_b_cv["_orden_modelo"] = (
    tabla_appendix_b_cv["Model"]
    .map(orden_modelos)
)

tabla_appendix_b_cv["_orden_metrica"] = (
    tabla_appendix_b_cv["Metric"]
    .map(orden_metricas)
)

tabla_appendix_b_cv = (
    tabla_appendix_b_cv
    .sort_values(
        [
            "Embedding_dimension",
            "_orden_modelo",
            "_orden_metrica",
        ]
    )
    .drop(
        columns=[
            "_orden_modelo",
            "_orden_metrica",
        ]
    )
    .reset_index(drop=True)
)

In [212]:
n_esperado = len(
    wilcoxon_escenarios
)

In [213]:
if len(tabla_appendix_b_cv) != n_esperado:
    raise ValueError(
        "The Appendix B row count does not match "
        f"con Wilcoxon. Esperadas: {n_esperado}. "
        f"Obtenidas: {len(tabla_appendix_b_cv)}."
    )

In [214]:
columnas_criticas = [
    "E1_Mean",
    "E1_SD",
    "E1_CI95_Lower",
    "E1_CI95_Upper",
    "E2_Mean",
    "E2_SD",
    "E2_CI95_Lower",
    "E2_CI95_Upper",
    "Wilcoxon_W",
    "p_value",
    "Holm_adjusted_p",
]

if tabla_appendix_b_cv[
    columnas_criticas
].isna().any().any():
    raise ValueError(
        "Appendix B has missing values "
        "in mandatory statistical columns."
    )

In [215]:
tabla_appendix_b_cv_table = (
    tabla_appendix_b_cv[
        [
            "Embedding_dimension",
            "Model",
            "Metric",
            "E1_Mean",
            "E1_SD",
            "E1_CI95_Lower",
            "E1_CI95_Upper",
            "E2_Mean",
            "E2_SD",
            "E2_CI95_Lower",
            "E2_CI95_Upper",
            "Wilcoxon_W",
            "p_value",
            "Holm_adjusted_p",
            "Significant_after_Holm",
        ]
    ]
    .copy()
)

In [216]:
tabla_appendix_b_cv_table["Scenario_1_Mean_SD"] = (
    tabla_appendix_b_cv_table.apply(
        lambda fila:
        f"{fila['E1_Mean']:.4f} ± "
        f"{fila['E1_SD']:.4f}",
        axis=1,
    )
)

In [217]:
tabla_appendix_b_cv_table["Scenario_1_CI95"] = (
    tabla_appendix_b_cv_table.apply(
        lambda fila:
        f"[{fila['E1_CI95_Lower']:.4f}, "
        f"{fila['E1_CI95_Upper']:.4f}]",
        axis=1,
    )
)

In [218]:
tabla_appendix_b_cv_table["Scenario_2_Mean_SD"] = (
    tabla_appendix_b_cv_table.apply(
        lambda fila:
        f"{fila['E2_Mean']:.4f} ± "
        f"{fila['E2_SD']:.4f}",
        axis=1,
    )
)

In [219]:
tabla_appendix_b_cv_table["Scenario_2_CI95"] = (
    tabla_appendix_b_cv_table.apply(
        lambda fila:
        f"[{fila['E2_CI95_Lower']:.4f}, "
        f"{fila['E2_CI95_Upper']:.4f}]",
        axis=1,
    )
)

In [220]:
tabla_appendix_b_cv_table = (
    tabla_appendix_b_cv_table[
        [
            "Embedding_dimension",
            "Model",
            "Metric",
            "Scenario_1_Mean_SD",
            "Scenario_1_CI95",
            "Scenario_2_Mean_SD",
            "Scenario_2_CI95",
            "Wilcoxon_W",
            "p_value",
            "Holm_adjusted_p",
            "Significant_after_Holm",
        ]
    ]
)

In [221]:
display_bokeh_table(
    tabla_appendix_b_cv_table,
    "Complete repeated cross-validation and statistical results",
    ancho=1750,
    alto=520,
    max_filas=60,
)

In [222]:
print(
    "Total number of rows:",
    len(tabla_appendix_b_cv)
)

print(
    "Dimensions:",
    sorted(
        tabla_appendix_b_cv[
            "Embedding_dimension"
        ].unique().tolist()
    )
)

print(
    "Models:",
    tabla_appendix_b_cv[
        "Model"
    ].unique().tolist()
)

print(
    "Metrics:",
    tabla_appendix_b_cv[
        "Metric"
    ].unique().tolist()
)

print(
    "Significant comparisons after Holm correction:",
    int(
        tabla_appendix_b_cv[
            "Significant_after_Holm"
        ].sum()
    )
)

Total number of rows: 48
Dimensions: [50, 100, 200]
Models: ['Decision Tree', 'KNN', 'Naive Bayes', 'SVM']
Metrics: ['Precision', 'Recall', 'F1-score', 'F2-score']
Significant comparisons after Holm correction: 0


# 8. Hashtag-blind external silver-standard validation

In [223]:
def evaluate_hashtag_external_validation(tweets_desarrollo: pd.DataFrame, holdout_silver: pd.DataFrame, trends_ciegos: pd.DataFrame, dimension: int, semillas: Sequence[str], top_n: int, epochs: int, seed: int) -> dict[str, Any]:
    """Evaluate hashtag external validation. Retains the original experimental logic and data-partition rules."""
    if "etiqueta_silver_hashtag" not in holdout_silver.columns:
        raise KeyError("The holdout must contain `etiqueta_silver_hashtag`.")

    y_silver = holdout_silver["etiqueta_silver_hashtag"].astype(int).to_numpy()
    if len(np.unique(y_silver)) < 2:
        raise ValueError("The silver reference must contain both classes 0 and 1.")

    tokens_train = extract_dataframe_tokens(tweets_desarrollo, "tokens_sin_stopwords")
    tokens_holdout = extract_dataframe_tokens(holdout_silver, "tokens_sin_stopwords")
    tokens_trends = extract_dataframe_tokens(trends_ciegos, "tokens_sin_stopwords")

    pipe = build_train_semantic_pipeline(
        tokens_train,
        tokens_trends,
        int(dimension),
        semillas,
        K_CANDIDATOS,
        int(top_n),
        int(epochs),
        int(seed),
    )

    X_train = vectorize_documents(tokens_train, pipe["modelo_word2vec"], agregacion="suma")
    X_holdout = vectorize_documents(tokens_holdout, pipe["modelo_word2vec"], agregacion="suma")

    y1_train, _, _ = label_with_centroids(X_train, pipe["centroides_1"], pipe["cluster_politico"])
    y2_train, _, _ = label_with_centroids(X_train, pipe["centroides_2"], pipe["cluster_politico"])
    y1_holdout, _, _ = label_with_centroids(X_holdout, pipe["centroides_1"], pipe["cluster_politico"])
    y2_holdout, _, _ = label_with_centroids(X_holdout, pipe["centroides_2"], pipe["cluster_politico"])

    filas: list[dict[str, Any]] = []
    predicciones: dict[str, np.ndarray] = {
        "Pseudo_E1": y1_holdout.astype(int),
        "Pseudo_E2": y2_holdout.astype(int),
    }

    for escenario, y_train, y_pseudo in [
        ("Scenario 1", y1_train, y1_holdout),
        ("Scenario 2", y2_train, y2_holdout),
    ]:
        fila_pseudo: dict[str, Any] = {
            "Dimension": int(dimension),
            "Scenario": escenario,
            "Modelo": "Direct pseudo-labeling",
            "Reference": "Hashtag silver standard: hashtag-blind input features",
            "N_holdout": int(len(y_silver)),
        }
        fila_pseudo.update(calculate_silver_metrics(y_silver, y_pseudo))
        filas.append(fila_pseudo)

        for nombre, modelo in create_classifiers(KNN_VECINOS, seed).items():
            modelo.fit(X_train, y_train)
            pred = np.asarray(modelo.predict(X_holdout)).astype(int)
            predicciones[f"{nombre}_{'E1' if escenario == 'Scenario 1' else 'E2'}"] = pred
            fila: dict[str, Any] = {
                "Dimension": int(dimension),
                "Scenario": escenario,
                "Modelo": nombre,
                "Reference": "Hashtag silver standard: hashtag-blind input features",
                "N_holdout": int(len(y_silver)),
            }
            fila.update(calculate_silver_metrics(y_silver, pred))
            filas.append(fila)

    # Report both natural and class-balanced direct pseudo-label results.
    rng = np.random.default_rng(seed)
    idx0 = np.flatnonzero(y_silver == 0)
    idx1 = np.flatnonzero(y_silver == 1)
    n_balance = int(min(len(idx0), len(idx1)))
    idx_balance = np.concatenate([
        rng.choice(idx0, size=n_balance, replace=False),
        rng.choice(idx1, size=n_balance, replace=False),
    ]) if n_balance > 0 else np.arange(len(y_silver))
    rng.shuffle(idx_balance)

    balance_filas: list[dict[str, Any]] = []
    for nombre_pred in ["Pseudo_E1", "Pseudo_E2"]:
        fila = {
            "Prediccion": nombre_pred,
            "Muestra": "Natural",
            "N": int(len(y_silver)),
            "Positivos": int(y_silver.sum()),
            "Negativos": int((y_silver == 0).sum()),
        }
        fila.update(calculate_silver_metrics(y_silver, predicciones[nombre_pred]))
        balance_filas.append(fila)
        fila_bal = {
            "Prediccion": nombre_pred,
            "Muestra": "Balanceada 1:1",
            "N": int(len(idx_balance)),
            "Positivos": int(y_silver[idx_balance].sum()),
            "Negativos": int((y_silver[idx_balance] == 0).sum()),
        }
        fila_bal.update(calculate_silver_metrics(y_silver[idx_balance], predicciones[nombre_pred][idx_balance]))
        balance_filas.append(fila_bal)

    por_evento: list[dict[str, Any]] = []
    if "evento" in holdout_silver.columns:
        for evento, indices in holdout_silver.groupby("evento").groups.items():
            idx = np.asarray(list(indices), dtype=int)
            if len(np.unique(y_silver[idx])) < 2:
                continue
            for nombre_pred in ["Pseudo_E1", "Pseudo_E2"]:
                fila_evento: dict[str, Any] = {
                    "Dimension": int(dimension),
                    "Evento": str(evento),
                    "Prediccion": nombre_pred,
                    "N": int(len(idx)),
                }
                fila_evento.update(calculate_silver_metrics(y_silver[idx], predicciones[nombre_pred][idx]))
                por_evento.append(fila_evento)

    mcnemar = exact_mcnemar_test(y_silver, y1_holdout, y2_holdout)
    mcnemar.insert(0, "Dimension", int(dimension))
    mcnemar.insert(1, "Comparacion", "Pseudo E1 vs Pseudo E2")

    return {
        "dimension": int(dimension),
        "pipeline": pipe,
        "metricas": pd.DataFrame(filas),
        "metricas_balance": pd.DataFrame(balance_filas),
        "metricas_evento": pd.DataFrame(por_evento),
        "mcnemar": mcnemar,
        "y_silver": y_silver,
        "predicciones": predicciones,
        "holdout": holdout_silver.reset_index(drop=True),
    }

In [224]:
def evaluate_hashtag_human_audit(
    anotaciones: pd.DataFrame,
    resultado_silver: Mapping[str, Any],
) -> dict[str, pd.DataFrame]:
    """Evaluate hashtag human audit. Retains the original experimental logic and data-partition rules."""
    holdout = resultado_silver["holdout"].copy().reset_index(drop=True)
    if "id_registro" not in holdout.columns:
        raise KeyError("The silver holdout is missing `id_registro`.")

    detalle = holdout[
        [c for c in [
            "id_registro",
            "evento",
            "texto_para_anotacion",
            "texto_limpio",
            "etiqueta_silver_hashtag",
        ] if c in holdout.columns]
    ].copy()

    predicciones = resultado_silver["predicciones"]
    for nombre, valores in predicciones.items():
        detalle[nombre] = np.asarray(valores).astype(int)

    humanas = anotaciones[["id_registro", "etiqueta_humana"]].copy()
    humanas["id_registro"] = humanas["id_registro"].astype(str)
    detalle["id_registro"] = detalle["id_registro"].astype(str)

    detalle = humanas.merge(
        detalle,
        on="id_registro",
        how="inner",
        validate="one_to_one",
    )

    if len(detalle) != len(humanas):
        raise RuntimeError(
            "Not all hashtag audit annotations matched the silver holdout."
        )

    y_humano = detalle["etiqueta_humana"].astype(int).to_numpy()
    dimension = int(resultado_silver["dimension"])

    columnas_pred = ["etiqueta_silver_hashtag"] + list(predicciones.keys())
    filas: list[dict[str, Any]] = []

    for columna in columnas_pred:
        y_pred = detalle[columna].astype(int).to_numpy()
        fila: dict[str, Any] = {
            "Dimension": dimension,
            "Prediccion": columna,
            "Reference": "Human-blind audit of the silver standard",
            "N": int(len(detalle)),
        }
        fila.update(calculate_silver_metrics(y_humano, y_pred))
        filas.append(fila)

    comparaciones: list[pd.DataFrame] = []

    if "Pseudo_E2" in detalle.columns:
        prueba = exact_mcnemar_test(
            y_humano,
            detalle["etiqueta_silver_hashtag"].astype(int).to_numpy(),
            detalle["Pseudo_E2"].astype(int).to_numpy(),
        )
        prueba.insert(0, "Dimension", dimension)
        prueba.insert(1, "Comparacion", "Silver hashtag vs Pseudo E2")
        comparaciones.append(prueba)

    if {"Pseudo_E1", "Pseudo_E2"}.issubset(detalle.columns):
        prueba = exact_mcnemar_test(
            y_humano,
            detalle["Pseudo_E1"].astype(int).to_numpy(),
            detalle["Pseudo_E2"].astype(int).to_numpy(),
        )
        prueba.insert(0, "Dimension", dimension)
        prueba.insert(1, "Comparacion", "Pseudo E1 vs Pseudo E2")
        comparaciones.append(prueba)

    detalle["acierto_silver"] = (
        detalle["etiqueta_silver_hashtag"].astype(int)
        == detalle["etiqueta_humana"].astype(int)
    )
    if "Pseudo_E1" in detalle.columns:
        detalle["acierto_pseudo_E1"] = (
            detalle["Pseudo_E1"].astype(int)
            == detalle["etiqueta_humana"].astype(int)
        )
    if "Pseudo_E2" in detalle.columns:
        detalle["acierto_pseudo_E2"] = (
            detalle["Pseudo_E2"].astype(int)
            == detalle["etiqueta_humana"].astype(int)
        )
        detalle["desacuerdo_silver_pseudoE2"] = (
            detalle["etiqueta_silver_hashtag"].astype(int)
            != detalle["Pseudo_E2"].astype(int)
        )

    mcnemar = (
        pd.concat(comparaciones, ignore_index=True)
        if comparaciones
        else pd.DataFrame()
    )

    return {
        "metricas": pd.DataFrame(filas),
        "mcnemar": mcnemar,
        "detalle": detalle,
    }

In [225]:
def update_hashtag_model_audit(
    ruta: Path,
    detalle: pd.DataFrame,
) -> Path:
    """Update hashtag model audit. Retains the original experimental logic and data-partition rules."""
    if not ruta.exists():
        raise FileNotFoundError(f"No existe el archivo: {ruta}")

    with pd.ExcelWriter(
        ruta,
        engine="openpyxl",
        mode="a",
        if_sheet_exists="replace",
    ) as writer:
        detalle.to_excel(
            writer,
            sheet_name="Auditoria_Modelo",
            index=False,
        )

    return ruta

In [226]:
resultados_silver: dict[int, dict[str, Any]] = {}

In [227]:
for dimension in DIMENSIONES:
    print("\n" + "=" * 100)
    print(f"VALIDACIÓN HASHTAG-BLIND — DIMENSIÓN {dimension}")
    print("=" * 100)

    resultados_silver[int(dimension)] = evaluate_hashtag_external_validation(
        tweets_desarrollo=tweets_desarrollo_ciego_hashtags,
        holdout_silver=silver_hashtags_ciego,
        trends_ciegos=trends_ciegos_hashtags,
        dimension=int(dimension),
        semillas=semillas_politicas_sin_hashtags,
        top_n=EXPANSION_TOP_N_FINAL,
        epochs=W2V_EPOCHS,
        seed=RANDOM_STATE,
    )


VALIDACIÓN HASHTAG-BLIND — DIMENSIÓN 50

VALIDACIÓN HASHTAG-BLIND — DIMENSIÓN 100

VALIDACIÓN HASHTAG-BLIND — DIMENSIÓN 200


In [228]:
metricas_silver = pd.concat(
    [r["metricas"] for r in resultados_silver.values()],
    ignore_index=True,
)

In [229]:
metricas_silver_balance = pd.concat(
    [r["metricas_balance"] for r in resultados_silver.values()],
    ignore_index=True,
)

In [230]:
mcnemar_silver = pd.concat(
    [r["mcnemar"] for r in resultados_silver.values()],
    ignore_index=True,
)

In [231]:
metricas_silver_evento = pd.concat(
    [
        r["metricas_evento"]
        for r in resultados_silver.values()
        if not r["metricas_evento"].empty
    ],
    ignore_index=True,
) if any(not r["metricas_evento"].empty for r in resultados_silver.values()) else pd.DataFrame()

In [232]:
display_bokeh_table(
    metricas_silver,
    "Agreement with the hashtag-based silver standard",
    ancho=1650,
    alto=520,
    max_filas=60,
)

In [233]:
display_bokeh_table(
    metricas_silver_balance,
    "Silver standard: natural distribution and 1:1 balanced evaluation",
    ancho=1550,
    alto=340,
    max_filas=40,
)

In [234]:
display_bokeh_table(
    mcnemar_silver,
    "Exact McNemar test: Scenario 1 vs Scenario 2",
    ancho=1050,
    alto=240,
)

In [235]:
if not metricas_silver_evento.empty:
    display_bokeh_table(
        metricas_silver_evento,
        "Silver-reference agreement by event",
        ancho=1550,
        alto=360,
        max_filas=40,
    )

In [236]:
resultado_silver_principal = resultados_silver[DIMENSION_BENCHMARK]

In [237]:
plot_hashtag_confusion_matrix(
    resultado_silver_principal["y_silver"],
    resultado_silver_principal["predicciones"]["Pseudo_E1"],
    f"Silver hashtags vs Pseudo E1 — {DIMENSION_BENCHMARK}D",
)

In [238]:
plot_hashtag_confusion_matrix(
    resultado_silver_principal["y_silver"],
    resultado_silver_principal["predicciones"]["Pseudo_E2"],
    f"Silver hashtags vs Pseudo E2 — {DIMENSION_BENCHMARK}D",
)

In [239]:
AUDITORIAS_HUMANAS_HASHTAG: dict[int, dict[str, pd.DataFrame]] = {}
for dimension, resultado in resultados_silver.items():
    AUDITORIAS_HUMANAS_HASHTAG[int(dimension)] = evaluate_hashtag_human_audit(
        ANOTACIONES_HASHTAG_PREVALIDADAS,
        resultado,
    )

In [240]:
METRICAS_AUDITORIA_HUMANA_HASHTAG = pd.concat(
    [
        auditoria["metricas"]
        for auditoria in AUDITORIAS_HUMANAS_HASHTAG.values()
    ],
    ignore_index=True,
)

In [241]:
MCNEMAR_AUDITORIA_HUMANA_HASHTAG = pd.concat(
    [
        auditoria["mcnemar"]
        for auditoria in AUDITORIAS_HUMANAS_HASHTAG.values()
        if not auditoria["mcnemar"].empty
    ],
    ignore_index=True,
)

In [242]:
DETALLE_AUDITORIA_HUMANA_HASHTAG = (
    AUDITORIAS_HUMANAS_HASHTAG[DIMENSION_BENCHMARK]["detalle"]
)

In [243]:
display_bokeh_table(
    METRICAS_AUDITORIA_HUMANA_HASHTAG,
    "Human validation of the silver standard and pseudo-labeling",
    ancho=1750,
    alto=520,
    max_filas=80,
)

In [244]:
display_bokeh_table(
    MCNEMAR_AUDITORIA_HUMANA_HASHTAG,
    "McNemar with human labels: silver versus pseudo-labels",
    ancho=1200,
    alto=280,
    max_filas=30,
)

In [245]:
ruta_auditoria_actualizada = update_hashtag_model_audit(
    RUTA_ANOTACION_DESACUERDOS_HASHTAG,
    DETALLE_AUDITORIA_HUMANA_HASHTAG,
)

In [246]:
print(f"Second annotation workbook updated with model audit: {ruta_auditoria_actualizada}")

Second annotation workbook updated with model audit: /home/lrecalde/data/cayala/datasets/anotaciones_trabajo/anotacion_desacuerdos_hashtag.xlsx


# 9. Representation benchmarks and computational efficiency

In [247]:
def train_fasttext(corpus: Sequence[Sequence[str]], dimension: int, window: int, min_count: int, epochs: int, seed: int) -> FastText:
    """Train fasttext. Retains the original experimental logic and data-partition rules."""
    corpus_util = [list(doc) for doc in corpus if len(doc) > 0]
    if not corpus_util:
        raise ValueError("The corpus is empty.")
    modelo = FastText(
        vector_size=int(dimension),
        window=int(window),
        min_count=int(min_count),
        sg=1,
        workers=min(8, max(1, (os.cpu_count() or 2) - 1)),
        seed=int(seed),
    )
    modelo.build_vocab(corpus_iterable=corpus_util)
    modelo.train(corpus_iterable=corpus_util, total_examples=len(corpus_util), epochs=int(epochs))
    return modelo

In [248]:
def vectorize_fasttext_documents(tokens_documentos: Sequence[Sequence[str]], modelo: FastText, agregacion: str = "media") -> np.ndarray:
    """Vectorize fasttext documents. Retains the original experimental logic and data-partition rules."""
    if agregacion not in {"media", "suma"}:
        raise ValueError("The aggregation option must be 'media' or 'suma'.")

    dim = int(modelo.vector_size)
    salida = np.zeros((len(tokens_documentos), dim), dtype=np.float32)
    for i, doc in enumerate(tokens_documentos):
        vectores = [modelo.wv.get_vector(str(tok)) for tok in doc if str(tok).strip()]
        if not vectores:
            continue
        matriz = np.asarray(vectores, dtype=np.float32)
        salida[i] = matriz.mean(axis=0) if agregacion == "media" else matriz.sum(axis=0)
    return salida

In [249]:
def extract_transformer_embeddings(
    textos: Sequence[str],
    nombre_modelo: str,
    batch_size: int,
    max_length: int,
) -> tuple[np.ndarray, float]:
    """Extract transformer embeddings. Retains the original experimental logic and data-partition rules."""
    if len(textos) == 0:
        raise ValueError("The text list is empty.")

    dispositivo = DISPOSITIVO_TORCH
    tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)
    modelo = AutoModel.from_pretrained(nombre_modelo).to(dispositivo)
    modelo.eval()

    vectores: list[np.ndarray] = []
    inicio_tiempo = time.perf_counter()

    with torch.no_grad():
        for inicio_lote in range(0, len(textos), int(batch_size)):
            lote = list(textos[inicio_lote:inicio_lote + int(batch_size)])
            entradas = tokenizer(
                lote,
                padding=True,
                truncation=True,
                max_length=int(max_length),
                return_tensors="pt",
            )
            entradas = {
                clave: valor.to(dispositivo)
                for clave, valor in entradas.items()
            }

            salida = modelo(**entradas).last_hidden_state
            mascara = (
                entradas["attention_mask"]
                .unsqueeze(-1)
                .expand(salida.size())
                .float()
            )
            suma = torch.sum(salida * mascara, dim=1)
            divisor = torch.clamp(mascara.sum(dim=1), min=1e-9)
            pooled = suma / divisor
            vectores.append(
                pooled.detach().cpu().numpy().astype(np.float32)
            )

    tiempo = time.perf_counter() - inicio_tiempo

    del modelo
    gc.collect()
    if dispositivo.type == "cuda":
        torch.cuda.empty_cache()

    return np.vstack(vectores), float(tiempo)

In [250]:
def sample_stratified_indices(y: np.ndarray, max_n: int | None, seed: int) -> np.ndarray:
    """Sample stratified indices. Retains the original experimental logic and data-partition rules."""
    y = np.asarray(y)

    if max_n is None or int(max_n) >= len(y):
        return np.arange(len(y), dtype=int)

    if int(max_n) <= 0:
        raise ValueError("max_n must be None or a positive integer.")

    indices = np.arange(len(y), dtype=int)
    seleccion, _ = train_test_split(
        indices,
        train_size=int(max_n),
        random_state=seed,
        stratify=y,
    )
    return np.sort(seleccion.astype(int))

In [251]:
def evaluate_linearsvc_representation(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray, y_test: np.ndarray, nombre: str, tiempo_representacion_s: float, seed: int, referencia: str) -> dict[str, Any]:
    """Evaluate linearsvc representation. Retains the original experimental logic and data-partition rules."""
    inicio = time.perf_counter()
    modelo = LinearSVC(random_state=seed, max_iter=5000, dual="auto")
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    tiempo_clasificador = time.perf_counter() - inicio

    fila: dict[str, Any] = {
        "Representation": nombre,
        "Reference": referencia,
        "N_train": int(len(y_train)),
        "N_test": int(len(y_test)),
        "Features": int(X_train.shape[1]),
        "Tiempo_representacion_s": float(tiempo_representacion_s),
        "Tiempo_clasificador_s": float(tiempo_clasificador),
        "Total_runtime_s": float(tiempo_representacion_s + tiempo_clasificador),
    }
    fila.update(calculate_binary_metrics(y_test, pred))
    return fila

In [252]:
def plot_quality_cost_tradeoff(benchmark: pd.DataFrame, metrica: str = "F1-score") -> None:
    """Plot quality cost tradeoff. Retains the original experimental logic and data-partition rules."""
    datos = benchmark.dropna(subset=["Total_runtime_s", metrica]).copy()
    if datos.empty:
        print("No completed benchmark results available for plotting.")
        return

    source = ColumnDataSource(datos)
    p = figure(
        height=460,
        width=900,
        title=f"Compromiso rendimiento–coste ({metrica})",
        x_axis_label="Tiempo total (s)",
        y_axis_label=metrica,
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )
    p.scatter("Total_runtime_s", metrica, source=source, size=11)
    p.add_tools(HoverTool(tooltips=[
        ("Representation", "@Representacion"),
        ("Tiempo", "@Tiempo_total_s{0.00}s"),
        (metrica, f"@{{{metrica}}}{{0.0000}}"),
        ("Features", "@Features"),
    ]))
    show(p)

In [253]:
def run_silver_representation_benchmark(
    tweets_desarrollo: pd.DataFrame,
    holdout_silver: pd.DataFrame,
    tokens_trends: Sequence[Sequence[str]],
    dimension: int,
    semillas: Sequence[str],
    top_n: int,
    epochs: int,
    fasttext_epochs: int,
    modelos_transformer: Mapping[str, str],
    seed: int,
) -> pd.DataFrame:
    """Run silver representation benchmark. Retains the original experimental logic and data-partition rules."""
    y_test = holdout_silver["etiqueta_silver_hashtag"].astype(int).to_numpy()
    if len(np.unique(y_test)) < 2:
        raise ValueError("The silver reference must contain both classes.")

    tokens_train = extract_dataframe_tokens(tweets_desarrollo, "tokens_sin_stopwords")
    tokens_test = extract_dataframe_tokens(holdout_silver, "tokens_sin_stopwords")

    pipe = build_train_semantic_pipeline(
        tokens_train,
        tokens_trends,
        int(dimension),
        semillas,
        K_CANDIDATOS,
        int(top_n),
        int(epochs),
        seed,
    )

    X_train_pipe = vectorize_documents(
        tokens_train,
        pipe["modelo_word2vec"],
        agregacion="suma",
    )
    y_train, _, _ = label_with_centroids(
        X_train_pipe,
        pipe["centroides_2"],
        pipe["cluster_politico"],
    )

    textos_train = tweets_desarrollo["texto_limpio"].fillna("").astype(str).tolist()
    textos_test = holdout_silver["texto_limpio"].fillna("").astype(str).tolist()
    referencia = "Hashtag silver standard - hashtag-blind external evaluation"

    filas: list[dict[str, Any]] = []

    inicio = time.perf_counter()
    tfidf = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=50000,
        sublinear_tf=True,
    )
    Xtr = tfidf.fit_transform(textos_train)
    Xte = tfidf.transform(textos_test)
    t_repr = time.perf_counter() - inicio
    filas.append(
        evaluate_linearsvc_representation(
            Xtr, y_train, Xte, y_test,
            "TF-IDF (1-2gram) + LinearSVC",
            t_repr, seed, referencia,
        )
    )

    inicio = time.perf_counter()
    w2v = train_word2vec(
        tokens_train,
        int(dimension),
        W2V_WINDOW,
        W2V_MIN_COUNT,
        int(epochs),
        seed,
    )
    Xtr = vectorize_documents(tokens_train, w2v, agregacion="media")
    Xte = vectorize_documents(tokens_test, w2v, agregacion="media")
    t_repr = time.perf_counter() - inicio
    filas.append(
        evaluate_linearsvc_representation(
            Xtr, y_train, Xte, y_test,
            "Word2Vec + LinearSVC",
            t_repr, seed, referencia,
        )
    )

    inicio = time.perf_counter()
    ft = train_fasttext(
        tokens_train,
        int(dimension),
        W2V_WINDOW,
        W2V_MIN_COUNT,
        int(fasttext_epochs),
        seed,
    )
    Xtr = vectorize_fasttext_documents(tokens_train, ft, agregacion="media")
    Xte = vectorize_fasttext_documents(tokens_test, ft, agregacion="media")
    t_repr = time.perf_counter() - inicio
    filas.append(
        evaluate_linearsvc_representation(
            Xtr, y_train, Xte, y_test,
            "FastText + LinearSVC",
            t_repr, seed, referencia,
        )
    )

    for nombre_visible, nombre_modelo in modelos_transformer.items():
        try:
            Xtr, t_train = extract_transformer_embeddings(
                textos_train,
                nombre_modelo,
                TRANSFORMER_BATCH_SIZE,
                TRANSFORMER_MAX_LENGTH,
            )
            Xte, t_test = extract_transformer_embeddings(
                textos_test,
                nombre_modelo,
                TRANSFORMER_BATCH_SIZE,
                TRANSFORMER_MAX_LENGTH,
            )
            filas.append(
                evaluate_linearsvc_representation(
                    Xtr,
                    y_train,
                    Xte,
                    y_test,
                    f"{nombre_visible} congelado + LinearSVC",
                    t_train + t_test,
                    seed,
                    referencia,
                )
            )
            del Xtr, Xte
            gc.collect()
            if DISPOSITIVO_TORCH.type == "cuda":
                torch.cuda.empty_cache()
        except Exception as exc:
            filas.append({
                "Representation": f"{nombre_visible} congelado + LinearSVC",
                "Reference": referencia,
                "N_train": int(len(y_train)),
                "N_test": int(len(y_test)),
                "Features": np.nan,
                "Tiempo_representacion_s": np.nan,
                "Tiempo_clasificador_s": np.nan,
                "Total_runtime_s": np.nan,
                "Accuracy": np.nan,
                "Precision": np.nan,
                "Recall": np.nan,
                "F1-score": np.nan,
                "Sensitivity": np.nan,
                "Specificity": np.nan,
                "Positive precision": np.nan,
                "False-positive rate": np.nan,
                "F2 score": np.nan,
                "F0.5 score": np.nan,
                "TP": np.nan,
                "TN": np.nan,
                "FP": np.nan,
                "FN": np.nan,
                "Error": str(exc),
            })

    return pd.DataFrame(filas)

In [254]:
def fine_tune_transformer_on_silver(
    textos_train: Sequence[str],
    y_train: np.ndarray,
    textos_test: Sequence[str],
    y_test: np.ndarray,
    nombre_visible: str,
    nombre_modelo: str,
    epochs: int,
    batch_size: int,
    max_length: int,
    learning_rate: float,
    seed: int,
) -> pd.DataFrame:
    """Fine tune transformer on silver. Retains the original experimental logic and data-partition rules."""
    if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
        raise ValueError("Training and test sets must contain both classes.")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)
    modelo = AutoModelForSequenceClassification.from_pretrained(
        nombre_modelo,
        num_labels=2,
    ).to(DISPOSITIVO_TORCH)

    optimizador = torch.optim.AdamW(
        modelo.parameters(),
        lr=float(learning_rate),
    )

    y_train = np.asarray(y_train).astype(np.int64)
    y_test = np.asarray(y_test).astype(np.int64)
    rng = np.random.default_rng(seed)

    inicio = time.perf_counter()
    modelo.train()

    for _ in range(int(epochs)):
        orden = rng.permutation(len(textos_train))
        for inicio_lote in range(0, len(orden), int(batch_size)):
            idx = orden[inicio_lote:inicio_lote + int(batch_size)]
            lote_textos = [str(textos_train[i]) for i in idx]
            lote_y = torch.tensor(
                y_train[idx],
                dtype=torch.long,
                device=DISPOSITIVO_TORCH,
            )

            entradas = tokenizer(
                lote_textos,
                padding=True,
                truncation=True,
                max_length=int(max_length),
                return_tensors="pt",
            )
            entradas = {
                k: v.to(DISPOSITIVO_TORCH)
                for k, v in entradas.items()
            }

            optimizador.zero_grad(set_to_none=True)
            salida = modelo(**entradas, labels=lote_y)
            salida.loss.backward()
            optimizador.step()

    modelo.eval()
    predicciones: list[int] = []

    with torch.no_grad():
        for inicio_lote in range(0, len(textos_test), int(batch_size)):
            lote = list(textos_test[inicio_lote:inicio_lote + int(batch_size)])
            entradas = tokenizer(
                lote,
                padding=True,
                truncation=True,
                max_length=int(max_length),
                return_tensors="pt",
            )
            entradas = {
                k: v.to(DISPOSITIVO_TORCH)
                for k, v in entradas.items()
            }
            logits = modelo(**entradas).logits
            predicciones.extend(
                torch.argmax(logits, dim=1).detach().cpu().numpy().astype(int).tolist()
            )

    tiempo = time.perf_counter() - inicio
    metricas = calculate_binary_metrics(
        y_test,
        np.asarray(predicciones, dtype=int),
    )

    fila: dict[str, Any] = {
        "Representation": f"{nombre_visible} fine-tuned",
        "Reference": "Hashtag silver standard - hashtag-blind external evaluation",
        "Device": str(DISPOSITIVO_TORCH),
        "Epochs": int(epochs),
        "Batch_size": int(batch_size),
        "N_train": int(len(y_train)),
        "N_test": int(len(y_test)),
        "Total_runtime_s": float(tiempo),
    }
    fila.update(metricas)

    del modelo
    gc.collect()
    if DISPOSITIVO_TORCH.type == "cuda":
        torch.cuda.empty_cache()

    return pd.DataFrame([fila])

In [255]:
benchmark_silver = run_silver_representation_benchmark(
    tweets_desarrollo=tweets_desarrollo_ciego_hashtags,
    holdout_silver=silver_hashtags_ciego,
    tokens_trends=extract_dataframe_tokens(trends_ciegos_hashtags, "tokens_sin_stopwords"),
    dimension=DIMENSION_BENCHMARK,
    semillas=semillas_politicas_sin_hashtags,
    top_n=EXPANSION_TOP_N_FINAL,
    epochs=W2V_EPOCHS,
    fasttext_epochs=FASTTEXT_EPOCHS,
    modelos_transformer=MODELOS_TRANSFORMER,
    seed=RANDOM_STATE,
)

tokenizer_config.json:   0%|          | 0.00/310 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


... (6 outputs left out)

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

In [256]:
display_bokeh_table(
    benchmark_silver,
    "Representation benchmark against silver labels: performance and cost",
    ancho=1700,
    alto=400,
    max_filas=20,
)

In [257]:
plot_quality_cost_tradeoff(benchmark_silver, "F1-score")

In [258]:
fine_tuning_silver = pd.DataFrame()

In [259]:
if EJECUTAR_FINE_TUNING_TRANSFORMER:
    tokens_dev_ciego = extract_dataframe_tokens(
        tweets_desarrollo_ciego_hashtags,
        "tokens_sin_stopwords",
    )
    tokens_trends_ciegos = extract_dataframe_tokens(
        trends_ciegos_hashtags,
        "tokens_sin_stopwords",
    )

    pipe_ft = build_train_semantic_pipeline(
        tokens_dev_ciego,
        tokens_trends_ciegos,
        DIMENSION_BENCHMARK,
        semillas_politicas_sin_hashtags,
        K_CANDIDATOS,
        EXPANSION_TOP_N_FINAL,
        W2V_EPOCHS,
        RANDOM_STATE,
    )

    X_dev_ft = vectorize_documents(
        tokens_dev_ciego,
        pipe_ft["modelo_word2vec"],
        agregacion="suma",
    )
    y_dev_ft, _, _ = label_with_centroids(
        X_dev_ft,
        pipe_ft["centroides_2"],
        pipe_ft["cluster_politico"],
    )

    textos_dev = tweets_desarrollo_ciego_hashtags["texto_limpio"].fillna("").astype(str).tolist()
    textos_silver = silver_hashtags_ciego["texto_limpio"].fillna("").astype(str).tolist()
    y_silver = silver_hashtags_ciego["etiqueta_silver_hashtag"].astype(int).to_numpy()

    filas_ft: list[pd.DataFrame] = []
    for nombre_visible, nombre_modelo in MODELOS_TRANSFORMER.items():
        print(f"Fine-tuning: {nombre_visible} on {DISPOSITIVO_TORCH}")
        filas_ft.append(
            fine_tune_transformer_on_silver(
                textos_train=textos_dev,
                y_train=y_dev_ft,
                textos_test=textos_silver,
                y_test=y_silver,
                nombre_visible=nombre_visible,
                nombre_modelo=nombre_modelo,
                epochs=TRANSFORMER_FINE_TUNE_EPOCHS,
                batch_size=TRANSFORMER_BATCH_SIZE,
                max_length=TRANSFORMER_MAX_LENGTH,
                learning_rate=TRANSFORMER_FINE_TUNE_LR,
                seed=RANDOM_STATE,
            )
        )

    fine_tuning_silver = pd.concat(filas_ft, ignore_index=True)
    display_bokeh_table(
        fine_tuning_silver,
        "Transformer fine-tuning against the silver standard",
        ancho=1600,
        alto=300,
        max_filas=10,
    )
else:
    print(
        "Legacy Transformer fine-tuning was skipped. "
        "The unified five-model fine-tuning benchmark runs in Section 11."
    )

Legacy Transformer fine-tuning was skipped. The unified five-model fine-tuning benchmark runs in Section 11.


# 10. Independent human evaluation

In [260]:
def evaluate_human_validation(tweets_desarrollo: pd.DataFrame, anotaciones: pd.DataFrame, tokens_trends: Sequence[Sequence[str]], dimension: int, semillas: Sequence[str], top_n: int, epochs: int, seed: int) -> pd.DataFrame:
    """Evaluate human validation. Retains the original experimental logic and data-partition rules."""
    validas = anotaciones.dropna(subset=["etiqueta_consenso"]).copy()
    if len(validas) < 20:
        raise ValueError("At least 20 adjudicated annotations are needed for human validation.")

    y_humano = validas["etiqueta_consenso"].astype(int).to_numpy()
    if len(np.unique(y_humano)) < 2:
        raise ValueError("Consensus annotations must include both binary classes.")

    tokens_train = extract_dataframe_tokens(tweets_desarrollo, "tokens_sin_stopwords")
    tokens_manual = [parse_tokens(x) for x in validas["texto_limpio"].fillna("").astype(str).tolist()]

    pipe = build_train_semantic_pipeline(
        tokens_train,
        tokens_trends,
        dimension,
        semillas,
        K_CANDIDATOS,
        top_n,
        epochs,
        seed,
    )
    X_train = vectorize_documents(tokens_train, pipe["modelo_word2vec"], agregacion="suma")
    X_manual = vectorize_documents(tokens_manual, pipe["modelo_word2vec"], agregacion="suma")

    y1_train, _, _ = label_with_centroids(X_train, pipe["centroides_1"], pipe["cluster_politico"])
    y2_train, _, _ = label_with_centroids(X_train, pipe["centroides_2"], pipe["cluster_politico"])
    y1_manual, _, _ = label_with_centroids(X_manual, pipe["centroides_1"], pipe["cluster_politico"])
    y2_manual, _, _ = label_with_centroids(X_manual, pipe["centroides_2"], pipe["cluster_politico"])

    filas: list[dict[str, Any]] = []
    for escenario, y_train, y_pseudo in [
        ("Scenario 1", y1_train, y1_manual),
        ("Scenario 2", y2_train, y2_manual),
    ]:
        fila_pseudo: dict[str, Any] = {
            "Dimension": int(dimension),
            "Scenario": escenario,
            "Modelo": "Direct pseudo-labeling",
            "Reference": "Human ground truth",
        }
        fila_pseudo.update(calculate_binary_metrics(y_humano, y_pseudo))
        filas.append(fila_pseudo)

        for nombre, modelo in create_classifiers(KNN_VECINOS, seed).items():
            modelo.fit(X_train, y_train)
            pred = modelo.predict(X_manual)
            fila: dict[str, Any] = {
                "Dimension": int(dimension),
                "Scenario": escenario,
                "Modelo": nombre,
                "Reference": "Human ground truth",
            }
            fila.update(calculate_binary_metrics(y_humano, pred))
            filas.append(fila)

    return pd.DataFrame(filas)

In [261]:
def run_human_representation_benchmark(tweets_desarrollo: pd.DataFrame, anotaciones: pd.DataFrame, max_train: int | None, max_test: int | None, dimension: int, semillas: Sequence[str], tokens_trends: Sequence[Sequence[str]], top_n: int, epochs: int, fasttext_epochs: int, modelos_transformer: Mapping[str, str], ejecutar_transformers: bool, seed: int) -> pd.DataFrame:
    """Run human representation benchmark. Retains the original experimental logic and data-partition rules."""
    validas = anotaciones.dropna(subset=["etiqueta_consenso"]).copy()
    if len(validas) < 20:
        raise ValueError("At least 20 human consensus labels are required.")
    y_test_full = validas["etiqueta_consenso"].astype(int).to_numpy()
    if len(np.unique(y_test_full)) < 2:
        raise ValueError("Human ground truth must contain classes 0 and 1.")

    tokens_dev = extract_dataframe_tokens(tweets_desarrollo, "tokens_sin_stopwords")
    pipe = build_train_semantic_pipeline(
        tokens_dev,
        tokens_trends,
        int(dimension),
        semillas,
        K_CANDIDATOS,
        int(top_n),
        int(epochs),
        seed,
    )
    X_dev_w2v = vectorize_documents(tokens_dev, pipe["modelo_word2vec"], agregacion="suma")
    y_train_full, _, _ = label_with_centroids(
        X_dev_w2v,
        pipe["centroides_2"],
        pipe["cluster_politico"],
    )

    idx_train = sample_stratified_indices(y_train_full, max_train, seed)
    idx_test = sample_stratified_indices(y_test_full, max_test, seed)

    train_df = tweets_desarrollo.iloc[idx_train].reset_index(drop=True)
    test_df = validas.iloc[idx_test].reset_index(drop=True)
    y_train = y_train_full[idx_train]
    y_test = y_test_full[idx_test]

    textos_train = train_df["texto_limpio"].fillna("").astype(str).tolist()
    textos_test = test_df["texto_limpio"].fillna("").astype(str).tolist()
    tokens_train = extract_dataframe_tokens(train_df, "tokens_sin_stopwords")
    tokens_test = [parse_tokens(x) for x in test_df["texto_limpio"].fillna("").astype(str).tolist()]
    referencia = "Independent human ground truth"

    filas: list[dict[str, Any]] = []

    inicio = time.perf_counter()
    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)
    Xtr_tfidf = tfidf.fit_transform(textos_train)
    Xte_tfidf = tfidf.transform(textos_test)
    tiempo = time.perf_counter() - inicio
    filas.append(evaluate_linearsvc_representation(
        Xtr_tfidf, y_train, Xte_tfidf, y_test, "TF-IDF (1-2gram) + LinearSVC", tiempo, seed, referencia
    ))

    inicio = time.perf_counter()
    w2v = train_word2vec(tokens_train, int(dimension), W2V_WINDOW, W2V_MIN_COUNT, int(epochs), seed)
    Xtr_w2v = vectorize_documents(tokens_train, w2v, agregacion="media")
    Xte_w2v = vectorize_documents(tokens_test, w2v, agregacion="media")
    tiempo = time.perf_counter() - inicio
    filas.append(evaluate_linearsvc_representation(
        Xtr_w2v, y_train, Xte_w2v, y_test, "Word2Vec + LinearSVC", tiempo, seed, referencia
    ))

    inicio = time.perf_counter()
    ft = train_fasttext(tokens_train, int(dimension), W2V_WINDOW, W2V_MIN_COUNT, int(fasttext_epochs), seed)
    Xtr_ft = vectorize_fasttext_documents(tokens_train, ft, agregacion="media")
    Xte_ft = vectorize_fasttext_documents(tokens_test, ft, agregacion="media")
    tiempo = time.perf_counter() - inicio
    filas.append(evaluate_linearsvc_representation(
        Xtr_ft, y_train, Xte_ft, y_test, "FastText + LinearSVC", tiempo, seed, referencia
    ))

    if ejecutar_transformers:
        for nombre_visible, nombre_modelo in modelos_transformer.items():
            try:
                Xtr, t_train = extract_transformer_embeddings(
                    textos_train, nombre_modelo, TRANSFORMER_BATCH_SIZE, TRANSFORMER_MAX_LENGTH
                )
                Xte, t_test = extract_transformer_embeddings(
                    textos_test, nombre_modelo, TRANSFORMER_BATCH_SIZE, TRANSFORMER_MAX_LENGTH
                )
                filas.append(evaluate_linearsvc_representation(
                    Xtr, y_train, Xte, y_test, nombre_visible, t_train + t_test, seed, referencia
                ))
            except Exception as exc:
                filas.append({
                    "Representation": nombre_visible,
                    "Reference": referencia,
                    "N_train": int(len(y_train)),
                    "N_test": int(len(y_test)),
                    "Features": np.nan,
                    "Tiempo_representacion_s": np.nan,
                    "Tiempo_clasificador_s": np.nan,
                    "Total_runtime_s": np.nan,
                    "Accuracy": np.nan,
                    "Precision": np.nan,
                    "Recall": np.nan,
                    "F1-score": np.nan,
                    "Sensitivity": np.nan,
                    "Specificity": np.nan,
                    "Positive precision": np.nan,
                    "False-positive rate": np.nan,
                    "F2 score": np.nan,
                    "F0.5 score": np.nan,
                    "TP": np.nan,
                    "TN": np.nan,
                    "FP": np.nan,
                    "FN": np.nan,
                    "Error": str(exc),
                })

    return pd.DataFrame(filas)

In [262]:
def plot_human_confusion_matrices(
    validacion_manual: pd.DataFrame
) -> None:
    """Plot human confusion matrices. Retains the original experimental logic and data-partition rules."""
    requeridas = {
        "Scenario",
        "Modelo",
        "TP",
        "TN",
        "FP",
        "FN",
    }

    faltantes = requeridas.difference(
        validacion_manual.columns
    )

    if faltantes:
        raise ValueError(
            f"Faltan columnas requeridas: {sorted(faltantes)}. "
            f"Columnas disponibles: "
            f"{validacion_manual.columns.tolist()}"
        )

    datos = validacion_manual[
        validacion_manual["Modelo"]
        == "Direct pseudo-labeling"
    ].copy()

    matrices = {}

    for escenario in [
        "Scenario 1",
        "Scenario 2",
    ]:

        fila = datos[
            datos["Scenario"] == escenario
        ]

        if fila.empty:
            raise ValueError(
                f"No se encontró el pseudoetiquetado directo "
                f"para {escenario}."
            )

        fila = fila.iloc[0]

        matrices[escenario] = {
            "TN": int(fila["TN"]),
            "FP": int(fila["FP"]),
            "FN": int(fila["FN"]),
            "TP": int(fila["TP"]),
        }

    vmax = max(
        valor
        for matriz in matrices.values()
        for valor in matriz.values()
    )

    color_mapper = LinearColorMapper(
        palette=Blues9,
        low=0,
        high=vmax,
    )

    def crear_matriz_bokeh(
        escenario: str,
        valores: dict,
    ):

        source = ColumnDataSource(
            data={
                "x": [
                    "Non-political",
                    "Political",
                    "Non-political",
                    "Political",
                ],
                "y": [
                    "Non-political",
                    "Non-political",
                    "Political",
                    "Political",
                ],
                "valor": [
                    valores["TN"],
                    valores["FP"],
                    valores["FN"],
                    valores["TP"],
                ],
                "tipo": [
                    "True Negative",
                    "False Positive",
                    "False Negative",
                    "True Positive",
                ],
                "abreviatura": [
                    "TN",
                    "FP",
                    "FN",
                    "TP",
                ],
            }
        )

        titulo = escenario.replace(
            "Scenario",
            "Scenario"
        )

        pan_tool = PanTool()
        wheel_zoom_tool = WheelZoomTool()
        box_zoom_tool = BoxZoomTool()
        reset_tool = ResetTool()
        save_tool = SaveTool()
        help_tool = HelpTool()

        p = figure(
            width=500,
            height=430,
            title=titulo,
            x_range=[
                "Non-political",
                "Political",
            ],
            y_range=[
                "Political",
                "Non-political",
            ],
            x_axis_label="Predicted label",
            y_axis_label="Human ground truth",
            tools=[
                pan_tool,
                wheel_zoom_tool,
                box_zoom_tool,
                reset_tool,
                save_tool,
                help_tool,
            ],
            toolbar_location="right",
        )

        p.toolbar.active_scroll = wheel_zoom_tool

        renderer = p.rect(
            x="x",
            y="y",
            width=1,
            height=1,
            source=source,
            fill_color={
                "field": "valor",
                "transform": color_mapper,
            },
            line_color="white",
            line_width=2,
        )

        etiquetas = LabelSet(
            x="x",
            y="y",
            text="valor",
            source=source,
            text_align="center",
            text_baseline="middle",
            text_font_size="15px",
            text_font_style="bold",
        )

        p.add_layout(etiquetas)

        hover = HoverTool(
            renderers=[renderer],
            tooltips=[
                ("Type", "@tipo"),
                ("Code", "@abreviatura"),
                ("Human label", "@y"),
                ("Predicted label", "@x"),
                ("Count", "@valor"),
            ],
        )

        p.add_tools(hover)

        p.xgrid.grid_line_color = None
        p.ygrid.grid_line_color = None

        return p

    p1 = crear_matriz_bokeh(
        "Scenario 1",
        matrices["Scenario 1"],
    )

    p2 = crear_matriz_bokeh(
        "Scenario 2",
        matrices["Scenario 2"],
    )

    color_bar = ColorBar(
        color_mapper=color_mapper,
        ticker=BasicTicker(),
        label_standoff=8,
        title="Count",
    )

    p2.add_layout(
        color_bar,
        "right"
    )

    show(
        row(
            p1,
            p2,
        )
    )

In [263]:
anotaciones_manual = load_human_annotations(RUTA_ANOTACION_MANUAL)
acuerdo_anotadores = calculate_annotator_agreement(anotaciones_manual)

In [264]:
display_bokeh_table(
    acuerdo_anotadores,
    "Inter-annotator agreement on human reference",
    alto=180,
)

In [265]:
validacion_manual = evaluate_human_validation(
    tweets_desarrollo=tweets_desarrollo,
    anotaciones=anotaciones_manual,
    tokens_trends=tokens_trends,
    dimension=DIMENSION_BENCHMARK,
    semillas=semillas_politicas,
    top_n=EXPANSION_TOP_N_FINAL,
    epochs=W2V_EPOCHS,
    seed=RANDOM_STATE,
)

In [266]:
benchmark_manual = run_human_representation_benchmark(
    tweets_desarrollo=tweets_desarrollo,
    anotaciones=anotaciones_manual,
    max_train=None,
    max_test=None,
    dimension=DIMENSION_BENCHMARK,
    semillas=semillas_politicas,
    tokens_trends=tokens_trends,
    top_n=EXPANSION_TOP_N_FINAL,
    epochs=W2V_EPOCHS,
    fasttext_epochs=FASTTEXT_EPOCHS,
    modelos_transformer=MODELOS_TRANSFORMER,
    ejecutar_transformers=True,
    seed=RANDOM_STATE,
)

Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-uncased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-uncased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [267]:
display_bokeh_table(
    validacion_manual,
    "Validation against independent human ground truth",
    ancho=1550,
    alto=430,
    max_filas=30,
)

In [268]:
plot_human_confusion_matrices(
    validacion_manual
)

In [269]:
display_bokeh_table(
    benchmark_manual,
    "Representation benchmark against independent human ground truth",
    ancho=1700,
    alto=400,
    max_filas=20,
)

In [270]:
plot_quality_cost_tradeoff(benchmark_manual, "F1-score")

In [271]:
def plot_representation_cost_tradeoff(
    benchmark_manual: pd.DataFrame
) -> None:
    """Plot representation cost tradeoff. Retains the original experimental logic and data-partition rules."""
    columnas_requeridas = {
        "Representation",
        "Total_runtime_s",
        "Sensitivity",
        "Specificity",
    }

    faltantes = columnas_requeridas.difference(
        benchmark_manual.columns
    )

    if faltantes:
        raise ValueError(
            f"Faltan columnas requeridas: {sorted(faltantes)}. "
            f"Columnas disponibles: "
            f"{benchmark_manual.columns.tolist()}"
        )

    datos = benchmark_manual.copy()
    datos["Balanced_Accuracy"] = (
        pd.to_numeric(
            datos["Sensitivity"],
            errors="coerce",
        )
        +
        pd.to_numeric(
            datos["Specificity"],
            errors="coerce",
        )
    ) / 2.0

    datos["Runtime_seconds"] = pd.to_numeric(
        datos["Total_runtime_s"],
        errors="coerce",
    )

    def normalizar_representacion(valor):
        texto = str(valor).lower()

        if "tf-idf" in texto or "tfidf" in texto:
            return "TF-IDF"

        if "word2vec" in texto:
            return "Word2Vec"

        if "fasttext" in texto:
            return "FastText"

        if "beto" in texto:
            return "BETO"

        if (
            "xlm-r" in texto
            or "xlm_r" in texto
            or "xlm" in texto
            or "roberta" in texto
        ):
            return "XLM-RoBERTa"

        return None

    datos["Representation_plot"] = (
        datos["Representation"]
        .apply(normalizar_representacion)
    )

    datos = datos[
        datos["Representation_plot"].notna()
    ].copy()

    datos = datos.dropna(
        subset=[
            "Runtime_seconds",
            "Balanced_Accuracy",
        ]
    ).copy()

    datos = datos[
        datos["Runtime_seconds"] > 0
    ].copy()

    representaciones_esperadas = {
        "TF-IDF",
        "Word2Vec",
        "FastText",
        "BETO",
        "XLM-RoBERTa",
    }

    representaciones_encontradas = set(
        datos["Representation_plot"]
        .unique()
        .tolist()
    )

    faltantes_modelos = (
        representaciones_esperadas
        - representaciones_encontradas
    )

    if faltantes_modelos:
        raise ValueError(
            "Missing representations for the figure: "
            f"{sorted(faltantes_modelos)}. "
            f"Encontradas: "
            f"{sorted(representaciones_encontradas)}"
        )

    datos = (
        datos
        .drop_duplicates(
            subset="Representation_plot",
            keep="first",
        )
        .copy()
    )

    orden_representaciones = [
        "TF-IDF",
        "Word2Vec",
        "FastText",
        "BETO",
        "XLM-RoBERTa",
    ]

    orden = {
        nombre: i
        for i, nombre
        in enumerate(orden_representaciones)
    }

    datos["_orden"] = (
        datos["Representation_plot"]
        .map(orden)
    )

    datos = (
        datos
        .sort_values("_orden")
        .drop(columns="_orden")
        .reset_index(drop=True)
    )

    tabla_figura = datos[
        [
            "Representation_plot",
            "Runtime_seconds",
            "Sensitivity",
            "Specificity",
            "Balanced_Accuracy",
        ]
    ].copy()

    print(
        "Data used for the figure "
        "performance-cost:"
    )

    display(tabla_figura)

    source = ColumnDataSource(
        data={
            "representation":
                datos["Representation_plot"].tolist(),

            "runtime":
                datos["Runtime_seconds"].tolist(),

            "balanced":
                datos["Balanced_Accuracy"].tolist(),

            "sensitivity":
                datos["Sensitivity"].tolist(),

            "specificity":
                datos["Specificity"].tolist(),
        }
    )

    pan_tool = PanTool()
    wheel_zoom_tool = WheelZoomTool()
    box_zoom_tool = BoxZoomTool()
    reset_tool = ResetTool()
    save_tool = SaveTool()
    help_tool = HelpTool()

    p = figure(
        width=850,
        height=520,
        title=(
            "Balanced Accuracy vs Computational Time"
        ),
        x_axis_type="log",
        x_axis_label=(
            "Total execution time (s, log scale)"
        ),
        y_axis_label="Balanced Accuracy",
        tools=[
            pan_tool,
            wheel_zoom_tool,
            box_zoom_tool,
            reset_tool,
            save_tool,
            help_tool,
        ],
        toolbar_location="right",
    )
    p.toolbar.active_scroll = wheel_zoom_tool

    renderer = p.scatter(
        x="runtime",
        y="balanced",
        source=source,
        size=14,
        alpha=0.85,
    )

    labels = LabelSet(
        x="runtime",
        y="balanced",
        text="representation",
        source=source,
        x_offset=8,
        y_offset=6,
        text_font_size="10pt",
    )

    p.add_layout(labels)

    hover = HoverTool(
        renderers=[renderer],
        tooltips=[
            (
                "Representation",
                "@representation",
            ),
            (
                "Total execution time",
                "@runtime{0.00} s",
            ),
            (
                "Balanced Accuracy",
                "@balanced{0.0000}",
            ),
            (
                "Sensitivity",
                "@sensitivity{0.0000}",
            ),
            (
                "Specificity",
                "@specificity{0.0000}",
            ),
        ],
    )

    p.add_tools(hover)
    p.grid.grid_line_alpha = 0.25
    p.outline_line_alpha = 0.5
    show(p)

In [272]:
plot_representation_cost_tradeoff(
    benchmark_manual
)

Data used for the figure performance-cost:


,Representation_plot,Runtime_seconds,Sensitivity,Specificity,Balanced_Accuracy
0,TF-IDF,11.205497,0.993007,0.296178,0.644593
1,Word2Vec,66.146355,0.996503,0.312102,0.654303
2,FastText,34.211735,0.996503,0.302548,0.649526
3,BETO,254.072581,0.996503,0.181529,0.589016
4,XLM-RoBERTa,254.802369,1.000000,0.200637,0.600318


## 11. Modern-model evaluation: controlled pseudo-label training and independent human testing

The original centroid-based Scenario 1/2 decisions and a Word2Vec+LinearSVC classifier are repeated on **this exact same split**, ensuring the comparison is not confounded by differing training datasets. All trainable models, including the five fine-tuned transformers, receive identical weak labels generated from the training partition. The 600 human annotations are never used for training, threshold choice, or checkpoint selection.


In [273]:
# A self-contained modern benchmark inside the final notebook.

In [274]:
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, f1_score, balanced_accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

MODERN_MODEL_REGISTRY = {
    "RoBERTuito": "pysentimiento/robertuito-base-uncased",
    "BETO": "dccuchile/bert-base-spanish-wwm-uncased",
    "XLM-RoBERTa": "xlm-roberta-base",
    "BERTweet": "vinai/bertweet-base",
    "PoliBERTweet": "kornosk/polibertweet-political-twitter-roberta-mlm",
}
MODERN_BENCHMARK_DIR = RESULTS_DIR / "benchmarks" / "modern_models"
MODERN_MODEL_DIR = MODELS_DIR / "benchmarks" / "modern_models"
MODERN_CHECKPOINT_DIR = CHECKPOINTS_DIR / "benchmarks" / "modern_models"
for _folder in (MODERN_BENCHMARK_DIR, MODERN_MODEL_DIR, MODERN_CHECKPOINT_DIR):
    _folder.mkdir(parents=True, exist_ok=True)

@dataclass
class ModernBenchmarkConfig:
    """A comparable evaluation protocol for modern pretrained models and classical baselines."""
    model_names: tuple = ("RoBERTuito", "BETO", "XLM-RoBERTa", "BERTweet", "PoliBERTweet")
    maximum_development_rows: int = 24000
    validation_fraction: float = 0.15
    epochs: int = 2
    max_length: int = 128
    train_batch_size: int = 16
    eval_batch_size: int = 32
    grad_accumulation_steps: int = 2
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_fraction: float = 0.06
    seed: int = 5
    balanced_loss_variant: bool = True
    benchmark_frozen_features: bool = True
    benchmark_fine_tuning: bool = True
    benchmark_classical: bool = True
    enable_rbf_baseline: bool = True
    enable_xgboost_baseline: bool = True
    enable_validation_threshold: bool = True
    save_fine_tuned_models: bool = True
    evaluate_cpu_inference: bool = True
    cuda_memory_fraction: float = 0.70
    gpu_parallel: bool = True
    num_cpu_threads: int = CPU_WORKERS
    # The independent 600-record human reference is NEVER used to train or tune a model.

MODERN_CFG = ModernBenchmarkConfig()
if not torch.cuda.is_available() and MODERN_CFG.benchmark_fine_tuning:
    raise RuntimeError("Modern transformer fine-tuning requires a CUDA GPU for this shared-server protocol.")
print("Modern model checkpoints:", MODERN_CFG.model_names)
print("Visible CUDA devices:", torch.cuda.device_count() if torch.cuda.is_available() else 0)
print("Results folder:", MODERN_BENCHMARK_DIR)

Modern model checkpoints: ('RoBERTuito', 'BETO', 'XLM-RoBERTa', 'BERTweet', 'PoliBERTweet')
Visible CUDA devices: 2
Results folder: /home/lrecalde/data/cayala/results/benchmarks/modern_models


In [275]:
def modern_binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Report class-sensitive metrics on a fixed, independent binary test set."""
    y_true, y_pred = np.asarray(y_true, dtype=int), np.asarray(y_pred, dtype=int)
    if len(y_true) != len(y_pred) or not set(np.unique(y_true)).issubset({0, 1}):
        raise ValueError("Binary metrics require aligned 0/1 labels and predictions.")
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / max(1, tn + fp)
    return dict(accuracy=float(np.mean(y_true == y_pred)),
                balanced_accuracy=float(balanced_accuracy_score(y_true, y_pred)),
                precision=float(precision_score(y_true, y_pred, zero_division=0)),
                recall=float(recall_score(y_true, y_pred, zero_division=0)),
                f1_political=float(f1_score(y_true, y_pred, zero_division=0)),
                f1_macro=float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
                specificity=float(specificity), false_positive_rate=float(1-specificity),
                tp=int(tp), tn=int(tn), fp=int(fp), fn=int(fn), test_size=len(y_true))

def choose_threshold_from_pseudo_validation(scores: np.ndarray, weak_labels: np.ndarray,
                                            default: float = 0.5) -> tuple[float,float]:
    """Select a threshold using only weak labels in the development validation split."""
    scores = np.asarray(scores, dtype=float)
    weak_labels = np.asarray(weak_labels, dtype=int)
    if len(scores) != len(weak_labels) or not np.isfinite(scores).all():
        raise ValueError("Validation scores and labels must be finite and aligned.")
    candidates = np.unique(np.r_[default, np.quantile(scores, np.linspace(0.01, 0.99, 99))])
    best_score, best_threshold = float("-inf"), default
    for threshold in candidates:
        score = float(balanced_accuracy_score(weak_labels, scores >= threshold))
        if (score, threshold) > (best_score, best_threshold):
            best_score, best_threshold = score, float(threshold)
    return best_threshold, best_score

def _benchmark_row(model_name: str, family: str, y_true: np.ndarray, predictions: np.ndarray,
                   training_seconds: float, inference_seconds: float,
                   n_train: int, device: str, **additional) -> dict:
    """Attach runtime, memory and input-size metadata to independent human-test metrics."""
    result = dict(model=model_name, family=family, training_seconds=float(training_seconds),
                  inference_seconds=float(inference_seconds), total_seconds=float(training_seconds+inference_seconds),
                  throughput_tweets_per_s=len(y_true)/max(1e-9,inference_seconds),
                  cpu_memory_rss_mb=psutil.Process().memory_info().rss/(1024**2),
                  n_train=int(n_train), device=device)
    result.update(modern_binary_metrics(y_true, predictions))
    result.update(additional)
    return result

def benchmark_prepared_split(development_df: pd.DataFrame, human_df: pd.DataFrame,
                             tokenized_trends: Sequence[Sequence[str]], seeds: Sequence[str],
                             cfg: ModernBenchmarkConfig):
    """Build weak labels after the split, using the training split only for semantic fitting."""
    human = human_df.dropna(subset=["etiqueta_consenso"]).copy().reset_index(drop=True)
    if len(human) != 600 or set(human["etiqueta_consenso"].astype(int).unique()) != {0,1}:
        raise ValueError("The complete 600-tweet adjudicated human reference is required.")
    development = development_df.reset_index(drop=True).copy()
    if not {'id_registro','texto_limpio','evento'}.issubset(development.columns):
        raise KeyError("Development records need stable IDs, cleaned text, and event labels.")
    if not {'id_registro','texto_limpio'}.issubset(human.columns):
        raise KeyError("Human records need stable IDs and cleaned text.")
    if set(development.id_registro.astype(str)) & set(human.id_registro.astype(str)):
        raise RuntimeError("Data leakage: human and development record identifiers overlap.")
    # Exact duplicate text across different events is excluded from this new benchmark only.
    human_texts = set(human.texto_limpio.fillna('').astype(str))
    overlap = development.texto_limpio.fillna('').astype(str).isin(human_texts)
    n_removed = int(overlap.sum())
    development = development.loc[~overlap].reset_index(drop=True)
    indices = np.arange(len(development))
    if cfg.maximum_development_rows and len(indices) > cfg.maximum_development_rows:
        indices, _ = train_test_split(indices, train_size=cfg.maximum_development_rows,
                                    stratify=development.evento.astype(str),random_state=cfg.seed)
    sample = development.iloc[np.sort(indices)].reset_index(drop=True)
    train_idx, valid_idx = train_test_split(np.arange(len(sample)), test_size=cfg.validation_fraction,
                                           stratify=sample.evento.astype(str),random_state=cfg.seed)
    train = sample.iloc[np.sort(train_idx)].reset_index(drop=True)
    valid = sample.iloc[np.sort(valid_idx)].reset_index(drop=True)
    tokens_train = extract_dataframe_tokens(train, "tokens_sin_stopwords")
    tokens_val = extract_dataframe_tokens(valid, "tokens_sin_stopwords")
    tokens_human = [parse_tokens(t) for t in human.texto_limpio.fillna('').astype(str)]
    start = time.perf_counter()
    semantic = build_train_semantic_pipeline(
        tokens_train, tokenized_trends, DIMENSION_BENCHMARK, seeds,
        K_CANDIDATOS, EXPANSION_TOP_N_FINAL, W2V_EPOCHS, cfg.seed)
    for token_collection in (tokens_train, tokens_val, tokens_human):
        if not isinstance(token_collection,list):
            raise TypeError("Token collections must be materialized sequences.")
    pseudo_labels=[]
    for collection in (tokens_train, tokens_val):
        vectors = vectorize_documents(collection, semantic["modelo_word2vec"], agregacion="suma")
        labels, _, _ = label_with_centroids(vectors, semantic["centroides_2"], semantic["cluster_politico"])
        pseudo_labels.append(np.asarray(labels,dtype=int))
    pseudo_tr, pseudo_val = pseudo_labels
    semantic_fit_s = time.perf_counter()-start
    if set(np.unique(pseudo_tr)) != {0, 1} or set(np.unique(pseudo_val)) != {0, 1}:
        raise RuntimeError("Weak labels must contain both classes in training and validation.")
    text_key_train = "texto_original" if "texto_original" in train else "texto_limpio"
    text_key_human = "texto_original_para_anotar" if "texto_original_para_anotar" in human else "texto_limpio"
    train_text = train[text_key_train].fillna('').astype(str).tolist()
    val_text = valid[text_key_train].fillna('').astype(str).tolist()
    human_text = human[text_key_human].fillna('').astype(str).tolist()
    test_labels = human.etiqueta_consenso.astype(int).to_numpy()
    if not len(train_text) or not len(val_text):
        raise ValueError("Empty train or validation split.")
    if set(train.id_registro.astype(str)) & set(valid.id_registro.astype(str)):
        raise RuntimeError("Development train and validation splits overlap.")
    if set(sample.id_registro.astype(str)) & set(human.id_registro.astype(str)):
        raise RuntimeError("Human test records overlap with benchmark development.")
    print(f"Train: {len(train):,}, validation: {len(valid):,}, human test: {len(human):,}")
    print("Weak label distribution train / validation:",np.bincount(pseudo_tr,minlength=2),np.bincount(pseudo_val,minlength=2))
    print("Excluded development rows with duplicate human texts:",n_removed)
    protocol = dict(run_date_utc=datetime.now(timezone.utc).isoformat(), configuration=asdict(cfg),
                    source_file="codigoBIAS_rs_CYG(7).ipynb", benchmark_train_size=len(train),
                    benchmark_validation_size=len(valid), human_test_size=len(human),
                    duplicate_texts_removed=n_removed, semantic_fit_seconds=semantic_fit_s,
                    training_text_field=text_key_train,human_text_field=text_key_human,
                    training_labels="Word2Vec expanded Scenario 2, trained only on train split",
                    checkpoint_selection="weak development validation balanced accuracy",
                    test_use="600 adjudicated human tweets, reporting only",
                    train_weak_distribution=np.bincount(pseudo_tr,minlength=2).tolist(),
                    val_weak_distribution=np.bincount(pseudo_val,minlength=2).tolist(),
                    human_distribution=np.bincount(test_labels,minlength=2).tolist(),
                    visible_gpu_indices=os.environ.get("CUDA_VISIBLE_DEVICES",""),
                    visible_gpu_names=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
                    if torch.cuda.is_available() else [],
                    python=sys.version,torch=str(torch.__version__),transformers=importlib_metadata.version('transformers'))
    (MODERN_BENCHMARK_DIR/"protocol.json").write_text(json.dumps(protocol,indent=2,ensure_ascii=False,default=str),encoding="utf-8")
    for name, df, labels in (("train",train,pseudo_tr),("validation",valid,pseudo_val)):
        pd.DataFrame({"record_id":df.id_registro.astype(str),"weak_label":labels,
                      "split":name}).to_csv(MODERN_BENCHMARK_DIR/f"{name}_record_ids.csv",index=False)
    return dict(train=train,valid=valid,human=human,
                train_text=train_text,val_text=val_text,human_text=human_text,
                y_train=pseudo_tr,y_val=pseudo_val,y_human=test_labels,
                tokens_train=tokens_train,tokens_val=tokens_val,tokens_human=tokens_human,
                protocol=protocol,semantic=semantic)

In [276]:
modern_splits = benchmark_prepared_split(
    tweets_desarrollo, anotaciones_manual, tokens_trends, semillas_politicas, MODERN_CFG)
modern_result_rows = []
modern_prediction_rows = []
modern_run_errors = []

def record_benchmark_predictions(model_name, predicted, scores=None):
    """Save IDs and predictions, never raw annotated tweet text."""
    frame = pd.DataFrame({"record_id":modern_splits["human"].id_registro.astype(str),
                          "model":model_name,"human_label":modern_splits["y_human"],
                          "prediction":np.asarray(predicted,dtype=int)})
    if scores is not None: frame["political_score"] = np.asarray(scores,dtype=float)
    modern_prediction_rows.append(frame)

def save_benchmark_progress():
    """Durably save completed models before the next GPU-intensive fit begins."""
    pd.DataFrame(modern_result_rows).to_csv(MODERN_BENCHMARK_DIR/"benchmark_results.csv",index=False)
    if modern_prediction_rows:
        pd.concat(modern_prediction_rows,ignore_index=True).to_csv(
            MODERN_BENCHMARK_DIR/"human_predictions.csv",index=False)
    if modern_run_errors:
        (MODERN_BENCHMARK_DIR/"errors.json").write_text(json.dumps(modern_run_errors,indent=2,ensure_ascii=False),encoding="utf-8")

Train: 20,400, validation: 3,600, human test: 600
Weak label distribution train / validation: [ 7879 12521] [1310 2290]
Excluded development rows with duplicate human texts: 3


In [277]:
def run_original_semantic_on_common_human_test(splits: dict):
    """Evaluate the unchanged scenario definitions on the same human set used by every new model."""
    semantic = splits['semantic']
    t0=time.perf_counter()
    human_vectors=vectorize_documents(
        splits['tokens_human'],semantic['modelo_word2vec'],agregacion='suma')
    for scenario,centroids in (('Scenario 1',semantic['centroides_1']),
                               ('Scenario 2',semantic['centroides_2'])):
        labels,_,_=label_with_centroids(human_vectors,centroids,semantic['cluster_politico'])
        elapsed=time.perf_counter()-t0
        model_id='Word2Vec_centroid_'+scenario.replace(' ','_')
        row=_benchmark_row(model_id,'original_semantic_reference',splits['y_human'],labels,
                           splits['protocol']['semantic_fit_seconds'],elapsed,
                           len(splits['y_train']),'cpu',
                           label_source='semantic nearest-centroid decision',
                           threshold_selection='no absolute similarity threshold')
        modern_result_rows.append(row)
        record_benchmark_predictions(model_id,labels)
        print(model_id,'human balanced accuracy:',round(row['balanced_accuracy'],4))
    save_benchmark_progress()


def run_common_word2vec_linearsvc(splits: dict,cfg: ModernBenchmarkConfig):
    """Control for the actual Word2Vec+LinearSVC benchmark on the exact same weak-label split."""
    from sklearn.svm import LinearSVC
    embedding= splits['semantic']['modelo_word2vec']
    t0=time.perf_counter()
    X_train=vectorize_documents(splits['tokens_train'],embedding,agregacion='media')
    X_valid=vectorize_documents(splits['tokens_val'],embedding,agregacion='media')
    train_rep_seconds=time.perf_counter()-t0
    t1=time.perf_counter()
    X_human=vectorize_documents(splits['tokens_human'],embedding,agregacion='media')
    human_rep_seconds=time.perf_counter()-t1
    estimator=LinearSVC(random_state=cfg.seed,max_iter=5000,dual='auto')
    t0=time.perf_counter()
    estimator.fit(X_train,splits['y_train'])
    train_seconds=time.perf_counter()-t0+train_rep_seconds
    t1=time.perf_counter()
    human_pred=estimator.predict(X_human)
    inference_seconds=time.perf_counter()-t1+human_rep_seconds
    label='Word2Vec_LinearSVC_common_split'
    val_scores=estimator.decision_function(X_valid)
    human_scores=estimator.decision_function(X_human)
    row=_benchmark_row(label,'classical',splits['y_human'],human_pred,
                       train_seconds,inference_seconds,len(splits['y_train']),'cpu',
                       feature_extraction_seconds=train_rep_seconds,
                       pseudo_validation_balanced_accuracy=float(balanced_accuracy_score(splits['y_val'],estimator.predict(X_valid))),
                       label_source='Scenario 2 weak labels',threshold_selection='default LinearSVC margin')
    modern_result_rows.append(row)
    record_benchmark_predictions(label,human_pred,human_scores)
    if cfg.enable_validation_threshold:
        threshold,weak_ba=choose_threshold_from_pseudo_validation(val_scores,splits['y_val'],0.0)
        adjusted=(human_scores>=threshold).astype(int)
        modern_result_rows.append(_benchmark_row(label+'_pseudo_threshold','classical_calibrated',
                 splits['y_human'],adjusted,train_seconds,inference_seconds,len(splits['y_train']),'cpu',
                 selected_weak_threshold=threshold,pseudo_validation_balanced_accuracy=weak_ba,
                 threshold_selection='weak validation only'))
        record_benchmark_predictions(label+'_pseudo_threshold',adjusted,human_scores)
    save_benchmark_progress()
    del X_train,X_valid,X_human,estimator
    gc.collect()

run_original_semantic_on_common_human_test(modern_splits)
if MODERN_CFG.benchmark_classical:
    run_common_word2vec_linearsvc(modern_splits,MODERN_CFG)

Word2Vec_centroid_Scenario_1 human balanced accuracy: 0.8108
Word2Vec_centroid_Scenario_2 human balanced accuracy: 0.7923


In [278]:
def run_classical_comparisons(splits: dict, cfg: ModernBenchmarkConfig):
    """Train classical algorithms on identical weakly labeled rows and evaluate on untouched humans."""
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import LinearSVC, SVC
    t0=time.perf_counter()
    vectorizer = TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=50000,sublinear_tf=True)
    Xtr=vectorizer.fit_transform(splits["train_text"])
    Xval=vectorizer.transform(splits["val_text"])
    representation_seconds=time.perf_counter()-t0
    t_human_rep=time.perf_counter()
    Xhuman=vectorizer.transform(splits["human_text"])
    human_rep_seconds=time.perf_counter()-t_human_rep
    estimators={
       "TFIDF_LinearSVC":LinearSVC(random_state=cfg.seed,max_iter=5000,dual="auto"),
       "TFIDF_LinearSVC_balanced":LinearSVC(random_state=cfg.seed,max_iter=5000,dual="auto",class_weight="balanced"),
       "TFIDF_LogisticRegression_balanced":LogisticRegression(random_state=cfg.seed,max_iter=1500,class_weight="balanced"),
    }
    if cfg.enable_rbf_baseline:
        estimators["TFIDF_RBF_SVM_balanced"] = SVC(kernel="rbf",class_weight="balanced",C=1.0)
    if cfg.enable_xgboost_baseline:
        from xgboost import XGBClassifier
        estimators["TFIDF_XGBoost"] = XGBClassifier(n_estimators=200,max_depth=5,learning_rate=0.05,
                                                     subsample=0.8,colsample_bytree=0.9,
                                                     tree_method="hist",n_jobs=cfg.num_cpu_threads,
                                                     eval_metric="logloss",random_state=cfg.seed)
    for label,estimator in estimators.items():
        print("Classical model:",label)
        t0=time.perf_counter()
        estimator.fit(Xtr,splits['y_train'])
        fit_seconds=time.perf_counter()-t0
        t1=time.perf_counter()
        pred=np.asarray(estimator.predict(Xhuman),dtype=int)
        infer_s=time.perf_counter()-t1
        row=_benchmark_row(label,"classical",splits['y_human'],pred,
                           fit_seconds+representation_seconds,infer_s+human_rep_seconds,len(splits['y_train']),"cpu",
                           feature_extraction_seconds=representation_seconds,
                           label_source="Scenario 2 weak labels",threshold_selection="default")
        if hasattr(estimator,'decision_function'):
            validation_scores=np.asarray(estimator.decision_function(Xval))
            test_scores=np.asarray(estimator.decision_function(Xhuman))
            cutoff=0.0
        elif hasattr(estimator,'predict_proba'):
            validation_scores=np.asarray(estimator.predict_proba(Xval)[:,1])
            test_scores=np.asarray(estimator.predict_proba(Xhuman)[:,1])
            cutoff=0.5
        else:
            validation_scores=None
        row['pseudo_validation_balanced_accuracy'] = float(balanced_accuracy_score(splits['y_val'],estimator.predict(Xval)))
        row['cpu_inference_seconds']=infer_s+human_rep_seconds
        row['cpu_throughput_tweets_per_s']=len(pred)/max(1e-9,infer_s+human_rep_seconds)
        modern_result_rows.append(row)
        record_benchmark_predictions(label,pred,test_scores if validation_scores is not None else None)
        if cfg.enable_validation_threshold and validation_scores is not None:
            threshold,selected_ba=choose_threshold_from_pseudo_validation(validation_scores,splits['y_val'],cutoff)
            adjusted=np.asarray(test_scores>=threshold,dtype=int)
            calibrated=_benchmark_row(label+"_pseudo_threshold","classical_calibrated",splits['y_human'],adjusted,
                                      fit_seconds+representation_seconds,infer_s+human_rep_seconds,len(splits['y_train']),"cpu",
                                      selected_weak_threshold=threshold,
                                      pseudo_validation_balanced_accuracy=selected_ba,
                                      threshold_selection="weak validation only")
            modern_result_rows.append(calibrated)
            record_benchmark_predictions(label+"_pseudo_threshold",adjusted,test_scores)
        save_benchmark_progress()
    del Xtr,Xval,Xhuman,vectorizer
    gc.collect()

if MODERN_CFG.benchmark_classical:
    run_classical_comparisons(modern_splits,MODERN_CFG)

Classical model: TFIDF_LinearSVC
Classical model: TFIDF_LinearSVC_balanced
Classical model: TFIDF_LogisticRegression_balanced
Classical model: TFIDF_RBF_SVM_balanced
Classical model: TFIDF_XGBoost


In [279]:
def preprocess_for_model(texts: Sequence[str], name: str) -> list[str]:
    """Apply the published RoBERTuito tweet preprocessing; other checkpoints use original text."""
    if name == "RoBERTuito":
        from pysentimiento.preprocessing import preprocess_tweet
        return [preprocess_tweet(str(t),lang="es") for t in texts]
    return [str(t) for t in texts]

class ModernTextDataset(torch.utils.data.Dataset):
    """Keep untokenized texts in memory; tokenize batches on demand."""
    def __init__(self,texts,labels=None):
        self.texts=list(texts)
        self.labels=None if labels is None else np.asarray(labels,dtype=int)
    def __len__(self):return len(self.texts)
    def __getitem__(self,index):
        row={'text':self.texts[index]}
        if self.labels is not None: row['labels']=int(self.labels[index])
        return row

def make_transformer_loader(texts,labels,tokenizer,cfg,shuffle=False,training=False):
    """Build batched, dynamic-padding, GPU-compatible DataLoaders."""
    dataset=ModernTextDataset(texts,labels)
    def collate(rows):
        tokens=tokenizer([r['text'] for r in rows],padding=True,truncation=True,
                         max_length=cfg.max_length,return_tensors='pt')
        if labels is not None:
            tokens['labels']=torch.tensor([r['labels'] for r in rows],dtype=torch.long)
        return tokens
    return torch.utils.data.DataLoader(dataset,batch_size=cfg.train_batch_size if training else cfg.eval_batch_size,
                                      shuffle=shuffle,num_workers=0,pin_memory=torch.cuda.is_available(),
                                      collate_fn=collate)

def model_with_visible_gpus(model,cfg):
    """Use DataParallel on allowed visible CUDA devices, without claiming busy physical GPUs."""
    if torch.cuda.is_available():
        ids=list(range(torch.cuda.device_count()))
        for i in ids:
            try: torch.cuda.set_per_process_memory_fraction(cfg.cuda_memory_fraction,i)
            except RuntimeError as ex: print(f"GPU memory cap unavailable on logical device {i}: {ex}")
        model=model.to('cuda:0')
        if cfg.gpu_parallel and len(ids)>1:
            model=torch.nn.DataParallel(model,device_ids=ids)
        return model,torch.device('cuda:0')
    return model.to('cpu'),torch.device('cpu')

def unwrap_transformer(model):
    """Return the underlying Hugging Face model when DataParallel is active."""
    return model.module if isinstance(model,torch.nn.DataParallel) else model

def classify_transformer(model,loader,device,use_amp):
    """Run inference and return discrete predictions and positive-class probabilities."""
    model.eval()
    predictions=[];scores=[]
    with torch.inference_mode():
        for batch in loader:
            inputs={k:v.to(device,non_blocking=True) for k,v in batch.items() if k!='labels'}
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=use_amp):
                logits=model(**inputs).logits.float()
            probabilities=torch.softmax(logits,dim=-1)[:,1]
            predictions.extend(logits.argmax(dim=-1).cpu().numpy().tolist())
            scores.extend(probabilities.cpu().numpy().tolist())
    return np.asarray(predictions,dtype=int),np.asarray(scores,dtype=float)

In [280]:
def run_frozen_transformer_linearsvc(model_name,splits,cfg):
    """Compare a frozen checkpoint plus common LinearSVC on the controlled split."""
    from transformers import AutoModel,AutoTokenizer
    model_id=MODERN_MODEL_REGISTRY[model_name]
    print(f"Frozen {model_name} + LinearSVC: {model_id}")
    t_all=time.perf_counter()
    texts={key:preprocess_for_model(splits[key+'_text'],model_name) for key in ('train','val','human')}
    tokenizer_options={'normalization':True} if model_name=='BERTweet' else {}
    tokenizer=AutoTokenizer.from_pretrained(model_id,**tokenizer_options)
    backbone=AutoModel.from_pretrained(model_id)
    backbone,device=model_with_visible_gpus(backbone,cfg)
    amp=device.type=='cuda'
    embeddings={}
    extraction_by_part={}
    for part in ('train','val','human'):
        t_extract_part=time.perf_counter()
        loader=make_transformer_loader(texts[part],None,tokenizer,cfg)
        rows=[]
        backbone.eval()
        with torch.inference_mode():
            for batch in tqdm(loader,desc=f'{model_name} frozen {part}',leave=False):
                inputs={k:v.to(device,non_blocking=True) for k,v in batch.items()}
                with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=amp):
                    output=backbone(**inputs)
                    hidden=output.last_hidden_state.float()
                    mask=inputs['attention_mask'].unsqueeze(-1).to(hidden.dtype)
                    pooled=(hidden*mask).sum(1)/mask.sum(1).clamp(min=1)
                rows.append(pooled.cpu().numpy())
        embeddings[part]=np.vstack(rows)
        extraction_by_part[part]=time.perf_counter()-t_extract_part
    extraction_seconds=time.perf_counter()-t_all
    clf=LinearSVC(random_state=cfg.seed,max_iter=5000,dual='auto')
    t0=time.perf_counter();clf.fit(embeddings['train'],splits['y_train']);fit_seconds=time.perf_counter()-t0
    t1=time.perf_counter();pred=clf.predict(embeddings['human']);inference_seconds=time.perf_counter()-t1
    val_scores=clf.decision_function(embeddings['val'])
    test_scores=clf.decision_function(embeddings['human'])
    label=model_name+'_frozen_LinearSVC'
    row=_benchmark_row(label,'frozen_feature_LinearSVC',splits['y_human'],pred,
                       fit_seconds+extraction_by_part['train']+extraction_by_part['val'],
                       inference_seconds+extraction_by_part['human'],len(splits['y_train']),str(device),
                       feature_extraction_seconds=extraction_seconds,
                       human_embedding_seconds=extraction_by_part['human'],
                       pseudo_validation_balanced_accuracy=float(balanced_accuracy_score(splits['y_val'],clf.predict(embeddings['val']))),
                       threshold_selection='default LinearSVC margin')
    modern_result_rows.append(row)
    record_benchmark_predictions(label,pred,test_scores)
    if cfg.enable_validation_threshold:
        threshold,weak_ba=choose_threshold_from_pseudo_validation(val_scores,splits['y_val'],0.0)
        adjusted=(test_scores>=threshold).astype(int)
        modern_result_rows.append(_benchmark_row(label+'_pseudo_threshold','frozen_calibrated',splits['y_human'],adjusted,
                fit_seconds+extraction_by_part['train']+extraction_by_part['val'],
                inference_seconds+extraction_by_part['human'],len(splits['y_train']),str(device),
                selected_weak_threshold=threshold,pseudo_validation_balanced_accuracy=weak_ba,
                threshold_selection='weak validation only'))
        record_benchmark_predictions(label+'_pseudo_threshold',adjusted,test_scores)
    del backbone,clf,embeddings,tokenizer
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    save_benchmark_progress()

if MODERN_CFG.benchmark_frozen_features:
    for model_name in MODERN_CFG.model_names:
        try:
            run_frozen_transformer_linearsvc(model_name,modern_splits,MODERN_CFG)
        except Exception as error:
            import traceback
            modern_run_errors.append({'model':model_name,'phase':'frozen_LinearSVC',
                                      'message':str(error),'traceback':traceback.format_exc()})
            save_benchmark_progress()
            print(f"FAILED frozen {model_name}: {error}")
            if torch.cuda.is_available():torch.cuda.empty_cache()

Frozen RoBERTuito + LinearSVC: pysentimiento/robertuito-base-uncased
FAILED frozen RoBERTuito: Failed to import transformers.trainer because of the following error (look up to see its traceback):
pyarrow requires NumPy 2.0 or newer, found 1.26.4
Frozen BETO + LinearSVC: dccuchile/bert-base-spanish-wwm-uncased


Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-uncased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BETO frozen train:   0%|          | 0/638 [00:00<?, ?it/s]

BETO frozen val:   0%|          | 0/113 [00:00<?, ?it/s]

BETO frozen human:   0%|          | 0/19 [00:00<?, ?it/s]

Frozen XLM-RoBERTa + LinearSVC: xlm-roberta-base


... (26 outputs left out)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

PoliBERTweet frozen val:   0%|          | 0/113 [00:00<?, ?it/s]

PoliBERTweet frozen human:   0%|          | 0/19 [00:00<?, ?it/s]

In [281]:
def run_fine_tuned_transformer(model_name,splits,cfg,balanced=False):
    """Train an end-to-end classifier, select checkpoint on weak validation, evaluate humans once."""
    from transformers import AutoTokenizer,AutoModelForSequenceClassification,get_linear_schedule_with_warmup
    label=model_name+('_weighted' if balanced else '_standard')
    model_id=MODERN_MODEL_REGISTRY[model_name]
    print(f"Fine-tuning {label} from {model_id}")
    torch.manual_seed(cfg.seed)
    np.random.seed(cfg.seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(cfg.seed)
    torch.set_num_threads(cfg.num_cpu_threads)
    start_prep=time.perf_counter()
    texts={part:preprocess_for_model(splits[part+'_text'],model_name) for part in ('train','val','human')}
    preprocessing_seconds=time.perf_counter()-start_prep
    tokenizer_options={'normalization':True} if model_name=='BERTweet' else {}
    t0=time.perf_counter()
    tokenizer=AutoTokenizer.from_pretrained(model_id,**tokenizer_options)
    base=AutoModelForSequenceClassification.from_pretrained(model_id,num_labels=2,
                   id2label={0:'non-political',1:'political'},
                   label2id={'non-political':0,'political':1},ignore_mismatched_sizes=True)
    n_parameters=sum(param.numel() for param in base.parameters())
    net,device=model_with_visible_gpus(base,cfg)
    loading_seconds=time.perf_counter()-t0
    amp=device.type=='cuda'
    train_loader=make_transformer_loader(texts['train'],splits['y_train'],tokenizer,cfg,shuffle=True,training=True)
    val_loader=make_transformer_loader(texts['val'],None,tokenizer,cfg)
    human_loader=make_transformer_loader(texts['human'],None,tokenizer,cfg)
    optimizer=torch.optim.AdamW(net.parameters(),lr=cfg.learning_rate,weight_decay=cfg.weight_decay)
    total_updates=max(1, math.ceil(len(train_loader)/cfg.grad_accumulation_steps)*cfg.epochs)
    scheduler=get_linear_schedule_with_warmup(optimizer,
                    num_warmup_steps=int(cfg.warmup_fraction*total_updates),num_training_steps=total_updates)
    scaler=torch.amp.GradScaler('cuda',enabled=amp)
    weights=None
    if balanced:
        counts=np.bincount(splits['y_train'],minlength=2)
        weights=torch.tensor(len(splits['y_train'])/(2*np.maximum(1,counts)),dtype=torch.float,device=device)
    checkpoint_dir=MODERN_CHECKPOINT_DIR/label
    checkpoint_dir.mkdir(parents=True,exist_ok=True)
    best_checkpoint=checkpoint_dir/'best_model_state.pt'
    best_score=float('-inf'); best_epoch=0
    training_log=[]
    if amp:
        torch.cuda.reset_peak_memory_stats(device)
    t_training=time.perf_counter()
    try:
        for epoch in range(cfg.epochs):
            net.train();optimizer.zero_grad(set_to_none=True)
            losses=[]
            for step,batch in enumerate(tqdm(train_loader,desc=f'{label} epoch {epoch+1}/{cfg.epochs}',leave=False)):
                labels=batch['labels'].to(device,non_blocking=True)
                inputs={k:v.to(device,non_blocking=True) for k,v in batch.items() if k!='labels'}
                with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=amp):
                    logits=net(**inputs).logits
                    loss=torch.nn.functional.cross_entropy(logits.float(),labels,weight=weights)
                    scaled_loss=loss/cfg.grad_accumulation_steps
                scaler.scale(scaled_loss).backward()
                losses.append(float(loss.detach().cpu()))
                if (step+1)%cfg.grad_accumulation_steps==0 or (step+1)==len(train_loader):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(net.parameters(),1.0)
                    scaler.step(optimizer)
                    scaler.update();scheduler.step();optimizer.zero_grad(set_to_none=True)
            val_pred,_=classify_transformer(net,val_loader,device,amp)
            val_ba=float(balanced_accuracy_score(splits['y_val'],val_pred))
            training_log.append({'model':label,'epoch':epoch+1,
                                 'training_loss':float(np.mean(losses)),
                                 'weak_validation_balanced_accuracy':val_ba})
            print(label,'epoch',epoch+1,'weak validation BA:',round(val_ba,4))
            if val_ba>best_score:
                best_score=val_ba;best_epoch=epoch+1
                torch.save({key:tensor.detach().cpu() for key,tensor in unwrap_transformer(net).state_dict().items()},best_checkpoint)
        training_seconds=time.perf_counter()-t_training
        weights_best=torch.load(best_checkpoint,map_location='cpu',weights_only=True)
        unwrap_transformer(net).load_state_dict(weights_best)
        del weights_best
        torch.cuda.empty_cache() if amp else None
        t_infer=time.perf_counter()
        predictions,probabilities=classify_transformer(net,human_loader,device,amp)
        inference_seconds=time.perf_counter()-t_infer
        _,validation_prob=classify_transformer(net,val_loader,device,amp)
        cuda_peak_mb=float(torch.cuda.max_memory_allocated(device)/1024**2) if amp else 0.0
        cpu_inference_seconds=float('nan')
        cpu_prediction_difference=float('nan')
        if cfg.evaluate_cpu_inference and amp:
            net=unwrap_transformer(net).cpu().eval()
            gc.collect();torch.cuda.empty_cache()
            t_cpu=time.perf_counter()
            cpu_pred,_=classify_transformer(net,human_loader,torch.device('cpu'),False)
            cpu_inference_seconds=time.perf_counter()-t_cpu
            cpu_prediction_difference=int(np.sum(cpu_pred!=predictions))
        full_inference_cost=inference_seconds
        row=_benchmark_row(label,'fine_tuned_weighted' if balanced else 'fine_tuned',
                           splits['y_human'],predictions,training_seconds,full_inference_cost,
                           len(splits['y_train']),'cuda' if amp else 'cpu',
                           model_id=model_id,model_parameters=n_parameters,best_epoch=best_epoch,
                           pseudo_validation_balanced_accuracy=best_score,
                           preprocessing_seconds=preprocessing_seconds,
                           model_loading_seconds=loading_seconds,
                           end_to_end_seconds=preprocessing_seconds+loading_seconds+training_seconds+inference_seconds,
                           gpu_peak_allocated_mb=cuda_peak_mb,
                           cpu_inference_seconds=cpu_inference_seconds,
                           cpu_throughput_tweets_per_s=len(predictions)/max(1e-9,cpu_inference_seconds)
                               if np.isfinite(cpu_inference_seconds) else float('nan'),
                           cpu_gpu_prediction_differences=cpu_prediction_difference,
                           threshold_selection='default argmax')
        modern_result_rows.append(row)
        record_benchmark_predictions(label,predictions,probabilities)
        if cfg.enable_validation_threshold:
            threshold,weak_ba=choose_threshold_from_pseudo_validation(validation_prob,splits['y_val'],0.5)
            pred_adjusted=(probabilities>=threshold).astype(int)
            modern_result_rows.append(_benchmark_row(label+'_pseudo_threshold','fine_tuned_calibrated',
                   splits['y_human'],pred_adjusted,training_seconds,inference_seconds,
                   len(splits['y_train']),'cuda' if amp else 'cpu',model_id=model_id,
                   selected_weak_threshold=threshold,pseudo_validation_balanced_accuracy=weak_ba,
                   best_epoch=best_epoch,threshold_selection='weak validation only'))
            record_benchmark_predictions(label+'_pseudo_threshold',pred_adjusted,probabilities)
        pd.DataFrame(training_log).to_csv(MODERN_BENCHMARK_DIR/f'{label}_training_history.csv',index=False)
        if cfg.save_fine_tuned_models:
            final_dir=MODERN_MODEL_DIR/label
            final_dir.mkdir(parents=True,exist_ok=True)
            unwrap_transformer(net).save_pretrained(final_dir,safe_serialization=True)
            tokenizer.save_pretrained(final_dir)
        save_benchmark_progress()
    finally:
        del net,base,optimizer,scheduler,scaler,train_loader,val_loader,human_loader,tokenizer
        gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
        # Checkpoints are transient when the final Hugging Face weights were saved.
        if best_checkpoint.exists() and cfg.save_fine_tuned_models:
            best_checkpoint.unlink()

if MODERN_CFG.benchmark_fine_tuning:
    for name in MODERN_CFG.model_names:
        variants=[False,True] if MODERN_CFG.balanced_loss_variant and name=='RoBERTuito' else [False]
        for weighted in variants:
            try:
                run_fine_tuned_transformer(name,modern_splits,MODERN_CFG,balanced=weighted)
            except Exception as error:
                import traceback
                modern_run_errors.append({'model':name,'phase':'fine_tuning',
                                          'weighted':weighted,'message':str(error),
                                          'traceback':traceback.format_exc()})
                save_benchmark_progress()
                print(f'FAILED fine-tuning {name} (weighted={weighted}): {error}')
                if torch.cuda.is_available():torch.cuda.empty_cache()
                gc.collect()

Fine-tuning RoBERTuito_standard from pysentimiento/robertuito-base-uncased


tokenizer_config.json:   0%|          | 0.00/323 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


... (27 outputs left out)

PoliBERTweet_standard epoch 1/2:   0%|          | 0/1275 [00:00<?, ?it/s]

PoliBERTweet_standard epoch 1 weak validation BA: 0.7893


PoliBERTweet_standard epoch 2/2:   0%|          | 0/1275 [00:00<?, ?it/s]

PoliBERTweet_standard epoch 2 weak validation BA: 0.8124


In [282]:
modern_benchmark_results=pd.DataFrame(modern_result_rows)
modern_protocol_table=pd.DataFrame([{'Field':key,'Value':str(value)} for key,value in modern_splits['protocol'].items()])
save_benchmark_progress()
if modern_benchmark_results.empty:
    raise RuntimeError('No modern benchmark model completed. Inspect errors.json for the exact model failures.')
print('Completed benchmark rows:',len(modern_benchmark_results))
print('Unsuccessful model runs:',len(modern_run_errors))
print('Human-test benchmark, ranked by balanced accuracy:')
cols=['model','family','balanced_accuracy','recall','specificity','false_positive_rate',
      'f1_political','training_seconds','inference_seconds','cpu_memory_rss_mb','n_train']
display(modern_benchmark_results.sort_values('balanced_accuracy',ascending=False)[cols])
modern_benchmark_results.to_excel(MODERN_BENCHMARK_DIR/'benchmark_results.xlsx',index=False)

Completed benchmark rows: 34
Unsuccessful model runs: 1
Human-test benchmark, ranked by balanced accuracy:


,model,family,balanced_accuracy,recall,specificity,false_positive_rate,f1_political,training_seconds,inference_seconds,cpu_memory_rss_mb,n_train
27,BETO_standard_pseudo_threshold,fine_tuned_calibrated,0.838181,0.870629,0.805732,0.194268,0.835570,655.736614,2.045688,"5,593.625000",20400
29,XLM-RoBERTa_standard_pseudo_threshold,fine_tuned_calibrated,0.835464,0.881119,0.789809,0.210191,0.834437,"1,423.931528",4.216002,"6,445.398438",20400
26,BETO_standard,fine_tuned,0.829529,0.926573,0.732484,0.267516,0.834646,655.736614,2.045688,"5,593.617188",20400
28,XLM-RoBERTa_standard,fine_tuned,0.818883,0.902098,0.735669,0.264331,0.822967,"1,423.931528",4.216002,"6,445.386719",20400
8,TFIDF_LogisticRegression_balanced,classical,0.817324,0.867133,0.767516,0.232484,0.817133,10.685175,0.007402,"4,211.882812",20400
13,TFIDF_XGBoost_pseudo_threshold,classical_calibrated,0.816957,0.751748,0.882166,0.117834,0.799257,4.343138,0.008740,"4,064.109375",20400
9,TFIDF_LogisticRegression_balanced_pseudo_threshold,classical_calibrated,0.815732,0.867133,0.764331,0.235669,0.815789,10.685175,0.007402,"4,211.882812",20400
0,Word2Vec_centroid_Scenario_1,original_semantic_reference,0.810799,0.863636,0.757962,0.242038,0.811166,28.510658,0.009191,"4,193.730469",20400
11,TFIDF_RBF_SVM_balanced_pseudo_threshold,classical_calibrated,0.809363,0.867133,0.751592,0.248408,0.810458,68.063295,0.666944,"4,033.601562",20400
5,TFIDF_LinearSVC_pseudo_threshold,classical_calibrated,0.806022,0.863636,0.748408,0.251592,0.807190,0.683921,0.007001,"4,211.757812",20400


In [283]:
def plot_modern_quality_efficiency(benchmark: pd.DataFrame):
    """Plot independent human balanced accuracy versus model inference latency."""
    frame=benchmark[benchmark['family'].isin(['classical','frozen_feature_LinearSVC','fine_tuned','fine_tuned_weighted'])].copy()
    frame=frame[(frame.inference_seconds>0)&frame.balanced_accuracy.notna()]
    if frame.empty:return
    from bokeh.models import ColumnDataSource,HoverTool,LabelSet
    from bokeh.plotting import figure,show
    source=ColumnDataSource(frame)
    p=figure(width=950,height=550,x_axis_type='log',
             title='Human-validated balanced accuracy versus inference time',
             x_axis_label='Human-test inference time (seconds, logarithmic)',
             y_axis_label='Balanced accuracy',tools='pan,wheel_zoom,box_zoom,reset,save')
    points=p.scatter('inference_seconds','balanced_accuracy',source=source,size=10,alpha=0.8)
    p.add_layout(LabelSet(x='inference_seconds',y='balanced_accuracy',text='model',source=source,
                          x_offset=5,y_offset=6,text_font_size='8pt'))
    p.add_tools(HoverTool(renderers=[points],tooltips=[('Model','@model'),('Family','@family'),
                                    ('Balanced accuracy','@balanced_accuracy{0.0000}'),
                                    ('Specificity','@specificity{0.0000}'),
                                    ('Inference seconds','@inference_seconds{0.000}')]))
    show(p)

plot_modern_quality_efficiency(modern_benchmark_results)
print('Final model weights:',MODERN_MODEL_DIR)
print('Benchmark reports:',MODERN_BENCHMARK_DIR)
print('Checkpoint storage:',MODERN_CHECKPOINT_DIR)

Final model weights: /home/lrecalde/data/cayala/models/benchmarks/modern_models
Benchmark reports: /home/lrecalde/data/cayala/results/benchmarks/modern_models
Checkpoint storage: /home/lrecalde/data/cayala/checkpoints/benchmarks/modern_models


# 12. Descriptive analysis after independent evaluation

In [284]:
def normalize_province(valor: Any) -> str:
    """Normalize province. Retains the original experimental logic and data-partition rules."""
    if pd.isna(valor) or not str(valor).strip():
        return "No location"

    texto = str(valor).lower().strip()
    reemplazos = str.maketrans("áéíóúüñ", "aeiouun")
    texto = texto.translate(reemplazos)
    texto = re.sub(r"[^a-z0-9#@\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()

    alias = {
        "Azuay": ["azuay", "cuenca", "gualaceo"],
        "Bolívar": ["bolivar", "guaranda"],
        "Cañar": ["canar", "azogues", "la troncal"],
        "Carchi": ["carchi", "tulcan"],
        "Chimborazo": ["chimborazo", "riobamba"],
        "Cotopaxi": ["cotopaxi", "latacunga"],
        "El Oro": ["el oro", "machala", "pasaje"],
        "Esmeraldas": ["esmeraldas", "atacames"],
        "Galápagos": ["galapagos", "puerto ayora", "san cristobal"],
        "Guayas": ["guayas", "guayaquil", "duran", "samborondon", "daule", "milagro"],
        "Imbabura": ["imbabura", "ibarra", "otavalo"],
        "Loja": ["loja", "catamayo"],
        "Los Ríos": ["los rios", "babahoyo", "quevedo", "vinces"],
        "Manabí": ["manabi", "manta", "portoviejo", "chone", "jipijapa"],
        "Morona Santiago": ["morona santiago", "macas"],
        "Napo": ["napo", "tena"],
        "Orellana": ["orellana", "coca", "francisco de orellana"],
        "Pastaza": ["pastaza", "puyo"],
        "Pichincha": ["pichincha", "quito", "cayambe", "sangolqui", "ruminahui"],
        "Santa Elena": ["santa elena", "salinas", "la libertad"],
        "Santo Domingo de los Tsáchilas": ["santo domingo", "tsachilas"],
        "Sucumbíos": ["sucumbios", "lago agrio", "nueva loja"],
        "Tungurahua": ["tungurahua", "ambato", "banos"],
        "Zamora Chinchipe": ["zamora chinchipe", "zamora"],
    }

    for provincia, variantes in alias.items():
        for variante in variantes:
            if re.search(rf"\b{re.escape(variante)}\b", texto):
                return provincia
    return "Unidentified"

In [285]:
def count_political_tweets_by_province(tweets: pd.DataFrame, etiquetas: np.ndarray, columna_ubicacion: str = "location") -> pd.DataFrame:
    """Count political tweets by province. Retains the original experimental logic and data-partition rules."""
    if len(tweets) != len(etiquetas):
        raise ValueError("The number of tweets and labels must match.")
    df = tweets[[columna_ubicacion]].copy()
    df["label"] = etiquetas.astype(int)
    df = df[df["label"] == 1].copy()
    df["Province"] = df[columna_ubicacion].map(normalize_province)
    return df["Province"].value_counts().rename_axis("Province").reset_index(name="Tweets_politicos")

In [286]:
def plot_scenario_provinces(
    provincias_1: pd.DataFrame,
    provincias_2: pd.DataFrame,
    dimension: int,
    top_n: int = 15
) -> None:
    """Plot scenario provinces. Retains the original experimental logic and data-partition rules."""

    a = provincias_1.rename(
        columns={"Tweets_politicos": "Escenario_1"}
    )

    b = provincias_2.rename(
        columns={"Tweets_politicos": "Escenario_2"}
    )

    df = a.merge(
        b,
        on="Province",
        how="outer"
    ).fillna(0)

    df["Total"] = (
        df["Escenario_1"] +
        df["Escenario_2"]
    )

    df = (
        df.sort_values("Total", ascending=False)
        .head(top_n)
    )

    factores = df["Province"].astype(str).tolist()

    source = ColumnDataSource(df)

    p = figure(
        x_range=factores,
        width=1100,
        height=450,
        title=f"Predicted political tweets by province — dimensión {dimension}"
    )

    p.vbar(
        x=dodge("Province", -0.18, range=p.x_range),
        top="Escenario_1",
        width=0.34,
        source=source,
        color="#1f77b4",
        legend_label="Scenario 1"
    )

    p.vbar(
        x=dodge("Province", 0.18, range=p.x_range),
        top="Escenario_2",
        width=0.34,
        source=source,
        color="#ff7f0e",
        legend_label="Scenario 2"
    )

    p.xaxis.major_label_orientation = math.pi / 3
    p.yaxis.axis_label = "Political tweets"

    p.legend.location = "top_right"

    p.add_tools(
        HoverTool(
            tooltips=[
                ("Province", "@Provincia"),
                ("Scenario 1", "@Escenario_1{0,0}"),
                ("Scenario 2", "@Escenario_2{0,0}")
            ]
        )
    )

    show(p)

In [287]:
tokens_desarrollo_total = extract_dataframe_tokens(
    tweets_desarrollo,
    "tokens_sin_stopwords",
)

In [288]:
pipeline_descriptivo = build_train_semantic_pipeline(
    tokens_desarrollo_total,
    tokens_trends,
    DIMENSION_BENCHMARK,
    semillas_politicas,
    K_CANDIDATOS,
    EXPANSION_TOP_N_FINAL,
    W2V_EPOCHS,
    RANDOM_STATE,
)

In [289]:
X_descriptivo = vectorize_documents(
    tokens_desarrollo_total,
    pipeline_descriptivo["modelo_word2vec"],
    agregacion="suma",
)

In [290]:
y_desc_e1, _, _ = label_with_centroids(
    X_descriptivo,
    pipeline_descriptivo["centroides_1"],
    pipeline_descriptivo["cluster_politico"],
)

In [291]:
y_desc_e2, _, _ = label_with_centroids(
    X_descriptivo,
    pipeline_descriptivo["centroides_2"],
    pipeline_descriptivo["cluster_politico"],
)

In [292]:
cobertura_descriptiva = pd.DataFrame([
    {
        "Scenario": "Scenario 1",
        "N": len(y_desc_e1),
        "Politicos": int(y_desc_e1.sum()),
        "No_politicos": int((y_desc_e1 == 0).sum()),
        "Politicos_pct": float(y_desc_e1.mean() * 100.0),
    },
    {
        "Scenario": "Scenario 2",
        "N": len(y_desc_e2),
        "Politicos": int(y_desc_e2.sum()),
        "No_politicos": int((y_desc_e2 == 0).sum()),
        "Politicos_pct": float(y_desc_e2.mean() * 100.0),
    },
])

In [293]:
display_bokeh_table(
    cobertura_descriptiva,
    "Descriptive political-content coverage",
    alto=190,
)

In [294]:
provincias_e1 = count_political_tweets_by_province(tweets_desarrollo, y_desc_e1)

In [295]:
provincias_e2 = count_political_tweets_by_province(tweets_desarrollo, y_desc_e2)

In [296]:
plot_scenario_provinces(
    provincias_e1,
    provincias_e2,
    DIMENSION_BENCHMARK,
    top_n=15,
)

# 13. Save final study results

In [297]:
def save_final_result_tables(tablas: Mapping[str, pd.DataFrame], output_dir: Path) -> Path:
    """Save final result tables. Retains the original experimental logic and data-partition rules."""
    output_dir.mkdir(parents=True, exist_ok=True)
    ruta_excel = output_dir / "study_results.xlsx"

    with pd.ExcelWriter(ruta_excel, engine="openpyxl") as writer:
        for nombre, tabla in tablas.items():
            if tabla is None or not isinstance(tabla, pd.DataFrame):
                continue
            hoja = re.sub(r"[^A-Za-z0-9_]", "_", str(nombre))[:31] or "Hoja"
            tabla.to_excel(writer, sheet_name=hoja, index=False)
            tabla.to_csv(output_dir / f"{hoja}.csv", index=False, encoding="utf-8-sig")

    return ruta_excel

In [298]:
tablas_finales = {
    "modern_benchmark": modern_benchmark_results,
    "benchmark_protocol": modern_protocol_table,
    "auditoria_librerias": AUDITORIA_LIBRERIAS,
    "limpieza": AUDITORIA_LIMPIEZA,
    "auditoria_hardware": AUDITORIA_HARDWARE,
    "auditoria_dataset": AUDITORIA_DATASET["resumen"],
    "auditoria_eventos": AUDITORIA_DATASET["eventos"],
    "auditoria_faltantes": AUDITORIA_DATASET["faltantes"],

    "silver_balance": resumen_silver,

    "holdout_resumen": resumen_holdout,
    "holdout_metricas": metricas_holdout,

    "k_objetivo": pipe_principal["tabla_k"],

    "lexico_topn": pipe_principal["palabras_anadidas"],

    "supplementary_top100": tabla_top100_suplementaria,
    "supplementary_top100_summary": verificacion_top100,

    "clustering_comp": comparacion_clustering,
    "sensibilidad_topn": sensibilidad_top_n,

    "cv_resultados": cv_resultados,
    "cv_resumen": cv_resumen,
    "wilcoxon_holm": wilcoxon_escenarios,
    
    "appendix_b_cv_complete": tabla_appendix_b_cv,
    "appendix_b_cv_table": tabla_appendix_b_cv_table,
    
    "silver_metricas": metricas_silver,
    "silver_balance_eval": metricas_silver_balance,
    "silver_evento": metricas_silver_evento,
    "silver_mcnemar": mcnemar_silver,

    "hash_humano_metricas": METRICAS_AUDITORIA_HUMANA_HASHTAG,
    "hash_humano_mcnemar": MCNEMAR_AUDITORIA_HUMANA_HASHTAG,
    "hash_humano_detalle": DETALLE_AUDITORIA_HUMANA_HASHTAG,

    "benchmark_silver": benchmark_silver,
    "finetuning_silver": fine_tuning_silver,

    "acuerdo_anotadores": acuerdo_anotadores,
    "validacion_manual": validacion_manual,
    "benchmark_manual": benchmark_manual,

    "cobertura_descriptiva": cobertura_descriptiva,
    "provincias_e1": provincias_e1,
    "provincias_e2": provincias_e2,
}

In [299]:
ruta_resultados_finales = save_final_result_tables(
    tablas_finales,
    REVISION_OUTPUT_DIR,
)

In [300]:
print(f"Consolidated results: {ruta_resultados_finales}")

Consolidated results: /home/lrecalde/data/cayala/results/study_results.xlsx


In [301]:
print(f"Version audit: {RUTA_REQUIREMENTS_RUNTIME}")

Version audit: /home/lrecalde/data/cayala/envs/requirements_runtime.txt
